# SA3 Native Science Lab on Colab L4

This notebook is a Colab L4 research instrument for frozen Stable Audio 3 Medium and SAME, organized from native objects upward.

Run the setup cells top-to-bottom on a fresh L4 runtime. The setup defaults are already ON for:

```text
SA3 Medium
Torch 2.7.1 + CUDA 12.6
FlashAttention
NumPy pinning for Colab binary wheels
optional scipy/sklearn/torchvision removal to avoid Transformers optional import failures
one automatic runtime restart after first install
this repo's latent_audio_primitives package
Hugging Face login
model loading
one short smoke test
```

Why the dependency reset exists:

```text
uv resolves the requested dependency graph correctly, but Colab starts with many
preinstalled packages outside that graph. Transformers may opportunistically
import optional scipy/sklearn/torchvision modules while loading T5Gemma. Those
wheels can fail after the NumPy/Torch stack changes. SA3 does
not need scipy, sklearn, or torchvision for inference, so the setup pins NumPy
and removes those optional import paths. Because Torch/NumPy are binary packages,
the setup intentionally restarts the Colab runtime once after the first install.
```

Research layers:

```text
1. SAME representation science
2. SA3 flow and conditioning science
3. SA3 internal trajectory science
4. SA3-over-SAME coupled editing
```

Evidence utilities are shared review tools, not a fifth model-object layer: player panels, descriptors, annotations, memory rows, disagreement rows, manifests, and ledger decisions.

Native spaces used here:

| Native space | Notation | Meaning |
|---|---|---|
| Audio waveform | \(x\) | Source or decoded waveform. |
| SAME latent | \(z = E(x)\), \(z \in \mathbb{R}^{T \times 256}\) | Time-major SAME latent sequence. |
| SA3 text condition | \(c = C(p)\) | Prompt-conditioning object. |
| SA3 flow field | \(v_\theta(z_t, t, c)\) | Frozen vector field over SAME-shaped latents. |
| SA3 residual stream | \(a_\ell\) | Activation inside DiT block \(\ell\). |
| Decoded audio | \(\hat{x} = D(z)\) | SAME decoder output. |

The notebook itinerary is: runtime and evidence setup, audio-to-SAME preparation, SAME representation and memory workbenches, SA3 flow/conditioning probes, SA3 internal trajectory probes, SA3-over-SAME coupled editing, external comparison, and ledger decisions.

Every result should name its research layer or evidence utility before making a claim. No procedure is promoted because it exists; promotion happens only after source/baseline/method evidence packets and listening-led ledger decisions.


## Research Run Protocol

Every run should be framed before execution:

- Research layer / evidence utility: SAME representation, SA3 flow/conditioning, SA3 internal trajectory, SA3-over-SAME coupled editing, or evidence utility.
- Object: native object under study, such as audio, SAME \(z_0\), SA3 \(z_t\), prompt condition, residual activation, control lane, memory row, or evidence packet.
- Transition: the object path being tested, such as audio -> \(z_0\), \(z_0\) -> \(z_0'\), target \(z_0\) -> flow probes, prompt -> condition, residual activation -> steering vector, memory -> donor, or output -> evidence packet.
- Operation: observe, select, intervene, render, compare, or decide.
- Measurement: descriptor, flow, latent, geometry, control-lane, periodicity, nearest-memory, residual, runtime, and listening evidence.
- Claim: what success would mean: controllability, prompt inversion, source preservation, loopability, novelty, style transfer, selector value, or microscope value.
- Decision: promote, revise, drop, unknown, or microscope only.

Use this run spine:

```text
choose source audio / dataset / prompt family
-> encode or load native objects
-> record baseline output or baseline measurements
-> choose one object transition and operation
-> run with explicit seed, steps, duration, prompt, and convention
-> collect descriptor / flow / latent / memory / geometry / lane / residual evidence
-> audition with the notebook player
-> annotate listening result
-> write or update the experiment ledger
-> decide maturity: microscope / selector / intervention candidate / promoted / dropped
```

Do not promote a method from latent metrics alone. A useful result needs baseline comparison, measurements, listening notes, and a ledger decision.


## Core Math

SAME encodes audio into a compressed continuous latent sequence:

$$
z = E(x), \qquad z \in \mathbb{R}^{T \times 256}
$$

For 44.1 kHz audio and downsampling ratio 4096:

$$
\text{latent rate} \approx \frac{44100}{4096} \approx 10.77 \text{ frames/sec}
$$

SA3 is treated here as a conditional latent generator over SAME latents:

$$
v_\theta(z_t, t, c)
$$

where:

- \(z_t\) is the intermediate noisy/flow latent.
- \(t\) is flow time.
- \(c\) is the SA3 text-conditioning tensor.

For a rectified-flow-style surrogate objective, define a straight probability path between a data latent \(z_0\) and Gaussian noise \(\epsilon\):

$$
\epsilon \sim \mathcal{N}(0, I)
$$

$$
z_t = (1 - t) z_0 + t\epsilon
$$

$$
u_t = \epsilon - z_0
$$

This notebook calls that the `noise_minus_data` convention. Some samplers and codebases reverse time or target sign. The opposite convention is:

$$
u_t = z_0 - \epsilon
$$

called `data_minus_noise` below. The correct sign should be checked empirically against the released SA3 sampler/model wrapper.

A native prompt/conditioning objective is:

$$
\mathcal{L}_{\mathrm{flow}}(c) =
\mathbb{E}_{t,\epsilon}
\left\| v_\theta(z_t, t, c) - u_t \right\|_2^2
$$

If this loss decreases, the conditioning better explains the target audio in SA3's own latent dynamics.

The exact target convention is an implementation detail. The research claim is not "SA3 must use this exact equation"; the claim is that frozen SA3 can be used as a native differentiable scorer for prompts/conditioning against SAME latents.


## Architecture Ontology

The notebook is not one generic method stack. It studies four model-native research layers separately before combining them. Evidence is the cross-layer review utility, not a fifth generative layer.

| Research layer | Native objects | Main question |
|---|---|---|
| SAME Representation Science | waveform \(x\), encoder \(E\), SAME latent \(z_0\), decoder \(D\), `LatentItem` | What does SAME preserve, expose, erase, or make editable on its own? |
| SA3 Flow and Conditioning Science | flow state \(z_t\), timestep/logSNR, prompt condition \(C(p)\), velocity \(v_\theta\) | What does frozen SA3 know through prompts, conditions, and flow timesteps? |
| SA3 Internal Trajectory Science | residual activations \(a_\ell\), sampler states, step windows | What do residuals and sampler states reveal or causally control? |
| SA3-over-SAME Coupled Editing | edited SAME latent \(z_0'\), SA3 polish/init/inpaint/continue paths | How does SA3 read, repair, erase, or amplify SAME latent edits? |

Evidence utilities collect descriptors, memory rows, player notes, manifests, disagreement rows, and ledger decisions. They review claims from all four research layers.

Placement rules:

```text
SAME-only claim: prove with direct decode / SAME rows first.
SA3-only claim: prove with flow, condition, residual, or trajectory rows first.
Coupled claim: compare direct SAME decode against SA3 polish.
Promoted claim: add descriptors, listening notes, and ledger decisions.
```


## Research Itinerary And Object Transitions

This notebook is organized as a bottom-up science instrument over native-object transitions. A workbench is not a claim by itself; it is where claims are tested. Evidence utilities review all layers.

| Sequence | Workbench | Native objects | Typical transitions | Main question |
|---:|---|---|---|---|
| 0 | Runtime and evidence setup | upstream handles, audio paths, descriptors, annotations, manifests | checkpoint -> model handle; output -> evidence packet | Are model assumptions explicit and are results reviewable? |
| 1 | Audio and SAME preparation | waveform \(x\), SAME latent \(z_0\), `LatentItem` | audio -> \(z_0\); \(z_0\) -> saved item | What exact native object enters each experiment? |
| 2 | SAME representation science | \(z_0\), summaries, geometry, lanes, direct decodes, memory rows | \(z_0\) -> measurements/selectors/edits/direct audio | What does SAME preserve, expose, erase, or make editable on its own? |
| 3 | SA3 flow and conditioning science | \(z_t\), timestep/logSNR, prompt condition \(C(p)\), flow field \(v_\theta\) | target \(z_0\) -> shared probe bank -> prompt/condition score | Which prompts or conditions explain audio under frozen SA3 flow? |
| 4 | SA3 internal trajectory science | residual activations \(a_\ell\), sampler states, guidance objectives | activation/state -> probe/vector/sweep -> output | Which internal signals are microscopes, selectors, or causal controls? |
| 5 | SA3-over-SAME coupled editing | edited SAME latents and SA3 polish/init paths | \(z_0\) -> \(z_0'\) -> direct decode / SA3 polish | What does SA3 preserve, repair, erase, or amplify from SAME edits? |
| 6 | External comparison and ledger | imported outputs, evidence packets, decisions | external output -> local packet; evidence -> promote/revise/drop | What is real, repeatable, useful, or only diagnostic? |

Claim maturity ladder:

```text
microscope              reveals structure, not a reliable control yet
selector                helps choose prompts, donors, seeds, chunks, channels, or recipes
intervention candidate  changes decoded/polished audio but lacks enough repeatability evidence
promoted method         survives baselines, measurement, audition, and repeated ledger decisions
```

Procedure honesty rule:

```text
root primitives define native objects and measurements
adapters isolate upstream SA3/SAME coupling
procedures execute SA3/SAME methods
evidence utilities support listening, annotation, disagreement, manifests, and decisions
maturity lives in docs, manifests, and ledger decisions
```

Canonical transitions:

```text
Runtime boundary: checkpoint/prompt/audio -> SA3/SAME handles
Evidence setup: output audio -> descriptor/player/annotation/manifest packet
Audio preparation: audio file -> SAME z_0 -> LatentItem
SAME representation: z_0 -> summaries, geometry, periodicity, control lanes, direct decode, memory rows
SA3 flow/conditioning: target z_0 -> flow probe bank -> prompt/condition ranking
SA3 internal trajectory: prompt/audio examples -> activations/states/vectors -> intervention sweeps
SA3-over-SAME coupled editing: z_0 -> edited z_0' -> direct decode or SA3 polish
External comparison: Underfit or other generated audio outputs -> same evidence packet as local runs
Ledger: evidence packet -> maturity update and next action
```


## Literature / Reference Map

Primary sources used for the design of this notebook:

| Area | Source | What it contributes here |
|---|---|---|
| SA3 architecture | [Stable Audio 3, Evans et al. 2026](https://arxiv.org/abs/2605.17991) | Variable-length latent diffusion for generation/editing, inpainting/continuation, SAME latent backbone, adversarial post-training. |
| SAME latent space | [SAME, Parker et al. 2026](https://arxiv.org/abs/2605.18613) | 4096x temporal compression, stereo audio autoencoder, semantic regularization, transformer backbone, SAME-L/S variants. |
| Released implementation | [Stability-AI/stable-audio-3](https://github.com/Stability-AI/stable-audio-3) | Official inference/training package, Medium requiring FlashAttention 2, model wrapper paths used by the adapters. |
| SA3 Medium model card | [stabilityai/stable-audio-3-medium](https://huggingface.co/stabilityai/stable-audio-3-medium) | Gated Medium weights, English text conditioning via T5Gemma, training-data/model-card details. |
| Flow matching | [Flow Matching, Lipman et al. 2022](https://arxiv.org/abs/2210.02747) | Vector-field regression view used by the prompt inversion objective. |
| Rectified flow | [Rectified Flow, Liu et al. 2022](https://arxiv.org/abs/2209.03003) | Straight-path data/noise transport intuition and low-step generation rationale. |
| Classifier guidance | [Diffusion Models Beat GANs, Dhariwal and Nichol 2021](https://arxiv.org/abs/2105.05233) | Gradient-based guidance idea: trade diversity/fidelity by adding external gradients. |
| Classifier-free guidance | [Classifier-Free Diffusion Guidance, Ho and Salimans 2022](https://arxiv.org/abs/2207.12598) | Conditional/unconditional score combination; motivates `cfg_scale` experiments. |
| Arbitrary guidance | [Universal Guidance, Bansal et al. 2023](https://arxiv.org/abs/2302.07121) | Conditioning with external guidance functions without retraining the base model. |
| Training-free guidance | [TFG, Ye et al. 2024](https://arxiv.org/abs/2409.15761) | Guidance design space and hyperparameter sensitivity; relevant to future direct sampler guidance. |
| Audio guidance gradients | [Controllable Music Production with Diffusion Models and Guidance Gradients, Levy et al. 2023](https://arxiv.org/abs/2311.00613) | Music continuation, inpainting, transitions, style transfer via sampling-time guidance. |
| Inference-time optimization | [DITTO, Novack et al. 2024](https://arxiv.org/abs/2401.12179) | Optimizing initial noise latents for control without fine-tuning. |
| LatCH | [Low-Resource Guidance for Controllable Latent Audio Diffusion, Novack et al. 2026](https://arxiv.org/abs/2603.04366) | Latent-control heads for cheap latent-space guidance over intensity, pitch, beats. |
| Live diffusion | [Live Music Diffusion Models, Novack et al. 2026](https://arxiv.org/abs/2605.22717) | Block-wise diffusion, KV-cache idea, ARC-Forcing, live interactive generation lens. |
| Hard prompt optimization | [PEZ / Hard Prompts Made Easy, Wen et al. 2023](https://arxiv.org/abs/2302.03668) | Soft-to-hard prompt discovery; adapted here only as search mechanics over frozen SA3 flow loss. |
| Residual steering | [Mood Vectors in Audio Diffusion, Camporese 2026](https://guglielmocamporese.github.io/blog/audio-mood-steering/) and [audioscope](https://github.com/guglielmocamporese/audioscope) | Non-peer-reviewed but directly relevant SA3 residual-stream steering method and torch.compile hook caveat. |
| Prior audio baselines | [MusicLM](https://arxiv.org/abs/2301.11325), [MusicGen](https://arxiv.org/abs/2306.05284), [AudioLDM](https://arxiv.org/abs/2301.12503), [Stable Audio Open](https://arxiv.org/abs/2407.14358) | Historical context for discrete-token music generation and latent-diffusion audio generation. |

Reading rule:

```text
SA3/SAME/HF/repo facts = confirmed source claims
prompt inversion / SAME style profile = experimental construction in this notebook
audioscope mood vectors = promising external experiment, not peer-reviewed ground truth
LMDM adaptation to SA3 = conceptual only unless SA3 sampler/attention code is modified and validated
```


## Confirmed Facts vs Notebook Assumptions

Confirmed from SA3/SAME/model-card sources:

```text
SA3 is a transformer-based latent diffusion family for variable-length audio generation/editing.
SA3 supports inpainting/continuation in the released pipeline.
SA3 Medium uses text conditioning through a public T5Gemma model according to the HF card.
SAME is a stereo audio autoencoder with 4096x temporal compression in the paper.
SAME-L and SAME-S are released; SA3 Medium is the target model for this Colab.
```

Notebook assumptions / experimental constructs:

```text
The flow-loss prompt inversion objective is a native surrogate, not a paper-claimed SA3 inversion method.
The velocity sign convention must be checked against the released wrapper.
SAME style profiles and SAME directions are latent edit hypotheses, not guaranteed perceptual controls.
Audio-derived residual vectors are a proposed extension of audioscope-style prompt vectors.
LatCH-style heads here are minimal scaffolds, not the full LatCH training recipe.
LMDM adaptation is conceptual unless SA3 attention/sampler code is modified and benchmarked.
```


## Research Program And Procedure Honesty Board

The notebook now treats methods as research programs over native objects, not as a historical list of modes. Each program has a layer, an operation role, and a promotion gate.

| Program | Research layer / utility | Operation role | Current maturity | Promotion gate |
|---|---|---|---|---|
| SAME bottleneck atlas | SAME representation | observe / compare | microscope | Direct-decode and descriptor/listening rows explain what SAME preserves or erases. |
| SAME memory and composition | SAME representation | select / compare | selector | Memory, lane, geometry, and bridge rankings improve donor/source/continuation auditions. |
| SAME latent DSP, blur, style, and graft primitives | SAME representation first; coupled editing when polished by SA3 | intervene / render | intervention candidate | Direct decode and SA3 polish packets separate survived, erased, amplified, and artifact edits. |
| Flow sign, timestep, attribution, and prompt scoring | SA3 flow/conditioning | observe / select | microscope / selector | Shared probe-bank flow rankings predict generated-audio or listening outcomes. |
| Hard/readable prompt search and semantic rows | SA3 flow/conditioning | select / compare | selector | Flow-found text remains readable and improves auditions versus baseline prompts. |
| Soft or null condition inversion | SA3 flow/conditioning | intervene / render | intervention candidate | Optimized conditions beat prompt and audio-to-audio baselines without losing reviewability. |
| Residual activation maps and feature atlases | SA3 internal trajectory | observe / select | microscope | Layer/time maps repeat across prompts, seeds, and examples. |
| Residual steering, cyclic trajectory, and guidance probes | SA3 internal trajectory | intervene / render | high-risk intervention candidate | Sweeps move target qualities predictably without sampler artifacts, collapse, or source copying. |
| SA3 polish, selective renoise, continuation, and rescue audits | SA3-over-SAME coupled editing | intervene / compare | intervention candidate | Source/baseline/method packets show what SA3 preserves, repairs, erases, or invents. |
| Evidence packets, disagreement rows, annotations, and ledger | evidence utility | decide | review infrastructure | Decisions change when evidence disagrees; repeated packets promote, revise, or drop methods. |

Procedure status is recorded in this research narrative and in ledger decisions. These are executable methods, not promoted claims:

| Procedure | Layer | Current maturity | Why it is not yet promoted |
|---|---|---|---|
| `flow_scoring.py` | SA3 flow/conditioning | microscope / selector | Needs predictive-validity runs linking flow rankings to generated audio and listening. |
| `soft_prompt.py` | SA3 flow/conditioning | intervention candidate | Needs baseline auditions proving optimized conditions are useful, not just low-loss. |
| `sa3_latent_sampling.py` | SA3-over-SAME coupled editing | intervention candidate | Needs direct-decode versus polish survival packets. |
| `selective_sa3.py` | SA3-over-SAME coupled editing | intervention candidate | Needs source/donor/copying checks across clips and seeds. |
| `cyclic_sa3.py` | SA3 internal trajectory | high-risk sampler microscope / candidate | Needs loop metrics and auditions without sampler-specific artifacts. |
| `residual_activation_vectors.py` | SA3 internal trajectory | microscope | Needs repeated layer maps before steering claims. |
| `audio_residual_vectors.py` | SA3 internal trajectory | high-risk microscope | Needs proof that audio-derived vectors are not source-identity or artifact leakage. |
| `residual_sweeps.py` | SA3 internal trajectory | high-risk candidate | Needs monotonic or interpretable alpha sweeps with bounded artifacts. |

Working rule: if a cell cannot say object, transition, operation, measurement, claim, and decision, keep it as a microscope or selector until the missing evidence exists.


In [ ]:
# @title 0. Runtime. Colab L4 runtime for upstream SA3 plus SA3 Native Lab

# Upstream Stable Audio 3 runtime plus this notebook/primitives repo.
SA3_REPO_URL = "https://github.com/Stability-AI/stable-audio-3.git"
LAB_REPO_URL = "https://github.com/sebastianrvhpk/sa3-native-lab.git"

# Defaults are ON for Stable Audio 3 Medium experiments.
CLONE_SA3_REPO = True
CLONE_LAB_REPO = True
INSTALL_SA3_REPO = True
INSTALL_REPO = True
INSTALL_TORCH_CU126 = True
INSTALL_FLASH_ATTN = True
USE_UV = True
PIN_NUMPY = True
REMOVE_TRANSFORMERS_OPTIONAL_SCIPY_SKLEARN = True
REMOVE_TRANSFORMERS_OPTIONAL_TORCHVISION = True
RESTART_RUNTIME_AFTER_FIRST_INSTALL = True
FORCE_FULL_SETUP = False

SA3_REPO_DIR = "/content/stable-audio-3"
PROJECT_DIR = "/content/sa3-native-lab"
SETUP_MARKER = "/content/.sa3_native_lab_setup_complete"
FLASH_ATTN_WHEEL_URL = ""  # Optional direct wheel URL matching Python/Torch/CUDA.
FLASH_ATTN_MAX_JOBS = "2"
FLASH_ATTN_FROM_SOURCE = False

import os
import sys
import subprocess
import shutil
import time
from pathlib import Path

os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")


def run(cmd, cwd=None, env=None, check=True):
    print("+", " ".join(str(part) for part in cmd))
    proc = subprocess.run(
        [str(part) for part in cmd],
        cwd=cwd,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(proc.stdout)
    if check and proc.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {proc.returncode}: "
            + " ".join(str(part) for part in cmd)
        )
    return proc


def ensure_uv():
    if not USE_UV:
        return
    if shutil.which("uv"):
        return
    run([sys.executable, "-m", "pip", "install", "-U", "uv"])


def install(args, env=None):
    args = [str(arg) for arg in args]
    if USE_UV:
        ensure_uv()
        uv_env = os.environ.copy()
        uv_env.setdefault("UV_LINK_MODE", "copy")
        if env:
            uv_env.update(env)
        return run(["uv", "pip", "install", "--system", *args], env=uv_env)
    return run([sys.executable, "-m", "pip", "install", *args], env=env)


def module_available(module_name):
    try:
        __import__(module_name)
        return True
    except Exception:
        return False


def verify_import(module_name, attr=None):
    import importlib

    module = importlib.import_module(module_name)
    value = getattr(module, attr) if attr else module
    location = getattr(module, "__file__", "<built-in>")
    version = getattr(module, "__version__", "unknown")
    print(f"OK import {module_name} version={version} path={location}")
    return value


def ensure_git_repo(path, url, *, clone=True, required=(), label="repo"):
    project = Path(path)
    has_project = (project / ".git").exists() and all((project / item).exists() for item in required)
    if has_project:
        if clone:
            run(["git", "-C", str(project), "pull", "--ff-only"], check=False)
        return project

    if clone:
        if not url:
            raise RuntimeError(f"Set the Git URL for {label}.")
        if project.exists():
            raise RuntimeError(
                f"{project} exists but is not a complete {label}. "
                "Delete it or set the matching path variable to a clean path."
            )
        run(["git", "clone", url, str(project)])
        if not all((project / item).exists() for item in required):
            raise RuntimeError(
                f"Cloned {url}, but it is missing required paths: "
                + ", ".join(required)
            )
        return project

    raise RuntimeError(f"{project} is not a complete {label} and cloning is disabled.")


def ensure_sa3_repo_dir():
    return ensure_git_repo(
        SA3_REPO_DIR,
        SA3_REPO_URL,
        clone=CLONE_SA3_REPO,
        required=("pyproject.toml", "stable_audio_3"),
        label="Stable Audio 3 repo",
    )


def ensure_project_dir():
    return ensure_git_repo(
        PROJECT_DIR,
        LAB_REPO_URL,
        clone=CLONE_LAB_REPO,
        required=("pyproject.toml", "latent_audio_primitives"),
        label="SA3 Native Lab repo",
    )


setup_marker = Path(SETUP_MARKER)
setup_complete = setup_marker.exists() and not FORCE_FULL_SETUP

if INSTALL_SA3_REPO:
    SA3_REPO_DIR = str(ensure_sa3_repo_dir())
    if SA3_REPO_DIR not in sys.path:
        sys.path.insert(0, SA3_REPO_DIR)

if INSTALL_REPO:
    PROJECT_DIR = str(ensure_project_dir())
    if PROJECT_DIR not in sys.path:
        sys.path.insert(0, PROJECT_DIR)

if setup_complete:
    print(f"Setup marker found: {SETUP_MARKER}")
    print("Skipping dependency installation. Delete the marker or set FORCE_FULL_SETUP=True to reinstall.")

if not setup_complete and INSTALL_TORCH_CU126:
    install(
        [
            "torch==2.7.1",
            "torchaudio==2.7.1",
            "--index-url",
            "https://download.pytorch.org/whl/cu126",
        ]
    )

if not setup_complete and INSTALL_FLASH_ATTN:
    if module_available("flash_attn"):
        print("flash_attn already importable; skipping FlashAttention install.")
    else:
        if USE_UV:
            ensure_uv()
            install(["-U", "setuptools", "wheel", "packaging", "psutil", "ninja"])
        else:
            run([sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel", "packaging", "psutil", "ninja"])
        env = os.environ.copy()
        env.setdefault("TORCH_CUDA_ARCH_LIST", "8.9")  # L4 / RTX 4090
        env.setdefault("MAX_JOBS", FLASH_ATTN_MAX_JOBS)
        if FLASH_ATTN_WHEEL_URL:
            install([FLASH_ATTN_WHEEL_URL], env=env)
        elif not FLASH_ATTN_FROM_SOURCE:
            # Fast path: allow flash-attn's setup to use a matching prebuilt
            # wheel when available for this Python/Torch/CUDA tuple.
            install(
                [
                    "flash-attn",
                    "--no-build-isolation",
                    "--no-cache-dir",
                    "--no-deps",
                ],
                env=env,
            )
        else:
            env.setdefault("FLASH_ATTENTION_FORCE_BUILD", "TRUE")
            env.setdefault("FLASH_ATTENTION_SKIP_CUDA_BUILD", "FALSE")
            install(
                [
                    "flash-attn",
                    "--no-build-isolation",
                    "--no-binary",
                    "flash-attn",
                    "--force-reinstall",
                    "--no-cache-dir",
                    "--no-deps",
                ],
                env=env,
            )
    run([sys.executable, "-c", "import flash_attn; from flash_attn import flash_attn_func; print('flash_attn', flash_attn.__version__)"])

if not setup_complete and INSTALL_SA3_REPO:
    install(["-e", SA3_REPO_DIR])

if not setup_complete and INSTALL_REPO:
    install(["-e", PROJECT_DIR])

if not setup_complete and PIN_NUMPY:
    # The upstream lower bound is numpy>=2.2.6. On Colab, resolving to newer
    # NumPy can leave optional scipy/sklearn stacks ABI-inconsistent. Pin the
    # minimum SA3 version before importing transformers/T5Gemma.
    install(["--force-reinstall", "numpy==2.2.6"])

if not setup_complete and REMOVE_TRANSFORMERS_OPTIONAL_SCIPY_SKLEARN:
    # Transformers imports scipy/sklearn opportunistically for generation/loss
    # utilities. SA3/T5Gemma does not need them, and Colab's scipy/sklearn wheels
    # can fail after the NumPy resolver changes above.
    run([sys.executable, "-m", "pip", "uninstall", "-y", "scipy", "scikit-learn", "sklearn"], check=False)

if not setup_complete and REMOVE_TRANSFORMERS_OPTIONAL_TORCHVISION:
    # Transformers also imports torchvision opportunistically from image_utils.
    # Colab's preinstalled torchvision is tied to its original Torch build and
    # can fail after pinning Torch to SA3's 2.7.1+cu126 requirement.
    run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"], check=False)

if not setup_complete:
    setup_marker.write_text("complete\n", encoding="utf-8")
    if RESTART_RUNTIME_AFTER_FIRST_INSTALL:
        print("\n=== first install complete ===")
        print("Restarting the Colab runtime now to clear old Torch/NumPy modules.")
        print("After Colab reconnects, rerun this cell. It will skip installs and verify imports.")
        time.sleep(2)
        os.kill(os.getpid(), 9)

if SA3_REPO_DIR not in sys.path:
    sys.path.insert(0, SA3_REPO_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print("\n=== setup verification ===")
verify_import("torch")
verify_import("torchaudio")
verify_import("flash_attn")
verify_import("stable_audio_3")
verify_import("latent_audio_primitives")
print("Setup complete. Continue with the next cell.")


In [ ]:
# @title 0. Runtime. Imports and experiment paths

from pathlib import Path
import json
import math
import random
import shutil
from dataclasses import asdict, dataclass

import numpy as np

try:
    import torch
    import torchaudio
except ImportError:
    torch = None
    torchaudio = None

# Core latent records, memory, and persistence
from latent_audio_primitives.schema import LatentItem
from latent_audio_primitives.index import LatentMemoryIndex
from latent_audio_primitives.io import save_items, load_items
from latent_audio_primitives.latent_math import latent_summary

# Model boundary adapters
from latent_audio_primitives.adapters.stable_audio3 import (
    SAMEAutoencoderAdapter,
    StableAudio3Adapter,
    audio_chunk_windows,
    latents_to_items,
)
from latent_audio_primitives.adapters.sa3_residual_hooks import SteeringVectors, ResidualSteerer

# Evidence instruments and decoded-audio measurement
from latent_audio_primitives.evidence.audio_player import display_audio_player
from latent_audio_primitives.evidence.annotations import search_audio_annotations
from latent_audio_primitives.evidence.disagreement import (
    disagreement_rows_to_table,
    make_disagreement_row,
    rank_disagreement_rows,
)
from latent_audio_primitives.audio_descriptors import audio_descriptor_report, descriptor_delta

# Prompt inversion and prompt search primitives
from latent_audio_primitives.prompt_optimization import (
    beam_token_prompt_search,
    coordinate_prompt_search,
    default_modifier_axes,
    greedy_token_prompt_search,
    prompt_seed_from_audio_path,
)
from latent_audio_primitives.flow_prompt import (
    flow_probe_bank_from_values,
    flow_probe_bank_to_manifest,
    flow_velocity_target,
    prompt_leave_one_out_attribution,
    summarize_flow_loss_rows,
    timesteps_from_logsnr_values,
)
from latent_audio_primitives.prompt_semantics import (
    make_prompt_variants,
    prompt_semantic_rows,
    prompt_variant_manifest,
    rank_prompt_semantic_rows,
    semantic_tag_counts,
)
from latent_audio_primitives.tokenizer_vocab import (
    native_tokenizer_vocabulary,
    preview_native_tokenizer_vocabulary,
)
from latent_audio_primitives.prompt_pairs import pairs_for_axis

# SA3/SAME executable procedures
from latent_audio_primitives.procedures.flow_scoring import (
    sa3_flow_loss_rows_for_prompts,
    sa3_flow_losses_for_prompts,
)
from latent_audio_primitives.procedures.soft_prompt import (
    SoftPromptState,
    optimize_soft_prompt_from_latents,
    generate_with_soft_prompt,
)
from latent_audio_primitives.procedures.sa3_latent_sampling import sa3_sample_from_init_latents
from latent_audio_primitives.procedures.selective_sa3 import selective_graft_sa3, selective_renoise_sa3
from latent_audio_primitives.procedures.cyclic_sa3 import sa3_cyclic_roll_sample_from_init_latents
from latent_audio_primitives.procedures.residual_activation_vectors import SA3ActivationVectorExtractor
from latent_audio_primitives.procedures.audio_residual_vectors import SA3AudioResidualVectorExtractor
from latent_audio_primitives.procedures.residual_sweeps import alpha_sweep

# Latent operators and interventions
from latent_audio_primitives.latent_blur import (
    LatentBlurSpec,
    apply_latent_blur,
)
from latent_audio_primitives.latent_dsp import (
    LatentDSPSpec,
    apply_latent_dsp,
    latent_change_report,
)
from latent_audio_primitives.selective_renoise import (
    LatentMaskSpec,
    graft_latent_channels,
    masked_latent_noise,
    select_latent_channels,
)
from latent_audio_primitives.looping import (
    cyclic_roll_audio,
    cyclic_roll_latents,
    frames_from_fraction,
    loop_boundary_metrics,
    repeated_loop_preview_audio,
    samples_from_fraction,
    seam_inpaint_bounds,
)
from latent_audio_primitives.guidance import gradient_guidance_step
from latent_audio_primitives.style import (
    fit_style_profile,
    style_direction,
    apply_profile_attraction,
    apply_profile_to_item,
    apply_style_direction,
    save_style_profile,
    load_style_profile,
    save_style_direction,
)
from latent_audio_primitives.composition import ranked_continuations, ranked_bridges, best_path

# Measurement, control, geometry, and dataset workflow
from latent_audio_primitives.geometry import (
    covariance_transport,
    fit_latent_geometry,
    geometry_report,
    latent_barycenter,
    mahalanobis_summary_distance,
)
from latent_audio_primitives.periodic import periodicity_report
from latent_audio_primitives.observability import fit_linear_control_probe, predict_control
from latent_audio_primitives.residual_features import fit_residual_feature_basis, project_residual_features
from latent_audio_primitives.control_lanes import (
    audio_envelope_lane,
    control_lane_svg,
    latent_channel_energy_lane,
    latent_motion_lane,
    normalize_control_lane,
    save_control_lanes,
)
from latent_audio_primitives.curriculum import (
    build_memory_curriculum,
    heldout_split_item_ids,
    nearest_memory_rows,
)

DEVICE = "cuda" if torch is not None and torch.cuda.is_available() else "cpu"
WORK_DIR = Path("/content/sa3_native_science_lab")
INPUT_DIR = WORK_DIR / "inputs"
OUTPUT_DIR = WORK_DIR / "outputs"
MEMORY_DIR = WORK_DIR / "memory"
VECTOR_DIR = WORK_DIR / "vectors"
FLOW_TARGET_CONVENTION = "noise_minus_data"  # Try "data_minus_noise" if the diagnostic favors it.

for directory in [INPUT_DIR, OUTPUT_DIR, MEMORY_DIR, VECTOR_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("device:", DEVICE)
print("work dir:", WORK_DIR)
print("flow target convention:", FLOW_TARGET_CONVENTION)


In [ ]:
# @title 0. Runtime. Hugging Face login and model loading

LOAD_MODELS = True
HF_LOGIN = True

SA3_MODEL_NAME = "medium"
SAME_MODEL_NAME = "same-l"  # SA3 Medium uses SAME-Large style latents.
MODEL_HALF = True

sa3_model = None
sa3 = None
same = None

if HF_LOGIN:
    from huggingface_hub import login
    login()

if LOAD_MODELS:
    if torch is None:
        raise RuntimeError("PyTorch is required before loading SA3/SAME.")
    import importlib
    import sys
    from pathlib import Path

    def import_stable_audio_model():
        importlib.invalidate_caches()
        try:
            from stable_audio_3 import StableAudioModel
            return StableAudioModel
        except ModuleNotFoundError as first_error:
            repo_dir = Path(globals().get("SA3_REPO_DIR", "/content/stable-audio-3"))
            source_dir = repo_dir / "stable_audio_3"
            if source_dir.exists():
                sys.path.insert(0, str(repo_dir))
                importlib.invalidate_caches()
                try:
                    from stable_audio_3 import StableAudioModel
                    print(f"Imported stable_audio_3 from source checkout: {repo_dir}")
                    return StableAudioModel
                except ModuleNotFoundError:
                    pass
            raise ModuleNotFoundError(
                "stable_audio_3 is not importable in this notebook kernel. "
                "Rerun cell 0 with SA3_REPO_URL set and let it finish, "
                "or run `uv pip install --system -e /content/stable-audio-3`. "
                f"Checked source path: {source_dir}"
            ) from first_error

    StableAudioModel = import_stable_audio_model()

    sa3_model = StableAudioModel.from_pretrained(
        SA3_MODEL_NAME,
        device=DEVICE,
        model_half=MODEL_HALF,
    )
    sa3 = StableAudio3Adapter(model=sa3_model, model_name=SA3_MODEL_NAME)
    same = SAMEAutoencoderAdapter.from_pretrained(SAME_MODEL_NAME, device=DEVICE)
    print("SA3 sample rate:", sa3.sample_rate)
    print("SA3 latent rate:", sa3.latent_rate)
else:
    print("Models not loaded. Set LOAD_MODELS=True when running on Colab L4.")


def require_models():
    if sa3_model is None or sa3 is None or same is None:
        raise RuntimeError("Set LOAD_MODELS=True and run the model-loading cell first.")


In [ ]:
# @title 0. Runtime. SA3 Medium smoke test

RUN_SMOKE_TEST = True
SMOKE_PROMPT = "a sparse glassy ambient loop, slow evolving texture"
SMOKE_DURATION = 4.0
SMOKE_STEPS = 4
SMOKE_SEED = 42

if RUN_SMOKE_TEST:
    require_models()
    latents = sa3.generate_latents(
        prompt=SMOKE_PROMPT,
        duration=SMOKE_DURATION,
        steps=SMOKE_STEPS,
        cfg_scale=1.0,
        seed=SMOKE_SEED,
    )
    print("smoke latents:", tuple(latents.shape), latents.dtype, latents.device)
    smoke_audio = sa3.decode_latents(latents).float().clamp(-1, 1).cpu()
    smoke_path = OUTPUT_DIR / "sa3_medium_smoke.wav"
    torchaudio.save(str(smoke_path), smoke_audio[0], sa3.sample_rate)
    print("saved:", smoke_path)
    try:
        display_audio_player([smoke_path], title="SA3 Medium smoke test")
    except Exception as exc:
        print("Custom audio player skipped:", exc)


## 0. Runtime. Shared Helpers

These helpers keep the notebook focused on experiments:

- audio file -> SAME latent tensor
- SA3 tensor \(B \times C \times T\) -> `LatentItem` memory format \(T \times D\)
- time-major latent \(T \times D\) -> SA3 channel-first tensor \(B \times C \times T\)
- prompt -> SA3 flow loss against target audio latent


In [ ]:
# @title 0. Runtime. Audio, latent, decode, and native flow-loss helpers

AUDIO_EXTS = {".wav", ".flac", ".mp3", ".ogg", ".m4a", ".aiff", ".aif"}


def list_audio_files(directory, limit=None):
    paths = sorted(
        path for path in Path(directory).rglob("*")
        if path.suffix.lower() in AUDIO_EXTS
    )
    return paths[:limit] if limit else paths


def load_audio(path, target_sample_rate=None, duration=None, stereo=True):
    if torchaudio is None:
        raise RuntimeError("torchaudio is required.")
    audio, sample_rate = torchaudio.load(str(path))
    if not stereo and audio.shape[0] > 1:
        audio = audio.mean(dim=0, keepdim=True)
    if target_sample_rate and sample_rate != target_sample_rate:
        audio = torchaudio.functional.resample(audio, sample_rate, target_sample_rate)
        sample_rate = target_sample_rate
    if duration is not None:
        target_samples = int(round(duration * sample_rate))
        if audio.shape[-1] < target_samples:
            pad = target_samples - audio.shape[-1]
            audio = torch.nn.functional.pad(audio, (0, pad))
        else:
            audio = audio[..., :target_samples]
    return audio, sample_rate


def item_to_sa3_tensor(item, device=None, dtype=None):
    if torch is None:
        raise RuntimeError("PyTorch is required.")
    arr = np.asarray(item.latent, dtype=np.float32).T[None, :, :]
    return torch.from_numpy(arr).to(device=device or DEVICE, dtype=dtype or torch.float32)


def sa3_tensor_to_item(latents, item_id="latent", prompt=None, metadata=None):
    require_models()
    return latents_to_items(
        latents,
        item_id_prefix=item_id,
        latent_rate=sa3.latent_rate,
        sample_rate=sa3.sample_rate,
        prompt=prompt,
        metadata=metadata or {},
    )[0]


def decode_sa3_latents_to_file(latents, path):
    require_models()
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with torch.inference_mode():
        audio = sa3.decode_latents(latents).float().clamp(-1, 1).cpu()
    torchaudio.save(str(path), audio[0], sa3.sample_rate)
    return path


def decode_sa3_latents_to_file_cropped(latents, path, *, duration=None):
    require_models()
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with torch.inference_mode():
        audio = sa3.decode_latents(latents).float().clamp(-1, 1).cpu()
    if duration is not None:
        audio = audio[..., : int(round(duration * sa3.sample_rate))]
    torchaudio.save(str(path), audio[0], sa3.sample_rate)
    return path


def decode_sa3_latents_to_file_with_descriptors(latents, path, *, duration=None):
    path = decode_sa3_latents_to_file_cropped(latents, path, duration=duration)
    audio, sample_rate = load_audio(path, stereo=True)
    return path, audio_descriptor_report(audio, sample_rate)


def encode_audio_file_to_item(path, item_id=None, prompt=None, duration=None):
    require_models()
    audio, sample_rate = load_audio(path, duration=duration, stereo=True)
    return same.encode_tensor(
        audio,
        sample_rate,
        item_id=item_id or Path(path).stem,
        prompt=prompt,
        chunked=False,
        metadata={"path": str(path)},
    )


def encode_audio_file_to_items(
    path,
    *,
    item_id=None,
    prompt=None,
    duration=None,
    use_chunks=False,
    chunk_duration=None,
    hop_duration=None,
    max_chunks=None,
    drop_last=False,
):
    require_models()
    path = Path(path)
    if use_chunks:
        chunk_duration = chunk_duration or duration
        if chunk_duration is None:
            raise ValueError("chunk_duration or duration is required when use_chunks=True.")
        return same.encode_file_chunks(
            path,
            chunk_duration=chunk_duration,
            hop_duration=hop_duration,
            max_chunks=max_chunks,
            drop_last=drop_last,
            item_id_prefix=item_id or path.stem,
            prompt=prompt,
            chunked=False,
            metadata={"path": str(path), "encoding_mode": "chunked_folder_window"},
        )
    return [encode_audio_file_to_item(path, item_id=item_id, prompt=prompt, duration=duration)]


def encode_audio_folder_to_items(
    directory,
    *,
    limit=None,
    duration=None,
    prompt_from_path=True,
    use_chunks=False,
    chunk_duration=None,
    hop_duration=None,
    max_chunks_per_file=None,
    drop_last=False,
):
    items = []
    for path in list_audio_files(directory, limit=limit):
        prompt = prompt_seed_from_audio_path(path) if prompt_from_path else None
        file_items = encode_audio_file_to_items(
            path,
            item_id=path.stem,
            prompt=prompt,
            duration=duration,
            use_chunks=use_chunks,
            chunk_duration=chunk_duration,
            hop_duration=hop_duration,
            max_chunks=max_chunks_per_file,
            drop_last=drop_last,
        )
        items.extend(file_items)
        print(f"{path.name}: {len(file_items)} item(s)")
    print(f"encoded {len(items)} latent item(s) from {directory}")
    return items


def preview_audio_folder_chunk_plan(
    directory,
    *,
    limit=None,
    chunk_duration=12.0,
    hop_duration=None,
    max_chunks_per_file=None,
    drop_last=False,
):
    if torchaudio is None:
        raise RuntimeError("torchaudio is required.")
    plan = []
    for path in list_audio_files(directory, limit=limit):
        info = torchaudio.info(str(path))
        windows = audio_chunk_windows(
            int(info.num_frames),
            int(info.sample_rate),
            chunk_duration,
            hop_duration=hop_duration,
            max_chunks=max_chunks_per_file,
            drop_last=drop_last,
        )
        plan.append(
            {
                "path": str(path),
                "source_seconds": round(info.num_frames / info.sample_rate, 3),
                "chunks": len(windows),
                "first_start": round(windows[0]["start_seconds"], 3) if windows else None,
                "last_start": round(windows[-1]["start_seconds"], 3) if windows else None,
            }
        )
    return plan


def pad_or_crop_channel_first(tensor, target_frames):
    if tensor.shape[-1] == target_frames:
        return tensor
    if tensor.shape[-1] > target_frames:
        return tensor[..., :target_frames]
    pad = target_frames - tensor.shape[-1]
    return torch.nn.functional.pad(tensor, (0, pad))


def items_to_latent_batch(items, *, target_frames=None, device=None, dtype=None):
    if not items:
        raise ValueError("items list is empty.")
    tensors = [item_to_sa3_tensor(item, device=device or DEVICE, dtype=dtype or torch.float32) for item in items]
    target_frames = target_frames or min(t.shape[-1] for t in tensors)
    tensors = [pad_or_crop_channel_first(t, target_frames) for t in tensors]
    return torch.cat(tensors, dim=0)


def freeze_sa3_and_same():
    require_models()
    for module in [sa3_model.model, same.autoencoder]:
        if hasattr(module, "parameters"):
            for parameter in module.parameters():
                parameter.requires_grad_(False)
        if hasattr(module, "eval"):
            module.eval()


def compare_flow_velocity_conventions(stable_model, target_latents, prompts, *, duration, seed=0):
    return {
        convention: sa3_flow_losses_for_prompts(
            stable_model,
            target_latents,
            prompts,
            duration=duration,
            seed=seed,
            velocity_convention=convention,
        )
        for convention in ["noise_minus_data", "data_minus_noise"]
    }



def has_probe_values(values):
    if values is None:
        return False
    if isinstance(values, str):
        return bool(values.strip())
    return len(values) > 0


def build_flow_probe_bank_from_controls(
    *,
    logsnr_values=None,
    timestep_values=None,
    seed=0,
    velocity_convention=None,
    antithetic_noise=False,
    shared_noise=True,
    metadata=None,
):
    if has_probe_values(timestep_values):
        return flow_probe_bank_from_values(
            timestep_values=timestep_values,
            seed=seed,
            velocity_convention=velocity_convention or FLOW_TARGET_CONVENTION,
            antithetic_noise=antithetic_noise,
            shared_noise=shared_noise,
            metadata=metadata,
        )
    if has_probe_values(logsnr_values):
        return flow_probe_bank_from_values(
            logsnr_values=logsnr_values,
            seed=seed,
            velocity_convention=velocity_convention or FLOW_TARGET_CONVENTION,
            antithetic_noise=antithetic_noise,
            shared_noise=shared_noise,
            metadata=metadata,
        )
    return None


def flow_probe_manifest_or_none(probe_bank):
    return None if probe_bank is None else flow_probe_bank_to_manifest(probe_bank)


## 0. Evidence Utilities. Custom Colab Audio Player

The notebook uses a small self-contained Web Audio/canvas player instead of `IPython.display.Audio` for generated outputs.

It supports:

```text
playlist comparison
waveform seek
track loop
loop-region in/out points
volume and speed
keyboard shortcuts after clicking the player
```

The audio is embedded into the notebook output as base64, so keep playlists short when files are long.


In [ ]:
# @title 0. Evidence Utilities. Audio player helper

def play_audio_files(
    paths,
    *,
    labels=None,
    metadata=None,
    title="Audio Player",
    peak_count=900,
    max_embed_mb=96.0,
    annotation_path=None,
):
    return display_audio_player(
        paths,
        labels=labels,
        metadata=metadata,
        title=title,
        peak_count=peak_count,
        max_embed_mb=max_embed_mb,
        annotation_path=annotation_path,
    )


## 0. Runtime. Dataset / Long-File Input Policy

For folders of long recordings, set `DATASET_USE_CHUNKS=True`.

Without chunking, each file is encoded as one item and `DATASET_DURATION=8.0` means "use the first 8 seconds of each file." With chunking, each long file becomes many SAME latent items:

$$
x_{\mathrm{file}} \rightarrow \{x_{[0,L]}, x_{[H,H+L]}, x_{[2H,2H+L]}, \ldots\}
$$

and then:

$$
z_i = E(x_i)
$$

This is the right default for DJ sets, live sets, mixtapes, stems, and archives where the interesting distribution is spread across time.


In [ ]:
# @title 0. Runtime. Dataset / long-file chunking controls

DATASET_DIR = INPUT_DIR / "dataset_positive"
REFERENCE_DIR = INPUT_DIR / "dataset_reference"

DATASET_DURATION = 8.0
DATASET_LIMIT = 2  # number of files before chunking; raise after the pipeline works

DATASET_USE_CHUNKS = True
DATASET_CHUNK_DURATION = 12.0
DATASET_HOP_DURATION = 12.0  # use 6.0 for 50% overlap on 12s chunks
DATASET_MAX_CHUNKS_PER_FILE = 4  # L4-safe default for optimization experiments
DATASET_DROP_LAST_CHUNK = False

RUN_DATASET_CHUNK_PREVIEW = False


def dataset_effective_duration():
    return DATASET_CHUNK_DURATION if DATASET_USE_CHUNKS else DATASET_DURATION


if RUN_DATASET_CHUNK_PREVIEW:
    plan = preview_audio_folder_chunk_plan(
        DATASET_DIR,
        limit=DATASET_LIMIT,
        chunk_duration=DATASET_CHUNK_DURATION,
        hop_duration=DATASET_HOP_DURATION,
        max_chunks_per_file=DATASET_MAX_CHUNKS_PER_FILE,
        drop_last=DATASET_DROP_LAST_CHUNK,
    )
    print(json.dumps(plan, indent=2))


## 0. Evidence Utilities. Annotation Retrieval

When the channel-selective renoise player is shown in Colab, each track has a small annotation panel:

```text
rating      numeric score, e.g. 0-5 or -5..5
tags        comma-separated labels such as drums, smear, keeper, bad-transient
use/value   a short role like loop, transition, texture, break
description free listening notes
```

The player saves these into:

```python
OUTPUT_DIR / "same_selective_renoise" / "annotations.json"
```

This turns listening into a retrieval layer over SA3 interventions: you can search notes/tags,
re-open the best files, and inspect the metadata for the corresponding channel mask.


In [ ]:
# @title 0. Evidence Utilities. Search annotated variations

RUN_EVIDENCE_ANNOTATION_RETRIEVAL = False

ANNOTATION_SEARCH_PATH = OUTPUT_DIR / "same_selective_renoise" / "annotations.json"
ANNOTATION_QUERY = ""
ANNOTATION_TAGS = []  # Example: ["keeper", "drums"]
ANNOTATION_MIN_RATING = None  # Example: 4.0
ANNOTATION_LIMIT = 24

if RUN_EVIDENCE_ANNOTATION_RETRIEVAL:
    matches = search_audio_annotations(
        ANNOTATION_SEARCH_PATH,
        query=ANNOTATION_QUERY,
        tags=ANNOTATION_TAGS,
        min_rating=ANNOTATION_MIN_RATING,
        limit=ANNOTATION_LIMIT,
    )
    print("matches:", len(matches))
    for item in matches:
        meta = item.get("metadata", {})
        print(
            item.get("rating"),
            item.get("tags"),
            item.get("label"),
            "variant=", meta.get("variant"),
            "kind=", meta.get("kind"),
            "channels=", meta.get("selected_channels", [])[:12],
        )
    if matches:
        play_audio_files(
            [item["path"] for item in matches],
            labels=[
                f"{item.get('rating')} | {', '.join(item.get('tags', []))} | {item.get('label')}"
                for item in matches
            ],
            metadata=[item.get("metadata", {}) for item in matches],
            title="Evidence Packet Setup: Annotation Retrieval",
            max_embed_mb=160.0,
            annotation_path=ANNOTATION_SEARCH_PATH,
        )


## 0. Evidence Utilities. Active Evidence Instruments

These cells are first-class instruments across workbenches:

```text
flow attribution -> loss-by-timestep panel -> control lanes -> memory curriculum
-> latent OT bench -> bridge search -> residual atlas -> null-condition probe
-> guidance-gradient latent editing -> posterior-guided audio edit -> baselines
```

All experiment cells default to off. Each cell writes JSON/audio artifacts under
`OUTPUT_DIR` and uses the existing custom audio player when audio is produced.
Treat each output as an evidence packet candidate: it still needs baseline,
measurement, audition, and a ledger decision before promotion.


In [ ]:
# @title 0. Evidence Utilities. SAME control lanes

RUN_EVIDENCE_CONTROL_LANES = False

CONTROL_LANE_AUDIO = INPUT_DIR / "source.wav"
CONTROL_LANE_DURATION = 12.0
CONTROL_LANE_FRAME_SECONDS = 0.10
CONTROL_LANE_OUTPUT = OUTPUT_DIR / "evidence_control_lanes.json"
CONTROL_LANE_SVG = OUTPUT_DIR / "evidence_control_lanes.svg"


if RUN_EVIDENCE_CONTROL_LANES:
    require_models()
    source_item = encode_audio_file_to_item(
        CONTROL_LANE_AUDIO,
        item_id="evidence_control_lane_source",
        duration=CONTROL_LANE_DURATION,
    )
    audio, sample_rate = load_audio(
        CONTROL_LANE_AUDIO,
        target_sample_rate=sa3.sample_rate,
        duration=CONTROL_LANE_DURATION,
        stereo=True,
    )
    lanes = [
        normalize_control_lane(audio_envelope_lane(audio, sample_rate, frame_seconds=CONTROL_LANE_FRAME_SECONDS, name="rms_envelope"), mode="minmax"),
        normalize_control_lane(audio_envelope_lane(audio, sample_rate, frame_seconds=CONTROL_LANE_FRAME_SECONDS, name="spectral_centroid_hz"), mode="minmax"),
        normalize_control_lane(latent_motion_lane(source_item, latent_rate=source_item.latent_rate, source=str(CONTROL_LANE_AUDIO)), mode="minmax"),
        normalize_control_lane(latent_channel_energy_lane(source_item, latent_rate=source_item.latent_rate, source=str(CONTROL_LANE_AUDIO)), mode="minmax"),
    ]
    save_control_lanes(lanes, CONTROL_LANE_OUTPUT)
    svg = control_lane_svg(lanes)
    CONTROL_LANE_SVG.write_text(svg, encoding="utf-8")
    try:
        from IPython.display import HTML, display

        display(HTML(svg))
    except Exception:
        print(svg[:2000])
    print("saved:", CONTROL_LANE_OUTPUT)


## 0. Evidence Utilities. Semantic Bottleneck Disagreement

Use this panel when flow fit, SAME movement, decoded descriptors, and listening notes point in different directions. Disagreement is reviewed inside the local evidence packet, using SA3/SAME-native measurements and listening notes.


In [ ]:
# @title 0. Evidence Utilities. Semantic bottleneck disagreement panel

RUN_SEMANTIC_BOTTLENECK_DISAGREEMENT = False

SEMANTIC_DISAGREEMENT_ROWS = [
    {
        "item_id": "example_prompt_variant",
        "label": "replace with a prompt, donor, or output id",
        "same_distance": None,
        "same_neighbor_score": None,
        "flow_loss": None,
        "descriptor_delta_norm": None,
        "listening_rating": None,
        "decision": "review",
        "notes": "Fill this row from prompt transparency, SAME geometry, descriptor deltas, and listening notes.",
    }
]
SEMANTIC_DISAGREEMENT_OUTPUT = OUTPUT_DIR / "semantic_bottleneck_disagreement.json"


if RUN_SEMANTIC_BOTTLENECK_DISAGREEMENT:
    rows = [
        make_disagreement_row(
            row["item_id"],
            label=row.get("label", ""),
            same_distance=row.get("same_distance"),
            same_neighbor_score=row.get("same_neighbor_score"),
            flow_loss=row.get("flow_loss"),
            descriptor_delta_norm_value=row.get("descriptor_delta_norm"),
            listening_rating=row.get("listening_rating"),
            decision=row.get("decision", ""),
            notes=row.get("notes", ""),
        )
        for row in SEMANTIC_DISAGREEMENT_ROWS
    ]
    ranked_rows = rank_disagreement_rows(rows)
    payload = {
        "rows": disagreement_rows_to_table(ranked_rows),
        "evidence_lanes": ["same_distance", "same_neighbor_score", "flow_loss", "descriptor_delta_norm", "listening_rating"],
        "excluded_lanes": ["external_text_audio_embedding", "external_latent_guidance"],
    }
    SEMANTIC_DISAGREEMENT_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    SEMANTIC_DISAGREEMENT_OUTPUT.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    try:
        import pandas as pd

        display(pd.DataFrame(payload["rows"]))
    except Exception:
        print(json.dumps(payload, indent=2)[:6000])
    print("saved:", SEMANTIC_DISAGREEMENT_OUTPUT)


## 1. SAME Representation Science. SAME Geometry and Bottleneck Audit

This is the measurement layer for native SAME representation before it becomes a control claim.

It does not try to make a new sound first. It asks what is measurable in the
current latent collection:

```text
latent geometry:     PCA spectrum, covariance, Mahalanobis distances
periodicity:         autocorrelation lags, boundary mismatch, latent FFT centroid
transport:           full-covariance Gaussian style transport probe
observability:       simple linear probes for existing descriptors/labels
```

The reason is methodological: before steering a control, check whether it is
visible in SAME latents and whether simple operators behave coherently.


In [ ]:
# @title 1. SAME Representation Science. SAME geometry and bottleneck audit

RUN_SAME_GEOMETRY_AUDIT = False

GEOMETRY_AUDIT_OUTPUT = OUTPUT_DIR / "same_geometry_audit.json"
GEOMETRY_AUDIT_SOURCE = "dataset"  # "dataset" or "memory"
GEOMETRY_AUDIT_COMPONENTS = 16
GEOMETRY_AUDIT_PERIOD_MIN_LAG = 2
GEOMETRY_AUDIT_PERIOD_MAX_LAG = 128
GEOMETRY_AUDIT_CONTROL_KEYS = []  # Example: ["brightness", "density", "usable_loop"]
GEOMETRY_AUDIT_TRANSPORT_DEMO = False


def load_geometry_audit_items():
    if GEOMETRY_AUDIT_SOURCE == "memory":
        return load_items(MEMORY_DIR)
    if GEOMETRY_AUDIT_SOURCE == "dataset":
        require_models()
        return encode_audio_folder_to_items(
            DATASET_DIR,
            limit=DATASET_LIMIT,
            duration=DATASET_DURATION,
            use_chunks=DATASET_USE_CHUNKS,
            chunk_duration=DATASET_CHUNK_DURATION,
            hop_duration=DATASET_HOP_DURATION,
            max_chunks_per_file=DATASET_MAX_CHUNKS_PER_FILE,
            drop_last=DATASET_DROP_LAST_CHUNK,
        )
    raise ValueError("GEOMETRY_AUDIT_SOURCE must be 'dataset' or 'memory'")


if RUN_SAME_GEOMETRY_AUDIT:
    items = load_geometry_audit_items()
    if len(items) < 2:
        raise ValueError("Geometry audit needs at least two latent items.")

    geometry = fit_latent_geometry(items, n_components=GEOMETRY_AUDIT_COMPONENTS)
    report = geometry_report(items, n_components=GEOMETRY_AUDIT_COMPONENTS)
    periodic = []
    for item in items:
        p = periodicity_report(
            item,
            min_lag=GEOMETRY_AUDIT_PERIOD_MIN_LAG,
            max_lag=GEOMETRY_AUDIT_PERIOD_MAX_LAG,
        )
        periodic.append(
            {
                "item_id": item.item_id,
                "best_lag": p.best_lag,
                "best_score": p.best_score,
                "boundary_l2": p.boundary_l2,
                "velocity_l2": p.velocity_l2,
                "spectral_centroid": p.spectral_centroid,
            }
        )

    distances = []
    anchor = items[0]
    for item in items[1:]:
        distances.append(
            {
                "from": anchor.item_id,
                "to": item.item_id,
                "mahalanobis_summary_distance": mahalanobis_summary_distance(anchor, item, geometry),
            }
        )

    probes = []
    for key in GEOMETRY_AUDIT_CONTROL_KEYS:
        try:
            probe = fit_linear_control_probe(items, key)
            predictions = [
                {"item_id": item.item_id, "prediction": predict_control(probe, item)}
                for item in items
            ]
            probes.append(
                {
                    "control": key,
                    "item_count": probe.item_count,
                    "train_r2": probe.r2_train,
                    "predictions": predictions,
                }
            )
        except Exception as exc:
            probes.append({"control": key, "error": str(exc)})

    transport_demo = None
    if GEOMETRY_AUDIT_TRANSPORT_DEMO and len(items) >= 2:
        transported = covariance_transport(items[0], items[1:], alpha=1.0)
        transport_demo = {
            "source": items[0].item_id,
            "reference_count": len(items) - 1,
            "source_mean_norm": float(np.linalg.norm(items[0].latent.mean(axis=0))),
            "transported_mean_norm": float(np.linalg.norm(transported.mean(axis=0))),
        }

    payload = {
        "item_count": len(items),
        "source": GEOMETRY_AUDIT_SOURCE,
        "geometry": report,
        "pca_variances": geometry.variances.astype(float).tolist(),
        "periodicity": periodic,
        "anchor_distances": distances,
        "control_probes": probes,
        "transport_demo": transport_demo,
    }
    GEOMETRY_AUDIT_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    GEOMETRY_AUDIT_OUTPUT.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(json.dumps(payload, indent=2)[:6000])
    print("saved:", GEOMETRY_AUDIT_OUTPUT)


## 1b. SAME Memory And Composition. Dataset Prompt Family

If a dataset is not one thing, cluster it in SAME summary space first.

Summary used here:

$$
s(z) =
\left[
\operatorname{mean}_t(z),
\operatorname{std}_t(z),
\operatorname{mean}_t|\Delta z|
\right]
$$

Then optimize one prompt or soft prompt per cluster:

$$
c_k^\star = \arg\min_c \mathbb{E}_{i \in k} \mathcal{L}_{\mathrm{flow}}(z_i, c)
$$


In [ ]:
# @title 1b. SAME Memory And Composition. Dataset -> prompt family by SAME clustering

RUN_DATASET_PROMPT_FAMILY = False

PROMPT_FAMILY_DIR = OUTPUT_DIR / "dataset_prompt_family"
CLUSTERS = 3


def kmeans_numpy(x, k, iterations=50, seed=0):
    rng = np.random.default_rng(seed)
    if len(x) < k:
        raise ValueError("not enough examples for k clusters.")
    centers = x[rng.choice(len(x), size=k, replace=False)].copy()
    labels = np.zeros(len(x), dtype=np.int64)
    for _ in range(iterations):
        distances = np.linalg.norm(x[:, None, :] - centers[None, :, :], axis=-1)
        labels = distances.argmin(axis=1)
        for cluster in range(k):
            members = x[labels == cluster]
            if len(members):
                centers[cluster] = members.mean(axis=0)
    return labels, centers


if RUN_DATASET_PROMPT_FAMILY:
    require_models()
    PROMPT_FAMILY_DIR.mkdir(parents=True, exist_ok=True)
    run_duration = dataset_effective_duration()
    dataset_items = encode_audio_folder_to_items(
        DATASET_DIR,
        limit=DATASET_LIMIT,
        duration=DATASET_DURATION,
        prompt_from_path=True,
        use_chunks=DATASET_USE_CHUNKS,
        chunk_duration=DATASET_CHUNK_DURATION,
        hop_duration=DATASET_HOP_DURATION,
        max_chunks_per_file=DATASET_MAX_CHUNKS_PER_FILE,
        drop_last=DATASET_DROP_LAST_CHUNK,
    )
    summaries = np.stack([latent_summary(item) for item in dataset_items])
    labels, centers = kmeans_numpy(summaries, CLUSTERS, seed=2)

    manifest = []
    for cluster_id in range(CLUSTERS):
        cluster_items = [item for item, label in zip(dataset_items, labels) if label == cluster_id]
        if not cluster_items:
            continue
        cluster_latents = items_to_latent_batch(
            cluster_items,
            device=DEVICE,
            dtype=next(sa3_model.model.model.parameters()).dtype,
        )
        state = optimize_soft_prompt_from_latents(
            sa3_model,
            cluster_latents,
            seed_prompt=f"audio cluster {cluster_id} texture",
            duration=run_duration,
            optimization_steps=SOFT_STEPS,
            lr=SOFT_LR,
            train_keys=("prompt",),
            seed=cluster_id,
            velocity_convention=FLOW_TARGET_CONVENTION,
        )
        path = PROMPT_FAMILY_DIR / f"cluster_{cluster_id:02d}_soft_prompt.pt"
        state.save(path)
        manifest.append({"cluster": cluster_id, "items": [item.item_id for item in cluster_items], "path": str(path)})

    (PROMPT_FAMILY_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(json.dumps(manifest, indent=2))


## 1b. SAME Memory And Composition. Continuation and Inpainting as Composition

Boundary-conditioned latent generation:

$$
p_\theta(z_{\mathrm{missing}} \mid z_{\mathrm{known}}, m, c)
$$

Mask merge:

$$
z = m \odot z_{\mathrm{known}} + (1 - m) \odot z_{\mathrm{generated}}
$$

Composition operators:

```text
continue after this sound
fill a hole
bridge A to B
extend a loop
replace a section while preserving boundaries
```


In [ ]:
# @title 1b. SAME Memory And Composition. Continuation / inpainting composition

RUN_DATASET_CONTINUATION_COMPOSITION = False

SOURCE_AUDIO = INPUT_DIR / "source.wav"
CONTINUATION_PROMPT = "continue this texture into a coherent evolving loop"
SOURCE_DURATION = 8.0
TARGET_CONTINUATION_DURATION = 16.0

if RUN_DATASET_CONTINUATION_COMPOSITION:
    require_models()
    audio, sample_rate = load_audio(SOURCE_AUDIO, duration=SOURCE_DURATION, stereo=True)
    latents = sa3.continuation_latents(
        prompt=CONTINUATION_PROMPT,
        inpaint_audio=(sample_rate, audio),
        source_duration=SOURCE_DURATION,
        target_duration=TARGET_CONTINUATION_DURATION,
        steps=12,
        cfg_scale=1.0,
        seed=300,
    )
    out_path = OUTPUT_DIR / "dataset_continuation.wav"
    decode_sa3_latents_to_file(latents, out_path)
    print("saved:", out_path)
    play_audio_files([out_path], title="SAME Memory And Composition: Continuation Composition")


## 1b. SAME Memory And Composition. Latent Memory Instrument

Memory item:

```text
{
  latent: z,
  summary: s(z),
  prompt,
  descriptors,
  labels,
  audio path
}
```

Composition loop:

```text
retrieve -> continue -> steer -> decode -> re-encode -> store
```

This turns generated and encoded audio into reusable material.


In [ ]:
# @title 1b. SAME Memory And Composition. Latent memory instrument

RUN_DATASET_MEMORY_INDEX = False

MEMORY_TEST_PROMPTS = [
    "a sparse glassy ambient loop, slow evolving texture",
    "a dense shimmering granular loop with bright high frequency motion",
    "a dark low drone with distant metallic harmonics",
]

if RUN_DATASET_MEMORY_INDEX:
    require_models()
    items = []
    for prompt_index, prompt in enumerate(MEMORY_TEST_PROMPTS):
        for seed in [0, 1]:
            generated = sa3.generate_items(
                prompt=prompt,
                duration=8.0,
                item_id_prefix=f"p{prompt_index}_seed{seed}",
                steps=8,
                cfg_scale=1.0,
                seed=seed,
            )
            items.extend(generated)
    save_items(items, MEMORY_DIR)
    index = LatentMemoryIndex(items)
    query_item = items[0]
    results = index.query(query_item, top_k=5, exclude_id=query_item.item_id)
    for result in results:
        print(result.item_id, result.score, result.item.prompt)


In [ ]:
# @title 1b. SAME Memory And Composition. Dataset memory as prompt/control curriculum

RUN_DATASET_MEMORY_CURRICULUM = False

MEMORY_CURRICULUM_SOURCE = "memory"  # "memory" or "dataset"
MEMORY_CURRICULUM_CLUSTERS = 4
MEMORY_CURRICULUM_DATASET_LIMIT = DATASET_LIMIT
MEMORY_CURRICULUM_HOLDOUT_FRACTION = 0.2
MEMORY_CURRICULUM_QUERY_AUDIO = INPUT_DIR / "query.wav"
MEMORY_CURRICULUM_RUN_QUERY = False
MEMORY_CURRICULUM_OUTPUT = OUTPUT_DIR / "dataset_memory_curriculum.json"


def memory_curriculum_load_items():
    if MEMORY_CURRICULUM_SOURCE == "memory":
        return load_items(MEMORY_DIR)
    if MEMORY_CURRICULUM_SOURCE == "dataset":
        require_models()
        return encode_audio_folder_to_items(
            DATASET_DIR,
            limit=MEMORY_CURRICULUM_DATASET_LIMIT,
            duration=DATASET_DURATION,
            use_chunks=DATASET_USE_CHUNKS,
            chunk_duration=DATASET_CHUNK_DURATION,
            hop_duration=DATASET_HOP_DURATION,
            max_chunks_per_file=DATASET_MAX_CHUNKS_PER_FILE,
            drop_last=DATASET_DROP_LAST_CHUNK,
        )
    raise ValueError("MEMORY_CURRICULUM_SOURCE must be 'memory' or 'dataset'")


if RUN_DATASET_MEMORY_CURRICULUM:
    items = memory_curriculum_load_items()
    if len(items) < 1:
        raise ValueError("Memory curriculum needs at least one latent memory item.")
    clusters = build_memory_curriculum(
        items,
        cluster_count=MEMORY_CURRICULUM_CLUSTERS,
        seed=0,
    )
    split = heldout_split_item_ids(clusters, holdout_fraction=MEMORY_CURRICULUM_HOLDOUT_FRACTION)
    nearest_rows = []
    if MEMORY_CURRICULUM_RUN_QUERY:
        require_models()
        query_item = encode_audio_file_to_item(MEMORY_CURRICULUM_QUERY_AUDIO, item_id="dataset_memory_curriculum_query", duration=DATASET_DURATION)
        nearest_rows = nearest_memory_rows(query_item, items, top_k=8)
    payload = {
        "source": MEMORY_CURRICULUM_SOURCE,
        "item_count": len(items),
        "clusters": [cluster.to_dict() for cluster in clusters],
        "split": split,
        "nearest_query_rows": nearest_rows,
    }
    MEMORY_CURRICULUM_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    MEMORY_CURRICULUM_OUTPUT.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    try:
        import pandas as pd

        display(pd.DataFrame([{k: v for k, v in row.items() if k != "centroid"} for row in payload["clusters"]]))
        if nearest_rows:
            display(pd.DataFrame(nearest_rows))
    except Exception:
        print(json.dumps(payload, indent=2)[:6000])
    print("saved:", MEMORY_CURRICULUM_OUTPUT)


In [ ]:
# @title 1b. SAME Memory And Composition. Continuation as bridge search

RUN_DATASET_BRIDGE_SEARCH = False

BRIDGE_SEARCH_SOURCE = "memory"
BRIDGE_SEARCH_START_ID = ""
BRIDGE_SEARCH_END_ID = ""
BRIDGE_SEARCH_TOP_K = 12
BRIDGE_SEARCH_OUTPUT = OUTPUT_DIR / "dataset_bridge_search.json"


if RUN_DATASET_BRIDGE_SEARCH:
    items = load_items(MEMORY_DIR) if BRIDGE_SEARCH_SOURCE == "memory" else memory_curriculum_load_items()
    if len(items) < 3:
        raise ValueError("Bridge search needs at least three memory items.")
    start = next((item for item in items if item.item_id == BRIDGE_SEARCH_START_ID), items[0])
    end = next((item for item in items if item.item_id == BRIDGE_SEARCH_END_ID), items[-1])
    continuation_rows = [
        {"item_id": item.item_id, "cost": float(cost), "prompt": item.prompt or ""}
        for item, cost in ranked_continuations(start, items, k=8)[:BRIDGE_SEARCH_TOP_K]
    ]
    bridge_rows = [
        {"item_id": item.item_id, "cost": float(cost), "prompt": item.prompt or ""}
        for item, cost in ranked_bridges(start, end, items, k=8)[:BRIDGE_SEARCH_TOP_K]
    ]
    try:
        path_items, path_cost = best_path(items, start.item_id, end.item_id, max_hops=3)
        path_row = {"cost": float(path_cost), "item_ids": [item.item_id for item in path_items]}
    except Exception as exc:
        path_row = {"error": str(exc)}
    payload = {
        "start_id": start.item_id,
        "end_id": end.item_id,
        "continuations": continuation_rows,
        "bridges": bridge_rows,
        "best_path": path_row,
    }
    BRIDGE_SEARCH_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    BRIDGE_SEARCH_OUTPUT.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    try:
        import pandas as pd

        display(pd.DataFrame(continuation_rows))
        display(pd.DataFrame(bridge_rows))
    except Exception:
        print(json.dumps(payload, indent=2)[:6000])
    print("saved:", BRIDGE_SEARCH_OUTPUT)


## 2. SA3 Flow And Conditioning Science. Flow Sign Diagnostic

Before doing serious prompt inversion, test both velocity conventions on one target audio and several candidate prompts.

This does not prove the full sampler convention, but it catches a common failure case: optimizing against a sign-flipped target.

If one convention consistently gives lower loss for human-plausible prompts and better optimization behavior, set:

```python
FLOW_TARGET_CONVENTION = "noise_minus_data"
```

or:

```python
FLOW_TARGET_CONVENTION = "data_minus_noise"
```


In [ ]:
# @title 2. SA3 Flow And Conditioning Science. Optional flow sign diagnostic

RUN_FLOW_SIGN_DIAGNOSTIC = False

DIAGNOSTIC_AUDIO = INPUT_DIR / "target.wav"
DIAGNOSTIC_DURATION = 8.0
DIAGNOSTIC_PROMPTS = [
    "audio texture",
    "a dense bright electronic loop with shimmering harmonics",
    "a sparse dark ambient drone with low sustained tones",
]

if RUN_FLOW_SIGN_DIAGNOSTIC:
    require_models()
    diagnostic_item = encode_audio_file_to_item(
        DIAGNOSTIC_AUDIO,
        item_id="diagnostic_target",
        duration=DIAGNOSTIC_DURATION,
    )
    diagnostic_latents = item_to_sa3_tensor(
        diagnostic_item,
        device=DEVICE,
        dtype=next(sa3_model.model.model.parameters()).dtype,
    )
    comparison = compare_flow_velocity_conventions(
        sa3_model,
        diagnostic_latents,
        DIAGNOSTIC_PROMPTS,
        duration=DIAGNOSTIC_DURATION,
        seed=0,
    )
    print(json.dumps({k: [round(v, 6) for v in values] for k, values in comparison.items()}, indent=2))
    print("Current FLOW_TARGET_CONVENTION:", FLOW_TARGET_CONVENTION)


## 2. SA3 Flow And Conditioning Science. Soft Prompt Inversion

Object optimized:

$$
c
$$

Frozen:

```text
SAME encoder/decoder
SA3 DiT / flow model
SA3 text conditioner weights
```

Objective:

$$
c^\star = \arg\min_c
\mathbb{E}_{t,\epsilon}
\left\| v_\theta(z_t, t, c) - u_t \right\|_2^2
$$

where \(u_t\) is selected by `FLOW_TARGET_CONVENTION`.

This produces a continuous conditioning preset:

```text
audio.wav -> soft_prompt.pt
```

It is an optimized SA3 conditioning state, distinct from natural-language prompt text and SAME audio latents:

```text
soft_prompt.pt =
  conditioning dicts
  optimized conditioning tensors
  loss curve
  metadata
```

Use the soft-prompt audition cell to hear what the `.pt` does. Use hard or readable prompt search if you want an actual text prompt.


In [ ]:
# @title 2. SA3 Flow And Conditioning Science. Audio -> soft prompt

RUN_FLOW_SOFT_PROMPT_INVERSION = False

TARGET_AUDIO = INPUT_DIR / "target.wav"
SOFT_PROMPT_PATH = OUTPUT_DIR / "flow_conditioning_soft_prompt.pt"

SEED_PROMPT = "experimental audio texture"
TARGET_DURATION = 8.0
SOFT_STEPS = 100
SOFT_LR = 1e-2

if RUN_FLOW_SOFT_PROMPT_INVERSION:
    require_models()
    freeze_sa3_and_same()
    target_item = encode_audio_file_to_item(TARGET_AUDIO, item_id="target", duration=TARGET_DURATION)
    target_latents = item_to_sa3_tensor(
        target_item,
        device=DEVICE,
        dtype=next(sa3_model.model.model.parameters()).dtype,
    )
    soft_state = optimize_soft_prompt_from_latents(
        sa3_model,
        target_latents,
        seed_prompt=SEED_PROMPT,
        duration=TARGET_DURATION,
        optimization_steps=SOFT_STEPS,
        lr=SOFT_LR,
        train_keys=("prompt",),
        reg_weight=1e-4,
        seed=0,
        velocity_convention=FLOW_TARGET_CONVENTION,
    )
    soft_state.save(SOFT_PROMPT_PATH)
    print("saved:", SOFT_PROMPT_PATH)
    print("initial/final loss:", soft_state.losses[0], soft_state.losses[-1])


## 2. SA3 Flow And Conditioning Science. Soft Prompt Audition

This consumes the `.pt` from soft prompt inversion:

$$
c^\star \rightarrow \operatorname{sample}_\theta(\epsilon, c^\star) \rightarrow z \rightarrow D(z)
$$

The output should be judged like an instrument preset, not like a readable prompt.


In [ ]:
# @title 2. SA3 Flow And Conditioning Science. Generate audio from a saved soft prompt

RUN_FLOW_SOFT_PROMPT_AUDITION = False

SOFT_PROMPT_GENERATION_DIR = OUTPUT_DIR / "flow_conditioning_soft_prompt_audition"
SOFT_PROMPT_GENERATION_STEPS = 8
SOFT_PROMPT_GENERATION_CFG = 1.0
SOFT_PROMPT_GENERATION_SEEDS = [10, 11, 12]

if RUN_FLOW_SOFT_PROMPT_AUDITION:
    require_models()
    SOFT_PROMPT_GENERATION_DIR.mkdir(parents=True, exist_ok=True)
    soft_state = SoftPromptState.load(SOFT_PROMPT_PATH)
    manifest = []
    for seed in SOFT_PROMPT_GENERATION_SEEDS:
        latents = generate_with_soft_prompt(
            sa3_model,
            soft_state,
            steps=SOFT_PROMPT_GENERATION_STEPS,
            cfg_scale=SOFT_PROMPT_GENERATION_CFG,
            seed=seed,
            return_latents=True,
        )
        out_path = SOFT_PROMPT_GENERATION_DIR / f"soft_prompt_seed_{seed}.wav"
        decode_sa3_latents_to_file(latents, out_path)
        manifest.append({"path": str(out_path), "seed": int(seed)})
        print("saved:", out_path)
    (SOFT_PROMPT_GENERATION_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    try:
        display_audio_player(
            [entry["path"] for entry in manifest],
            labels=[f"soft prompt seed {entry['seed']}" for entry in manifest],
            title="SA3 Flow And Conditioning Science: Soft Prompt Audition Generations",
        )
    except Exception as exc:
        print("Custom audio player skipped:", exc)


## 2. SA3 Flow And Conditioning Science. Hard Prompt Search

Object optimized:

$$
p = [w_1,\dots,w_N]
$$

Objective:

$$
p^\star = \arg\min_p \mathcal{L}_{\mathrm{flow}}(C(p))
$$

The result may be linguistically strange. That is acceptable if it reliably moves SA3 into the target audio region.

By default this cell uses candidate text pieces extracted from the tokenizer that SA3's T5Gemma prompt conditioner actually owns.

Important distinction:

```text
native tokenizer vocabulary = text/subword pieces seen by T5Gemma
SAME latent space           = continuous audio latents generated by SA3
```

So this is native to the text-conditioning path, not a symbolic music vocabulary.

Search-cost rule:

```text
search width per position      = HARD_PROMPT_TOKEN_SUBSET
beam width                    = HARD_PROMPT_BEAM_WIDTH
GPU prompts per forward chunk  = HARD_PROMPT_CANDIDATE_BATCH_SIZE
positions per run              = HARD_PROMPT_TOKENS_GENERATED
noise/timestep samples averaged= HARD_PROMPT_SCORE_SAMPLES
```

Better-math defaults:

- fixed shared log-SNR timestep/noise probe bank, so candidates are compared fairly
- antithetic noise pairs \(\epsilon, -\epsilon\), which reduce Monte Carlo variance
- normalized flow residuals, so a high-energy probe does not dominate just by scale
- beam search, so a useful token can survive even if it is not the greedy winner at one position
- MSE plus optional cosine direction loss over the SA3 velocity field
- optional conditional-delta score, comparing the prompt-specific vector-field change against the target-specific residual from the null prompt
- chunked GPU scoring, so you can raise search width without needing all candidates in VRAM at once

This is frozen-model prompt inversion through SA3 flow agreement. Good
results may be strange strings if those strings steer SA3's native conditioner.


In [ ]:
# @title 2. SA3 Flow And Conditioning Science. Audio -> flow-scored hard prompt

RUN_FLOW_HARD_PROMPT_SEARCH = False

USE_SA3_TOKENIZER_VOCAB = True
HARD_PROMPT_TOKENIZER_VOCAB_LIMIT = 16384
HARD_PROMPT_TOKENIZER_SCAN_LIMIT = 65536
HARD_PROMPT_REQUIRE_WORD_START = True
HARD_PROMPT_MIN_CHARS = 3
HARD_PROMPT_REJECT_STOPWORDS = True
HARD_PROMPT_AUDIO_PRIOR = True

# Search strategy:
#   "beam"   = best default; keeps multiple partial prompts alive.
#   "greedy" = fast approximate method.
HARD_PROMPT_SEARCH_STRATEGY = "beam"

# Search compute knobs. For beam search:
#   branch_factor is sampled per live beam at every token position.
#   beam_width keeps this many partial prompts after each position.
HARD_PROMPT_TOKEN_SUBSET = 256          # greedy width, or fallback beam branch factor
HARD_PROMPT_BEAM_WIDTH = 4
HARD_PROMPT_BRANCH_FACTOR = 256
HARD_PROMPT_CANDIDATE_BATCH_SIZE = 128
HARD_PROMPT_TOKENS_GENERATED = 16
HARD_PROMPT_GREEDY_RUNS = 6

# Native SA3 flow score. If HARD_PROMPT_LOGSNR_VALUES is non-empty, it becomes the
# timestep probe bank using t = sigmoid(-logSNR).
# These values span clean-ish, mid, and noisy regimes.
HARD_PROMPT_LOGSNR_VALUES = [2.0, 0.0, -2.0]
HARD_PROMPT_TIMESTEP_VALUES = []  # Manual override. Example: [0.12, 0.32, 0.55, 0.78]
HARD_PROMPT_SCORE_SAMPLES = 4
HARD_PROMPT_SHARED_NOISE = True
HARD_PROMPT_ANTITHETIC_NOISE = True
HARD_PROMPT_NORMALIZE_MSE = True
HARD_PROMPT_T_MIN = 0.05
HARD_PROMPT_T_MAX = 0.95
HARD_PROMPT_SCORE_SEED = 123
HARD_PROMPT_COSINE_WEIGHT = 0.25

# Expensive but conceptually useful. 0.0 disables it. Try 0.10-0.25 if you want
# to score the prompt-specific vector-field contribution relative to null text.
HARD_PROMPT_CONDITIONAL_DELTA_WEIGHT = 0.0

FALLBACK_HARD_PROMPT_VOCAB = [
    "shimmer", "velvet", "glass", "grain", "pressure", "hollow", "dense", "thin",
    "wide", "mono", "dust", "chrome", "choir", "fold", "pulse", "drift",
    "warm", "cold", "broken", "slow", "fast", "tape", "bright", "dark",
    "sub", "spark", "rubber", "metal", "wet", "dry", "attack", "bloom",
    "loop", "riser", "drop", "intro", "ghost", "close", "distant", "aura",
]

HARD_PROMPT_JSON = OUTPUT_DIR / "flow_conditioning_hard_prompt_search.json"

if RUN_FLOW_HARD_PROMPT_SEARCH:
    require_models()
    if USE_SA3_TOKENIZER_VOCAB:
        hard_prompt_vocab = native_tokenizer_vocabulary(
            sa3_model,
            max_candidates=HARD_PROMPT_TOKENIZER_VOCAB_LIMIT,
            scan_limit=HARD_PROMPT_TOKENIZER_SCAN_LIMIT,
            min_chars=HARD_PROMPT_MIN_CHARS,
            require_word_start=HARD_PROMPT_REQUIRE_WORD_START,
            ascii_only=True,
            lowercase=True,
            reject_stopwords=HARD_PROMPT_REJECT_STOPWORDS,
            rank_by_audio_prior=HARD_PROMPT_AUDIO_PRIOR,
        )
        print(f"Using {len(hard_prompt_vocab)} native SA3/T5Gemma tokenizer-derived candidates.")
        print(preview_native_tokenizer_vocabulary(hard_prompt_vocab, columns=8, rows=4))
    else:
        hard_prompt_vocab = FALLBACK_HARD_PROMPT_VOCAB
        print(f"Using {len(hard_prompt_vocab)} fallback hand-written candidates.")

    strategy = HARD_PROMPT_SEARCH_STRATEGY.lower()
    hard_prompt_probe_bank = build_flow_probe_bank_from_controls(
        logsnr_values=HARD_PROMPT_LOGSNR_VALUES,
        timestep_values=HARD_PROMPT_TIMESTEP_VALUES,
        seed=HARD_PROMPT_SCORE_SEED,
        velocity_convention=FLOW_TARGET_CONVENTION,
        antithetic_noise=HARD_PROMPT_ANTITHETIC_NOISE,
        shared_noise=HARD_PROMPT_SHARED_NOISE,
        metadata={"bench": "hard_prompt_search"},
    )
    resolved_timesteps = [probe.timestep for probe in hard_prompt_probe_bank.probes] if hard_prompt_probe_bank else []
    effective_score_samples = len(hard_prompt_probe_bank.probes) if hard_prompt_probe_bank else HARD_PROMPT_SCORE_SAMPLES
    antithetic_multiplier = 1 if hard_prompt_probe_bank else (2 if HARD_PROMPT_ANTITHETIC_NOISE else 1)
    null_multiplier = 2 if HARD_PROMPT_CONDITIONAL_DELTA_WEIGHT else 1
    greedy_width = min(HARD_PROMPT_TOKEN_SUBSET or len(hard_prompt_vocab), len(hard_prompt_vocab))
    branch_factor = min(HARD_PROMPT_BRANCH_FACTOR or greedy_width, len(hard_prompt_vocab))
    gpu_batch = HARD_PROMPT_CANDIDATE_BATCH_SIZE or greedy_width
    if strategy == "beam":
        candidates_per_position = HARD_PROMPT_BEAM_WIDTH * branch_factor
        approx_forward_chunks = (
            HARD_PROMPT_TOKENS_GENERATED
            * math.ceil(candidates_per_position / gpu_batch)
            * effective_score_samples
            * antithetic_multiplier
            * null_multiplier
        )
        approx_prompt_scores = (
            HARD_PROMPT_TOKENS_GENERATED
            * candidates_per_position
            * effective_score_samples
            * antithetic_multiplier
        )
    else:
        candidates_per_position = greedy_width
        approx_forward_chunks = (
            HARD_PROMPT_GREEDY_RUNS
            * HARD_PROMPT_TOKENS_GENERATED
            * math.ceil(greedy_width / gpu_batch)
            * effective_score_samples
            * antithetic_multiplier
            * null_multiplier
        )
        approx_prompt_scores = (
            HARD_PROMPT_GREEDY_RUNS
            * HARD_PROMPT_TOKENS_GENERATED
            * greedy_width
            * effective_score_samples
            * antithetic_multiplier
        )
    print(
        "Hard prompt search budget:",
        {
            "strategy": strategy,
            "greedy_width": greedy_width,
            "beam_width": HARD_PROMPT_BEAM_WIDTH if strategy == "beam" else None,
            "branch_factor": branch_factor if strategy == "beam" else None,
            "candidates_per_position": candidates_per_position,
            "gpu_batch": gpu_batch,
            "tokens_generated": HARD_PROMPT_TOKENS_GENERATED,
            "greedy_runs": HARD_PROMPT_GREEDY_RUNS if strategy == "greedy" else None,
            "score_samples": effective_score_samples,
            "logsnr_values": HARD_PROMPT_LOGSNR_VALUES if HARD_PROMPT_LOGSNR_VALUES and not HARD_PROMPT_TIMESTEP_VALUES else None,
            "timesteps": resolved_timesteps if resolved_timesteps else "random",
            "antithetic_noise": HARD_PROMPT_ANTITHETIC_NOISE,
            "normalize_mse": HARD_PROMPT_NORMALIZE_MSE,
            "cosine_weight": HARD_PROMPT_COSINE_WEIGHT,
            "conditional_delta_weight": HARD_PROMPT_CONDITIONAL_DELTA_WEIGHT,
            "approx_prompt_scores": approx_prompt_scores,
            "approx_dit_forwards": approx_forward_chunks,
        },
    )

    target_item = encode_audio_file_to_item(TARGET_AUDIO, item_id="target", duration=TARGET_DURATION)
    target_latents = item_to_sa3_tensor(
        target_item,
        device=DEVICE,
        dtype=next(sa3_model.model.model.parameters()).dtype,
    )

    def batch_scorer(prompts):
        losses = sa3_flow_losses_for_prompts(
            sa3_model,
            target_latents,
            prompts,
            duration=TARGET_DURATION,
            seed=HARD_PROMPT_SCORE_SEED,
            min_t=HARD_PROMPT_T_MIN,
            max_t=HARD_PROMPT_T_MAX,
            score_samples=HARD_PROMPT_SCORE_SAMPLES,
            shared_noise=HARD_PROMPT_SHARED_NOISE,
            timestep_values=None if hard_prompt_probe_bank else (resolved_timesteps or None),
            cosine_weight=HARD_PROMPT_COSINE_WEIGHT,
            antithetic_noise=False if hard_prompt_probe_bank else HARD_PROMPT_ANTITHETIC_NOISE,
            normalize_mse=HARD_PROMPT_NORMALIZE_MSE,
            conditional_delta_weight=HARD_PROMPT_CONDITIONAL_DELTA_WEIGHT,
            velocity_convention=FLOW_TARGET_CONVENTION,
            probe_bank=hard_prompt_probe_bank,
        )
        return [-loss for loss in losses]

    if strategy == "beam":
        result = beam_token_prompt_search(
            hard_prompt_vocab,
            batch_scorer,
            tokens_generated=HARD_PROMPT_TOKENS_GENERATED,
            beam_width=HARD_PROMPT_BEAM_WIDTH,
            branch_factor=branch_factor,
            candidate_batch_size=HARD_PROMPT_CANDIDATE_BATCH_SIZE,
            seed=0,
            higher_is_better=True,
        )
    elif strategy == "greedy":
        result = greedy_token_prompt_search(
            hard_prompt_vocab,
            batch_scorer,
            tokens_generated=HARD_PROMPT_TOKENS_GENERATED,
            runs=HARD_PROMPT_GREEDY_RUNS,
            token_subset=HARD_PROMPT_TOKEN_SUBSET,
            candidate_batch_size=HARD_PROMPT_CANDIDATE_BATCH_SIZE,
            seed=0,
            higher_is_better=True,
        )
    else:
        raise ValueError("HARD_PROMPT_SEARCH_STRATEGY must be 'beam' or 'greedy'")

    payload = {
        "prompt": result.prompt,
        "score": result.score,
        "tokens": result.tokens,
        "strategy": strategy,
        "vocab_source": "sa3_t5gemma_tokenizer" if USE_SA3_TOKENIZER_VOCAB else "fallback_hand_vocab",
        "vocab_size": len(hard_prompt_vocab),
        "greedy_width": greedy_width,
        "beam_width": HARD_PROMPT_BEAM_WIDTH if strategy == "beam" else None,
        "branch_factor": branch_factor if strategy == "beam" else None,
        "candidate_batch_size": HARD_PROMPT_CANDIDATE_BATCH_SIZE,
        "tokens_generated": HARD_PROMPT_TOKENS_GENERATED,
        "greedy_runs": HARD_PROMPT_GREEDY_RUNS if strategy == "greedy" else None,
        "score_samples": effective_score_samples,
        "flow_probe_bank": flow_probe_manifest_or_none(hard_prompt_probe_bank),
        "shared_noise": HARD_PROMPT_SHARED_NOISE,
        "logsnr_values": HARD_PROMPT_LOGSNR_VALUES if HARD_PROMPT_LOGSNR_VALUES and not HARD_PROMPT_TIMESTEP_VALUES else None,
        "timesteps": resolved_timesteps if resolved_timesteps else None,
        "t_range": [HARD_PROMPT_T_MIN, HARD_PROMPT_T_MAX] if not resolved_timesteps else None,
        "antithetic_noise": HARD_PROMPT_ANTITHETIC_NOISE,
        "normalize_mse": HARD_PROMPT_NORMALIZE_MSE,
        "cosine_weight": HARD_PROMPT_COSINE_WEIGHT,
        "conditional_delta_weight": HARD_PROMPT_CONDITIONAL_DELTA_WEIGHT,
        "final_beams": getattr(result, "beams", None),
        "history_tail": result.history[-20:],
    }
    HARD_PROMPT_JSON.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(result.prompt)
    print("score:", result.score)
    print("saved:", HARD_PROMPT_JSON)


## 2. SA3 Flow And Conditioning Science. Readable Prompt Search

This constrains search to interpretable descriptors.

Same native score:

$$
\text{score}(p) = -\mathcal{L}_{\mathrm{flow}}(C(p))
$$

Tradeoff:

```text
readable prompt = easier to use
opaque hard prompt = often a stronger conditioning key
soft prompt     = strongest, least readable
```


In [ ]:
# @title 2. SA3 Flow And Conditioning Science. Audio -> readable prompt

RUN_FLOW_READABLE_PROMPT_SEARCH = False

READABLE_PROMPT_LOGSNR_VALUES = [2.0, 0.0, -2.0]
READABLE_PROMPT_SEED = 321
READABLE_PROMPT_ANTITHETIC_NOISE = True
READABLE_PROMPT_COSINE_WEIGHT = 0.25
READABLE_PROMPT_JSON = OUTPUT_DIR / "flow_conditioning_readable_prompt_search.json"

if RUN_FLOW_READABLE_PROMPT_SEARCH:
    require_models()
    target_item = encode_audio_file_to_item(TARGET_AUDIO, item_id="target", duration=TARGET_DURATION)
    target_latents = item_to_sa3_tensor(
        target_item,
        device=DEVICE,
        dtype=next(sa3_model.model.model.parameters()).dtype,
    )
    seed_prompt = prompt_seed_from_audio_path(TARGET_AUDIO, extra_tags=["audio texture"])
    readable_probe_bank = build_flow_probe_bank_from_controls(
        logsnr_values=READABLE_PROMPT_LOGSNR_VALUES,
        seed=READABLE_PROMPT_SEED,
        velocity_convention=FLOW_TARGET_CONVENTION,
        antithetic_noise=READABLE_PROMPT_ANTITHETIC_NOISE,
        shared_noise=True,
        metadata={"bench": "readable_prompt_search"},
    )

    def scorer(prompt):
        return -sa3_flow_losses_for_prompts(
            sa3_model,
            target_latents,
            [prompt],
            duration=TARGET_DURATION,
            seed=READABLE_PROMPT_SEED,
            normalize_mse=True,
            cosine_weight=READABLE_PROMPT_COSINE_WEIGHT,
            velocity_convention=FLOW_TARGET_CONVENTION,
            probe_bank=readable_probe_bank,
        )[0]

    result = coordinate_prompt_search(
        seed_prompt,
        default_modifier_axes(),
        scorer,
        rounds=3,
    )
    payload = {
        "prompt": result.prompt,
        "score": result.score,
        "seed_prompt": seed_prompt,
        "flow_probe_bank": flow_probe_manifest_or_none(readable_probe_bank),
        "history_tail": result.history[-20:],
    }
    READABLE_PROMPT_JSON.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(result.prompt)
    print("score:", result.score)
    print("saved:", READABLE_PROMPT_JSON)


## 2. SA3 Flow And Conditioning Science. Dataset Soft Prompt

Object optimized:

$$
c
$$

Target set:

$$
\{z_i = E(x_i)\}_{i=1}^N
$$

Objective:

$$
c^\star =
\arg\min_c
\mathbb{E}_{i,t,\epsilon}
\left\| v_\theta(z_{i,t}, t, c) - u_{i,t} \right\|_2^2
$$

This asks: what single conditioning key makes SA3 explain this folder?


In [ ]:
# @title 2. SA3 Flow And Conditioning Science. Dataset -> soft prompt

RUN_FLOW_DATASET_SOFT_PROMPT = False

DATASET_SOFT_PROMPT_PATH = OUTPUT_DIR / "flow_conditioning_dataset_soft_prompt.pt"

if RUN_FLOW_DATASET_SOFT_PROMPT:
    require_models()
    freeze_sa3_and_same()
    run_duration = dataset_effective_duration()
    dataset_items = encode_audio_folder_to_items(
        DATASET_DIR,
        limit=DATASET_LIMIT,
        duration=DATASET_DURATION,
        prompt_from_path=True,
        use_chunks=DATASET_USE_CHUNKS,
        chunk_duration=DATASET_CHUNK_DURATION,
        hop_duration=DATASET_HOP_DURATION,
        max_chunks_per_file=DATASET_MAX_CHUNKS_PER_FILE,
        drop_last=DATASET_DROP_LAST_CHUNK,
    )
    dataset_latents = items_to_latent_batch(
        dataset_items,
        device=DEVICE,
        dtype=next(sa3_model.model.model.parameters()).dtype,
    )
    dataset_state = optimize_soft_prompt_from_latents(
        sa3_model,
        dataset_latents,
        seed_prompt="audio dataset texture",
        duration=run_duration,
        optimization_steps=SOFT_STEPS,
        lr=SOFT_LR,
        train_keys=("prompt",),
        reg_weight=1e-4,
        seed=1,
        velocity_convention=FLOW_TARGET_CONVENTION,
    )
    dataset_state.save(DATASET_SOFT_PROMPT_PATH)
    print("saved:", DATASET_SOFT_PROMPT_PATH)
    print("initial/final loss:", dataset_state.losses[0], dataset_state.losses[-1])


In [ ]:
# @title 2. SA3 Flow And Conditioning Science. Flow attribution prompt microscope

RUN_FLOW_ATTRIBUTION = False

FLOW_ATTRIBUTION_TARGET_AUDIO = INPUT_DIR / "target.wav"
FLOW_ATTRIBUTION_PROMPT = "warm wide evolving audio texture"
FLOW_ATTRIBUTION_REPLACEMENTS = [
    "warm",
    "cold",
    "bright",
    "dark",
    "wide",
    "narrow",
    "dense",
    "sparse",
    "loop",
    "texture",
]
FLOW_ATTRIBUTION_DURATION = 8.0
FLOW_ATTRIBUTION_LOGSNR_VALUES = [2.0, 0.0, -2.0]
FLOW_ATTRIBUTION_SEED = 123
FLOW_ATTRIBUTION_OUTPUT = OUTPUT_DIR / "flow_conditioning_attribution.json"


def flow_conditioning_loss_scorer(target_latents, *, duration, logsnr_values, seed):
    probe_bank = build_flow_probe_bank_from_controls(
        logsnr_values=logsnr_values,
        seed=seed,
        velocity_convention=FLOW_TARGET_CONVENTION,
        antithetic_noise=True,
        shared_noise=True,
        metadata={"bench": "flow_attribution"},
    )

    def scorer(prompts):
        return sa3_flow_losses_for_prompts(
            sa3_model,
            target_latents,
            prompts,
            duration=duration,
            seed=seed,
            shared_noise=True,
            antithetic_noise=False,
            normalize_mse=True,
            cosine_weight=0.25,
            velocity_convention=FLOW_TARGET_CONVENTION,
            probe_bank=probe_bank,
        )

    scorer.probe_bank = probe_bank
    return scorer


if RUN_FLOW_ATTRIBUTION:
    require_models()
    target_item = encode_audio_file_to_item(
        FLOW_ATTRIBUTION_TARGET_AUDIO,
        item_id="flow_conditioning_attribution_target",
        duration=FLOW_ATTRIBUTION_DURATION,
    )
    target_latents = item_to_sa3_tensor(target_item, device=DEVICE)
    scorer = flow_conditioning_loss_scorer(
        target_latents,
        duration=target_item.duration_seconds,
        logsnr_values=FLOW_ATTRIBUTION_LOGSNR_VALUES,
        seed=FLOW_ATTRIBUTION_SEED,
    )
    attribution_rows = prompt_leave_one_out_attribution(
        FLOW_ATTRIBUTION_PROMPT,
        scorer,
        replacement_candidates=FLOW_ATTRIBUTION_REPLACEMENTS,
    )
    payload = {
        "target_audio": str(FLOW_ATTRIBUTION_TARGET_AUDIO),
        "prompt": FLOW_ATTRIBUTION_PROMPT,
        "logsnr_values": FLOW_ATTRIBUTION_LOGSNR_VALUES,
        "flow_probe_bank": flow_probe_manifest_or_none(scorer.probe_bank),
        "rows": [asdict(row) for row in attribution_rows],
    }
    FLOW_ATTRIBUTION_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    FLOW_ATTRIBUTION_OUTPUT.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    try:
        import pandas as pd

        display(pd.DataFrame(payload["rows"]))
    except Exception:
        print(json.dumps(payload, indent=2)[:6000])
    print("saved:", FLOW_ATTRIBUTION_OUTPUT)


In [ ]:
# @title 2. SA3 Flow And Conditioning Science. Loss-by-timestep flow panel

RUN_FLOW_TIMESTEP_PANEL = False

FLOW_TIMESTEP_TARGET_AUDIO = FLOW_ATTRIBUTION_TARGET_AUDIO
FLOW_TIMESTEP_PROMPTS = [
    "warm wide evolving audio texture",
    "sparse glassy ambient loop",
    "bright rhythmic synthetic music",
]
FLOW_TIMESTEP_DURATION = 8.0
FLOW_TIMESTEP_LOGSNR_VALUES = [4.0, 2.0, 0.0, -2.0, -4.0]
FLOW_TIMESTEP_SEED = 321
FLOW_TIMESTEP_OUTPUT = OUTPUT_DIR / "flow_conditioning_timestep_panel.json"


if RUN_FLOW_TIMESTEP_PANEL:
    require_models()
    target_item = encode_audio_file_to_item(
        FLOW_TIMESTEP_TARGET_AUDIO,
        item_id="flow_conditioning_timestep_target",
        duration=FLOW_TIMESTEP_DURATION,
    )
    target_latents = item_to_sa3_tensor(target_item, device=DEVICE)
    timestep_probe_bank = build_flow_probe_bank_from_controls(
        logsnr_values=FLOW_TIMESTEP_LOGSNR_VALUES,
        seed=FLOW_TIMESTEP_SEED,
        velocity_convention=FLOW_TARGET_CONVENTION,
        antithetic_noise=True,
        shared_noise=True,
        metadata={"bench": "loss_by_timestep"},
    )
    rows = sa3_flow_loss_rows_for_prompts(
        sa3_model,
        target_latents,
        FLOW_TIMESTEP_PROMPTS,
        duration=target_item.duration_seconds,
        seed=FLOW_TIMESTEP_SEED,
        shared_noise=True,
        antithetic_noise=False,
        normalize_mse=True,
        cosine_weight=0.25,
        velocity_convention=FLOW_TARGET_CONVENTION,
        probe_bank=timestep_probe_bank,
    )
    payload = {
        "target_audio": str(FLOW_TIMESTEP_TARGET_AUDIO),
        "flow_probe_bank": flow_probe_manifest_or_none(timestep_probe_bank),
        "rows": [asdict(row) for row in rows],
        "summary": summarize_flow_loss_rows(rows),
    }
    FLOW_TIMESTEP_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    FLOW_TIMESTEP_OUTPUT.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    try:
        import pandas as pd

        display(pd.DataFrame(payload["summary"]))
        display(pd.DataFrame(payload["rows"]))
    except Exception:
        print(json.dumps(payload, indent=2)[:6000])
    print("saved:", FLOW_TIMESTEP_OUTPUT)


## 2. SA3 Flow And Conditioning Science. Prompt Semantic Transparency

Use this panel to make prompt variants auditable before treating any prompt as a discovered description. Each row records the intended wording change, its semantic tags, and the frozen-SA3 flow loss over the same probe bank.


In [ ]:
# @title 2. SA3 Flow And Conditioning Science. Prompt semantic transparency panel

RUN_PROMPT_SEMANTIC_TRANSPARENCY = False

PROMPT_TRANSPARENCY_TARGET_AUDIO = INPUT_DIR / "target.wav"
PROMPT_TRANSPARENCY_DURATION = 8.0
PROMPT_TRANSPARENCY_PROMPTS = {
    "seed": "warm wide evolving audio texture",
    "material": "warm tape-saturated synth texture with soft noise grain",
    "gesture": "slow pulsing loop with gentle rises and blurred attacks",
    "space": "wide distant ambient loop with soft reverb tails",
    "metadata": "electronic ambient loop, studio recording, modern synth sound",
}
PROMPT_TRANSPARENCY_TAGS = {
    "seed": ["instruction"],
    "material": ["material", "production"],
    "gesture": ["gesture", "time"],
    "space": ["space", "affect"],
    "metadata": ["metadata"],
}
PROMPT_TRANSPARENCY_LOGSNR_VALUES = [2.0, 0.0, -2.0]
PROMPT_TRANSPARENCY_SEED = 234
PROMPT_TRANSPARENCY_OUTPUT = OUTPUT_DIR / "prompt_semantic_transparency.json"


if RUN_PROMPT_SEMANTIC_TRANSPARENCY:
    require_models()
    variants = make_prompt_variants(
        PROMPT_TRANSPARENCY_PROMPTS,
        tags_by_variant=PROMPT_TRANSPARENCY_TAGS,
        source="manual_semantic_variants",
    )
    probe_bank = build_flow_probe_bank_from_controls(
        logsnr_values=PROMPT_TRANSPARENCY_LOGSNR_VALUES,
        seed=PROMPT_TRANSPARENCY_SEED,
        velocity_convention=FLOW_TARGET_CONVENTION,
        antithetic_noise=True,
        shared_noise=True,
        metadata={"bench": "prompt_semantic_transparency"},
    )
    target_item = encode_audio_file_to_item(
        PROMPT_TRANSPARENCY_TARGET_AUDIO,
        item_id="prompt_semantic_target",
        duration=PROMPT_TRANSPARENCY_DURATION,
    )
    target_latents = item_to_sa3_tensor(target_item, device=DEVICE)
    flow_rows = sa3_flow_loss_rows_for_prompts(
        sa3_model,
        target_latents,
        [variant.prompt for variant in variants],
        duration=target_item.duration_seconds,
        seed=PROMPT_TRANSPARENCY_SEED,
        normalize_mse=True,
        cosine_weight=0.25,
        velocity_convention=FLOW_TARGET_CONVENTION,
        probe_bank=probe_bank,
    )
    flow_summary = summarize_flow_loss_rows(flow_rows)
    flow_losses_by_prompt = {row["prompt"]: row["loss_mean"] for row in flow_summary}
    semantic_rows = rank_prompt_semantic_rows(
        prompt_semantic_rows(variants, flow_losses_by_prompt=flow_losses_by_prompt)
    )
    payload = {
        "target_audio": str(PROMPT_TRANSPARENCY_TARGET_AUDIO),
        "prompt_variants": prompt_variant_manifest(variants),
        "semantic_tag_counts": semantic_tag_counts(variants),
        "flow_probe_bank": flow_probe_manifest_or_none(probe_bank),
        "semantic_rows": [asdict(row) for row in semantic_rows],
        "flow_rows": [asdict(row) for row in flow_rows],
    }
    PROMPT_TRANSPARENCY_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    PROMPT_TRANSPARENCY_OUTPUT.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    try:
        import pandas as pd

        table = pd.DataFrame(payload["semantic_rows"])
        table["semantic_tags"] = table["semantic_tags"].apply(lambda tags: ", ".join(tags))
        display(table)
        display(pd.DataFrame(payload["flow_rows"]))
    except Exception:
        print(json.dumps(payload, indent=2)[:6000])
    print("saved:", PROMPT_TRANSPARENCY_OUTPUT)


In [ ]:
# @title 2. SA3 Flow And Conditioning Science. SA3 null-condition inversion probe

RUN_FLOW_NULL_INVERSION = False

NULL_INVERSION_TARGET_AUDIO = INPUT_DIR / "target.wav"
NULL_INVERSION_PROMPT = "audio texture"
NULL_INVERSION_DURATION = 8.0
NULL_INVERSION_STEPS = 40
NULL_INVERSION_LR = 5e-3
NULL_INVERSION_LOGSNR_VALUES = [2.0, 0.0, -2.0]
NULL_INVERSION_OUTPUT = OUTPUT_DIR / "flow_conditioning_null_inversion.json"


def flow_conditioning_null_inputs(prompt, target, duration):
    core = sa3_model.model
    dtype = next(core.model.parameters()).dtype
    conditioning = [{"prompt": prompt, "seconds_total": duration}]
    cond = dict(core.conditioner(conditioning, DEVICE))
    batch, channels, frames = target.shape
    cond["inpaint_mask"] = [torch.zeros((batch, 1, frames), device=DEVICE)]
    cond["inpaint_masked_input"] = [torch.zeros((batch, channels, frames), device=DEVICE, dtype=dtype)]
    return {
        key: value.to(dtype) if isinstance(value, torch.Tensor) and value.is_floating_point() else value
        for key, value in core.get_conditioning_inputs(cond).items()
    }


if RUN_FLOW_NULL_INVERSION:
    require_models()
    target_item = encode_audio_file_to_item(NULL_INVERSION_TARGET_AUDIO, item_id="flow_conditioning_null_inversion_target", duration=NULL_INVERSION_DURATION)
    target = item_to_sa3_tensor(target_item, device=DEVICE, dtype=next(sa3_model.model.model.parameters()).dtype)
    timesteps = timesteps_from_logsnr_values(NULL_INVERSION_LOGSNR_VALUES)
    null_inputs = flow_conditioning_null_inputs("", target, target_item.duration_seconds)
    trainable = {
        key: value.detach().clone().requires_grad_(True)
        for key, value in null_inputs.items()
        if isinstance(value, torch.Tensor) and value.is_floating_point()
    }
    static_inputs = {key: value for key, value in null_inputs.items() if key not in trainable}
    optimizer = torch.optim.Adam(list(trainable.values()), lr=NULL_INVERSION_LR)
    losses = []
    core = sa3_model.model
    for step in range(NULL_INVERSION_STEPS):
        optimizer.zero_grad(set_to_none=True)
        total = 0.0
        for timestep in timesteps:
            generator = torch.Generator(device=DEVICE)
            generator.manual_seed(1000 + step)
            noise = torch.randn(target.shape, device=DEVICE, dtype=target.dtype, generator=generator)
            t = torch.full((target.shape[0],), float(timestep), device=DEVICE)
            z_t = (1 - t[:, None, None].to(target.dtype)) * target + t[:, None, None].to(target.dtype) * noise
            wanted = flow_velocity_target(target, noise, convention=FLOW_TARGET_CONVENTION)
            pred = core.model(z_t, t, **{**static_inputs, **trainable}, cfg_scale=1.0, batch_cfg=True)
            total = total + torch.mean((pred.float() - wanted.float()) ** 2)
        loss = total / max(len(timesteps), 1)
        loss.backward()
        optimizer.step()
        losses.append(float(loss.detach().cpu()))
    payload = {"target_audio": str(NULL_INVERSION_TARGET_AUDIO), "prompt": NULL_INVERSION_PROMPT, "losses": losses}
    NULL_INVERSION_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    NULL_INVERSION_OUTPUT.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print("initial/final loss:", losses[0], losses[-1])
    print("saved:", NULL_INVERSION_OUTPUT)


## 3. SA3 Internal Trajectory Science. Cyclic Projection Inside the Denoising Trajectory

This is the sampler-state version of trying to impose cyclic continuity during generation.

Instead of repairing an end/start boundary after the fact, we intervene during
the rectified-flow trajectory itself. Start from an arbitrary input audio clip
encoded into SAME latents:

$$
z_0 = E(x), \qquad s_T = (1 - \sigma)z_0 + \sigma \epsilon
$$

The SAME decoder itself is feed-forward, so there is no decoder-time iterative
loop to scroll. The iterative signal we can actually intervene on is SA3's
denoising / rectified-flow state \(s_i\).

The experiment closest to the tiling intuition is cyclic mixing:

$$
\bar{s}_i = \frac{1}{2}(s_i + R_s s_i)
$$

$$
s_{i+1} = \left[s_i + \Delta t_i\,v_\theta(s_i, t_i, c)\right]
\quad\text{then}\quad
s_{i+1} \leftarrow s_{i+1} + \beta(\bar{s}_{i+1} - s_{i+1})
$$

where \(R_s\) is a cyclic time roll, usually \(s=T/2\), and \(\beta\) is the
soft projection strength.

Unlike the earlier literal roll/unroll probe, this changes the latent state. It
also changes the latent state when `CYCLIC_STEP_INIT_NOISE = 0`, because the
projection itself still runs even if the flow update has \(\Delta t=0\).

Two diagnostic variants remain available:

- `alternate`: literally rolls the state after every step. This is mostly a
  coordinate transform if the output is unrolled.
- `paired_average`: evaluates the velocity field at both origins and averages:

$$
v_{\mathrm{cyc}}(s_i, t_i, c) = \frac{1}{2}\left[
v_\theta(s_i, t_i, c) + R_{-s}v_\theta(R_s s_i, t_i, c)
\right]
$$

No inpainting boundary is used here. The helper still passes zero inpaint tensors
because the SA3 model architecture expects them, but no mask region is active.
This is a sampler-level experiment, not a guaranteed loop constraint.


In [ ]:
# @title 3. SA3 Internal Trajectory Science. Cyclic projection inside each denoising step

RUN_CAUSAL_CYCLIC_TRAJECTORY = False

CYCLIC_STEP_AUDIO = INPUT_DIR / "known_9s.wav"
CYCLIC_STEP_OUTPUT_DIR = OUTPUT_DIR / "causal_cyclic_trajectory"
CYCLIC_STEP_DURATION = 9.0
CYCLIC_STEP_PROMPT = ""
CYCLIC_STEP_STEPS = 20
CYCLIC_STEP_CFG = 1.0
CYCLIC_STEP_SEED = 1070

# Start at 0.0 to isolate the cyclic projection. Raise to 0.08-0.25 if you want
# SA3 to re-musicalize / re-denoise the projected latent.
CYCLIC_STEP_INIT_NOISE = 0.0

# 0.5 is the faithful half-scroll analogue.
CYCLIC_STEP_ROLL_FRACTION = 0.5

# "cyclic_mix" is the useful tiling-style version.
# Optional diagnostics: "alternate", "paired_average".
CYCLIC_STEP_VARIANTS = ["cyclic_mix"]

# Soft projection strength per step:
#   0.02-0.06 = gentle
#   0.08-0.15 = audible
#   0.25+     = strong half-period collapse risk
CYCLIC_STEP_ROLL_MIXES = [0.08]
CYCLIC_STEP_MIX_EVERY_N = 1
CYCLIC_STEP_SYMMETRIC_MIX = True

# For the alternate variant, unroll the final state back into the source orientation.
CYCLIC_STEP_UNROLL_OUTPUT = True
CYCLIC_STEP_ADD_REPEATED_PREVIEW = False
CYCLIC_STEP_PREVIEW_REPEATS = 2
CYCLIC_STEP_SAVE_PT = True


if RUN_CAUSAL_CYCLIC_TRAJECTORY:
    require_models()
    CYCLIC_STEP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    source_audio, source_sr = load_audio(
        CYCLIC_STEP_AUDIO,
        target_sample_rate=sa3.sample_rate,
        duration=CYCLIC_STEP_DURATION,
        stereo=True,
    )
    source_item = encode_audio_file_to_item(
        CYCLIC_STEP_AUDIO,
        item_id="cyclic_step_source",
        duration=CYCLIC_STEP_DURATION,
    )
    source_latents = item_to_sa3_tensor(
        source_item,
        device=DEVICE,
        dtype=next(sa3_model.model.model.parameters()).dtype,
    )

    source_path = CYCLIC_STEP_OUTPUT_DIR / "source_reference.wav"
    torchaudio.save(str(source_path), source_audio.cpu(), source_sr)

    roll_frames = frames_from_fraction(source_latents, CYCLIC_STEP_ROLL_FRACTION)
    manifest = {
        "source_audio": str(CYCLIC_STEP_AUDIO),
        "duration": float(CYCLIC_STEP_DURATION),
        "prompt": CYCLIC_STEP_PROMPT,
        "steps": int(CYCLIC_STEP_STEPS),
        "cfg": float(CYCLIC_STEP_CFG),
        "init_noise": float(CYCLIC_STEP_INIT_NOISE),
        "roll_fraction": float(CYCLIC_STEP_ROLL_FRACTION),
        "roll_frames": int(roll_frames),
        "mix_every_n": int(CYCLIC_STEP_MIX_EVERY_N),
        "symmetric_mix": bool(CYCLIC_STEP_SYMMETRIC_MIX),
        "source_metrics": metrics_payload(source_latents),
        "outputs": [],
    }
    player_paths = [source_path]
    player_labels = ["source reference"]
    if CYCLIC_STEP_ADD_REPEATED_PREVIEW:
        source_preview_path = save_loop_preview(source_path, CYCLIC_STEP_PREVIEW_REPEATS, "preview")
        player_paths.append(source_preview_path)
        player_labels.append(f"source x{CYCLIC_STEP_PREVIEW_REPEATS}")

    variant_index = 0
    for cyclic_variant in CYCLIC_STEP_VARIANTS:
        for roll_mix in CYCLIC_STEP_ROLL_MIXES:
            print("running cyclic step roll:", cyclic_variant, "mix", roll_mix)
            out_latents = sa3_cyclic_roll_sample_from_init_latents(
                sa3_model,
                source_latents,
                prompt=CYCLIC_STEP_PROMPT,
                duration=CYCLIC_STEP_DURATION,
                steps=CYCLIC_STEP_STEPS,
                cfg_scale=CYCLIC_STEP_CFG,
                init_noise_level=CYCLIC_STEP_INIT_NOISE,
                roll_fraction=CYCLIC_STEP_ROLL_FRACTION,
                mode=cyclic_variant,
                roll_mix=roll_mix,
                mix_every_n=CYCLIC_STEP_MIX_EVERY_N,
                symmetric_mix=CYCLIC_STEP_SYMMETRIC_MIX,
                unroll_output=CYCLIC_STEP_UNROLL_OUTPUT,
                seed=CYCLIC_STEP_SEED + variant_index,
            )

            safe_variant = safe_loop_name(cyclic_variant)
            mix_tag = safe_loop_name(f"{roll_mix:.3f}")
            out_path = CYCLIC_STEP_OUTPUT_DIR / f"cyclic_step_roll_{safe_variant}_mix_{mix_tag}.wav"
            decode_sa3_latents_to_file_cropped(
                out_latents,
                out_path,
                duration=CYCLIC_STEP_DURATION,
            )
            preview_path = None
            if CYCLIC_STEP_ADD_REPEATED_PREVIEW:
                preview_path = save_loop_preview(out_path, CYCLIC_STEP_PREVIEW_REPEATS, "preview")
            metrics = metrics_payload(out_latents)

            pt_path = None
            if CYCLIC_STEP_SAVE_PT:
                pt_path = CYCLIC_STEP_OUTPUT_DIR / f"cyclic_step_roll_{safe_variant}_mix_{mix_tag}.pt"
                torch.save(
                    {
                        "latents": out_latents.detach().cpu(),
                        "variant": cyclic_variant,
                        "roll_mix": roll_mix,
                        "roll_fraction": CYCLIC_STEP_ROLL_FRACTION,
                        "roll_frames": roll_frames,
                        "init_noise": CYCLIC_STEP_INIT_NOISE,
                        "steps": CYCLIC_STEP_STEPS,
                        "seed": CYCLIC_STEP_SEED + variant_index,
                    },
                    pt_path,
                )

            manifest["outputs"].append(
                {
                    "variant": cyclic_variant,
                    "roll_mix": float(roll_mix),
                    "audio": str(out_path),
                    "preview": str(preview_path) if preview_path else None,
                    "latents_pt": str(pt_path) if pt_path else None,
                    "metrics": metrics,
                }
            )
            player_paths.append(out_path)
            player_labels.append(f"cyclic step {cyclic_variant} mix {roll_mix:.3f}")
            if preview_path is not None:
                player_paths.append(preview_path)
                player_labels.append(f"cyclic step {cyclic_variant} mix {roll_mix:.3f} x{CYCLIC_STEP_PREVIEW_REPEATS}")
            print(cyclic_variant, "mix", roll_mix, metrics)
            variant_index += 1

    manifest_path = CYCLIC_STEP_OUTPUT_DIR / "manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print("saved manifest:", manifest_path)
    play_audio_files(
        player_paths,
        labels=player_labels,
        title="SA3 Internal Trajectory Science: Cyclic Projection Inside Denoising",
        max_embed_mb=220.0,
    )


## 3. SA3 Internal Trajectory Science. Prompt-Derived Steering

Collect hidden activations from paired prompts:

$$
a_\ell^+,\quad a_\ell^-
$$

Compute a layer-wise direction:

$$
v_\ell = \operatorname{mean}(a_\ell^+) - \operatorname{mean}(a_\ell^-)
$$

Patch during generation:

$$
a_\ell \leftarrow a_\ell + \alpha v_\ell
$$

This is audioscope-style activation steering. It is inference-time and does not update SA3 weights.


In [ ]:
# @title 3. SA3 Internal Trajectory Science. SA3 residual steering from prompt pairs

RUN_CAUSAL_PROMPT_RESIDUAL = False

PROMPT_VECTOR_DIR = VECTOR_DIR / "causal_prompt_residual_valence"
PROMPT_VECTOR_AXIS = "valence"
PROMPT_VECTOR_LAYERS = None  # Example: [8, 9, 10, 11, 12]
ALPHAS = [-2.0, -1.0, 0.0, 1.0, 2.0]

if RUN_CAUSAL_PROMPT_RESIDUAL:
    require_models()
    pairs = pairs_for_axis(PROMPT_VECTOR_AXIS)
    extractor = SA3ActivationVectorExtractor(
        sa3_model,
        layer_indices=PROMPT_VECTOR_LAYERS,
        cpu_offload=True,
    )
    result = extractor.extract(
        pairs=pairs,
        duration=8.0,
        steps=8,
        cfg_scale=1.0,
        seed=100,
        normalize=True,
        probe=True,
    )
    result.save(PROMPT_VECTOR_DIR)
    print("saved vectors:", PROMPT_VECTOR_DIR)
    print("probe accuracy:", result.vectors.probe_accuracy)

    sweep_outputs = alpha_sweep(
        sa3,
        prompt="a cinematic electronic loop with clear harmony and evolving texture",
        vectors=result.vectors,
        alphas=ALPHAS,
        output_dir=OUTPUT_DIR / "causal_prompt_residual_alpha_sweep",
        duration=8.0,
        steps=8,
        cfg_scale=1.0,
        seed=101,
        top_k=1,
        save_audio=True,
    )
    print(sweep_outputs)
    play_audio_files(
        [output.audio_path for output in sweep_outputs if output.audio_path],
        labels=[f"alpha {output.alpha:+.2f}" for output in sweep_outputs if output.audio_path],
        title="SA3 Internal Trajectory Science: Prompt-Residual Alpha Sweep",
    )


## 3. SA3 Internal Trajectory Science. Audio-Derived Steering

Instead of positive/negative prompt pairs, use audio examples to create residual directions.

Positive audio pass:

- run `init_audio=(sample_rate, audio)`
- record residual activations \(a_\ell^{\mathrm{positive}}\)

Baseline can be:

```text
prompt-only generation
reference audio folder
negative audio folder
```

Direction:

$$
v_\ell = \operatorname{mean}(a_\ell^{\mathrm{audio}}) - \operatorname{mean}(a_\ell^{\mathrm{baseline}})
$$


In [ ]:
# @title 3. SA3 Internal Trajectory Science. SA3 residual steering from audio files

RUN_CAUSAL_AUDIO_RESIDUAL = False

AUDIO_VECTOR_DIR = VECTOR_DIR / "causal_audio_residual"
AUDIO_RESIDUAL_BASELINE = "prompt"  # "prompt" or "negative_audio"
AUDIO_RESIDUAL_PROMPT = "audio texture"
AUDIO_RESIDUAL_NOISE = 0.35

if RUN_CAUSAL_AUDIO_RESIDUAL:
    require_models()
    run_duration = dataset_effective_duration()
    positive_paths = list_audio_files(DATASET_DIR, limit=DATASET_LIMIT)
    negative_paths = list_audio_files(REFERENCE_DIR, limit=DATASET_LIMIT) if AUDIO_RESIDUAL_BASELINE == "negative_audio" else None
    extractor = SA3AudioResidualVectorExtractor(
        sa3_model,
        layer_indices=PROMPT_VECTOR_LAYERS,
        cpu_offload=True,
    )
    result = extractor.extract(
        positive_paths=positive_paths,
        negative_paths=negative_paths,
        prompt=AUDIO_RESIDUAL_PROMPT,
        duration=run_duration,
        steps=8,
        cfg_scale=1.0,
        seed=200,
        init_noise_level=AUDIO_RESIDUAL_NOISE,
        baseline_mode=AUDIO_RESIDUAL_BASELINE,
        normalize=True,
        probe=True,
    )
    result.save(AUDIO_VECTOR_DIR)
    print("saved vectors:", AUDIO_VECTOR_DIR)
    print("probe accuracy:", result.vectors.probe_accuracy)


## 3. SA3 Internal Trajectory Science. Flow-State Optimization Scaffold

This experiment optimizes variables in the generation trajectory rather than prompts.

Potential trainable objects:

- initial noise \(\epsilon\)
- intermediate latent \(z_t\)
- per-step guidance scales
- per-layer steering alphas

The clean formulation requires access to SA3's sampler loop:

$$
z_{k+1} = \Phi_\theta(z_k, t_k, c)
$$

Then optimize:

$$
\epsilon^\star =
\arg\min_\epsilon
d\left(D(\operatorname{sample}_\theta(\epsilon, c)), x_{\mathrm{target}}\right)
$$

or, natively:

$$
\epsilon^\star =
\arg\min_\epsilon
\left\| z_{\mathrm{generated}}(\epsilon,c) - z_{\mathrm{target}} \right\|^2
$$

This first-pass cell exposes a flow-state loss. A full differentiable sampler implementation should be added after inspecting the released SA3 sampler code on Colab.


In [ ]:
# @title 3. SA3 Internal Trajectory Science. Flow-state optimization scaffold

RUN_CAUSAL_FLOW_STATE_OPTIMIZATION = False


def optimize_single_flow_state(
    stable_model,
    target_latents,
    prompt,
    *,
    duration,
    steps=100,
    lr=1e-2,
    t_value=0.5,
    seed=0,
):
    # This is not a full sampler inversion.
    # It optimizes an intermediate z_t so the frozen SA3 velocity agrees with
    # the target flow velocity. It is useful for debugging trajectory losses.
    core = stable_model.model
    device = str(stable_model.device)
    dtype = next(core.model.parameters()).dtype
    target = target_latents.to(device=device, dtype=dtype)
    if target.ndim == 2:
        target = target.unsqueeze(0)
    batch, channels, frames = target.shape
    generator = torch.Generator(device=device)
    generator.manual_seed(seed)
    epsilon = torch.randn(target.shape, device=device, dtype=dtype, generator=generator)
    t = torch.full((batch,), float(t_value), device=device)
    velocity_target = flow_velocity_target(target, epsilon, convention=FLOW_TARGET_CONVENTION)
    z_t_init = (1 - t[:, None, None].to(dtype)) * target + t[:, None, None].to(dtype) * epsilon
    z_t = torch.nn.Parameter(z_t_init.detach().clone())

    conditioning = [{"prompt": prompt, "seconds_total": duration}] * batch
    cond = core.conditioner(conditioning, device)
    cond = dict(cond)
    cond["inpaint_mask"] = [torch.zeros((batch, 1, frames), device=device)]
    cond["inpaint_masked_input"] = [torch.zeros((batch, channels, frames), device=device, dtype=dtype)]
    cond_inputs = core.get_conditioning_inputs(cond)
    cond_inputs = {
        key: value.to(dtype) if isinstance(value, torch.Tensor) and value.is_floating_point() else value
        for key, value in cond_inputs.items()
    }
    for parameter in core.parameters():
        parameter.requires_grad_(False)
    optimizer = torch.optim.AdamW([z_t], lr=lr)
    losses = []
    for _ in range(steps):
        optimizer.zero_grad(set_to_none=True)
        pred = core.model(z_t, t, **cond_inputs, cfg_scale=1.0, batch_cfg=True)
        loss = torch.nn.functional.mse_loss(pred.float(), velocity_target.float())
        loss.backward()
        optimizer.step()
        losses.append(float(loss.detach().cpu()))
    return z_t.detach(), losses


if RUN_CAUSAL_FLOW_STATE_OPTIMIZATION:
    require_models()
    target_item = encode_audio_file_to_item(TARGET_AUDIO, item_id="target", duration=TARGET_DURATION)
    target_latents = item_to_sa3_tensor(
        target_item,
        device=DEVICE,
        dtype=next(sa3_model.model.model.parameters()).dtype,
    )
    z_t_opt, losses = optimize_single_flow_state(
        sa3_model,
        target_latents,
        "audio texture",
        duration=TARGET_DURATION,
        steps=100,
        lr=1e-2,
    )
    print("initial/final loss:", losses[0], losses[-1])


## 3. SA3 Internal Trajectory Science. LatCH-Style Control Heads

LatCH-style sidecar:

$$
h_\psi(z) \rightarrow y
$$

where \(y\) can be:

```text
brightness
density
tension
loopability
section role
prompt match
```

The base model stays frozen. The control head learns to observe SAME latent properties.

Later, the head can become a guidance source:

$$
\mathcal{L}_{\mathrm{total}}
= \mathcal{L}_{\mathrm{flow}}
+ \lambda \mathcal{L}_{\mathrm{control}}(h_\psi(z), y_{\mathrm{target}})
$$


In [ ]:
# @title 3. SA3 Internal Trajectory Science. Minimal LatCH-style control head on SAME summaries

RUN_CAUSAL_CONTROL_HEAD = False

CONTROL_NAME = "brightness"


class LatentControlHead(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim=256, output_dim=1):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(input_dim, hidden_dim),
            torch.nn.SiLU(),
            torch.nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        return self.net(x)


def train_control_head(items, control_name, *, epochs=200, lr=1e-3, device=DEVICE):
    examples = []
    labels = []
    for item in items:
        if control_name in item.descriptors:
            examples.append(latent_summary(item))
            labels.append(float(item.descriptors[control_name]))
        elif control_name in item.labels:
            examples.append(latent_summary(item))
            labels.append(float(item.labels[control_name]))
    if not examples:
        raise ValueError(f"No labels/descriptors found for control {control_name!r}.")
    x = torch.tensor(np.stack(examples), dtype=torch.float32, device=device)
    y = torch.tensor(np.asarray(labels)[:, None], dtype=torch.float32, device=device)
    model = LatentControlHead(x.shape[1]).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    losses = []
    for _ in range(epochs):
        optimizer.zero_grad(set_to_none=True)
        pred = model(x)
        loss = torch.nn.functional.mse_loss(pred, y)
        loss.backward()
        optimizer.step()
        losses.append(float(loss.detach().cpu()))
    return model, losses


if RUN_CAUSAL_CONTROL_HEAD:
    # Put controls in item.descriptors before this cell, e.g.
    # item.descriptors["brightness"] = 0.8
    items = load_items(MEMORY_DIR)
    head, losses = train_control_head(items, CONTROL_NAME)
    torch.save({"state_dict": head.state_dict(), "control": CONTROL_NAME}, OUTPUT_DIR / "causal_control_head.pt")
    print("initial/final loss:", losses[0], losses[-1])


In [ ]:
# @title 3. SA3 Internal Trajectory Science. Residual feature atlas

RUN_CAUSAL_RESIDUAL_FEATURE_ATLAS = False

RESIDUAL_ATLAS_AXIS = "brightness"
RESIDUAL_ATLAS_NUM_PAIRS = 4
RESIDUAL_ATLAS_LAYERS = [8, 12, 16, 20]
RESIDUAL_ATLAS_DURATION = 6.0
RESIDUAL_ATLAS_STEPS = 6
RESIDUAL_ATLAS_OUTPUT = OUTPUT_DIR / "causal_residual_feature_atlas.json"


if RUN_CAUSAL_RESIDUAL_FEATURE_ATLAS:
    require_models()
    pairs = pairs_for_axis(RESIDUAL_ATLAS_AXIS)[:RESIDUAL_ATLAS_NUM_PAIRS]
    extractor = SA3ActivationVectorExtractor(sa3, layer_indices=RESIDUAL_ATLAS_LAYERS)
    examples = extractor.collect_examples(
        pairs,
        duration=RESIDUAL_ATLAS_DURATION,
        steps=RESIDUAL_ATLAS_STEPS,
        cfg_scale=1.0,
        seed=0,
    )
    atlas_rows = []
    for layer in RESIDUAL_ATLAS_LAYERS:
        activations = [example.layer_activations[layer].float().cpu().numpy() for example in examples if layer in example.layer_activations]
        if len(activations) < 2:
            atlas_rows.append({"layer": layer, "error": "not enough activations"})
            continue
        basis = fit_residual_feature_basis(activations, layer=str(layer), n_components=8)
        projected = [project_residual_features(value, basis, whiten=True) for value in activations]
        labels = [example.label for example in examples if layer in example.layer_activations]
        pos = np.stack([row for row, label in zip(projected, labels) if label == 1]).mean(axis=0)
        neg = np.stack([row for row, label in zip(projected, labels) if label == 0]).mean(axis=0)
        atlas_rows.append(
            {
                "layer": layer,
                "example_count": len(activations),
                "explained_variance": basis.variances.astype(float).tolist(),
                "contrastive_feature_delta": (pos - neg).astype(float).tolist(),
            }
        )
    payload = {"axis": RESIDUAL_ATLAS_AXIS, "rows": atlas_rows}
    RESIDUAL_ATLAS_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    RESIDUAL_ATLAS_OUTPUT.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(json.dumps(payload, indent=2)[:6000])
    print("saved:", RESIDUAL_ATLAS_OUTPUT)


In [ ]:
# @title 3. SA3 Internal Trajectory Science. Guidance-gradient latent edit probe

RUN_CAUSAL_GRADIENT_EDIT = False

CAUSAL_GRADIENT_SOURCE_AUDIO = INPUT_DIR / "source.wav"
CAUSAL_GRADIENT_REFERENCE_AUDIO = INPUT_DIR / "reference.wav"
CAUSAL_GRADIENT_DURATION = 8.0
CAUSAL_GRADIENT_STEPS = 24
CAUSAL_GRADIENT_SCALE = 0.02
CAUSAL_GRADIENT_POLISH = True
CAUSAL_GRADIENT_POLISH_NOISE = 0.08
CAUSAL_GRADIENT_PROMPT = "audio texture"
CAUSAL_GRADIENT_OUTPUT_DIR = OUTPUT_DIR / "causal_gradient_edit"


def torch_latent_summary(z):
    time_major = z[0].transpose(0, 1)
    mean = time_major.mean(dim=0)
    std = time_major.std(dim=0, unbiased=False)
    if time_major.shape[0] > 1:
        velocity = torch.abs(time_major[1:] - time_major[:-1]).mean(dim=0)
    else:
        velocity = torch.zeros_like(mean)
    return torch.cat([mean, std, velocity], dim=0)


if RUN_CAUSAL_GRADIENT_EDIT:
    require_models()
    CAUSAL_GRADIENT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    source_item = encode_audio_file_to_item(CAUSAL_GRADIENT_SOURCE_AUDIO, item_id="causal_gradient_source", duration=CAUSAL_GRADIENT_DURATION)
    reference_item = encode_audio_file_to_item(CAUSAL_GRADIENT_REFERENCE_AUDIO, item_id="causal_gradient_reference", duration=CAUSAL_GRADIENT_DURATION)
    source = item_to_sa3_tensor(source_item, device=DEVICE)
    target_summary = torch.as_tensor(latent_summary(reference_item), device=DEVICE, dtype=source.dtype)
    current = source
    trace = []
    for _step in range(CAUSAL_GRADIENT_STEPS):
        result = gradient_guidance_step(
            current,
            lambda z: torch.mean((torch_latent_summary(z) - target_summary) ** 2),
            scale=CAUSAL_GRADIENT_SCALE,
            normalize=True,
        )
        current = result.latents
        trace.append({"loss": result.loss, "grad_norm": result.grad_norm})
    direct_path = CAUSAL_GRADIENT_OUTPUT_DIR / "guided_direct.wav"
    source_path = CAUSAL_GRADIENT_OUTPUT_DIR / "source_direct.wav"
    decode_sa3_latents_to_file_cropped(source, source_path, duration=CAUSAL_GRADIENT_DURATION)
    decode_sa3_latents_to_file_cropped(current, direct_path, duration=CAUSAL_GRADIENT_DURATION)
    paths = [source_path, direct_path]
    labels = ["source", "guided direct"]
    if CAUSAL_GRADIENT_POLISH:
        polished = sa3_sample_from_init_latents(
            sa3_model,
            current,
            prompt=CAUSAL_GRADIENT_PROMPT,
            duration=CAUSAL_GRADIENT_DURATION,
            steps=8,
            init_noise_level=CAUSAL_GRADIENT_POLISH_NOISE,
            seed=0,
        )
        polished_path = CAUSAL_GRADIENT_OUTPUT_DIR / "guided_polished.wav"
        decode_sa3_latents_to_file_cropped(polished, polished_path, duration=CAUSAL_GRADIENT_DURATION)
        paths.append(polished_path)
        labels.append("guided polished")
    (CAUSAL_GRADIENT_OUTPUT_DIR / "trace.json").write_text(json.dumps(trace, indent=2), encoding="utf-8")
    play_audio_files(paths, labels=labels, title="SA3 Internal Trajectory Science: Guidance-Gradient Latent Edit")
    print("saved:", CAUSAL_GRADIENT_OUTPUT_DIR)


In [ ]:
# @title 3. SA3 Internal Trajectory Science. Audio-to-audio posterior guidance scaffold

RUN_CAUSAL_AUDIO_POSTERIOR = False

CAUSAL_POSTERIOR_SOURCE_AUDIO = INPUT_DIR / "source.wav"
CAUSAL_POSTERIOR_REFERENCE_AUDIO = INPUT_DIR / "reference.wav"
CAUSAL_POSTERIOR_DURATION = 8.0
CAUSAL_POSTERIOR_STEPS = 24
POSTERIOR_CAUSAL_GRADIENT_SCALE = 0.015
POSTERIOR_PRESERVE_WEIGHT = 0.5
CAUSAL_POSTERIOR_PROMPT = "audio texture"
CAUSAL_POSTERIOR_OUTPUT_DIR = OUTPUT_DIR / "causal_audio_posterior"


if RUN_CAUSAL_AUDIO_POSTERIOR:
    require_models()
    CAUSAL_POSTERIOR_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    source_item = encode_audio_file_to_item(CAUSAL_POSTERIOR_SOURCE_AUDIO, item_id="causal_posterior_source", duration=CAUSAL_POSTERIOR_DURATION)
    reference_item = encode_audio_file_to_item(CAUSAL_POSTERIOR_REFERENCE_AUDIO, item_id="causal_posterior_reference", duration=CAUSAL_POSTERIOR_DURATION)
    source = item_to_sa3_tensor(source_item, device=DEVICE)
    current = source.detach().clone()
    target_summary = torch.as_tensor(latent_summary(reference_item), device=DEVICE, dtype=current.dtype)
    source_summary = torch.as_tensor(latent_summary(source_item), device=DEVICE, dtype=current.dtype)
    trace = []
    for _step in range(CAUSAL_POSTERIOR_STEPS):
        def posterior_loss(z):
            summary = torch_latent_summary(z)
            target_loss = torch.mean((summary - target_summary) ** 2)
            preserve_loss = torch.mean((summary[: source_summary.shape[0]] - source_summary) ** 2)
            return target_loss + POSTERIOR_PRESERVE_WEIGHT * preserve_loss

        result = gradient_guidance_step(current, posterior_loss, scale=POSTERIOR_CAUSAL_GRADIENT_SCALE, normalize=True)
        current = result.latents
        trace.append({"loss": result.loss, "grad_norm": result.grad_norm})
    direct_path = CAUSAL_POSTERIOR_OUTPUT_DIR / "posterior_guided_direct.wav"
    decode_sa3_latents_to_file_cropped(current, direct_path, duration=CAUSAL_POSTERIOR_DURATION)
    polished = sa3_sample_from_init_latents(
        sa3_model,
        current,
        prompt=CAUSAL_POSTERIOR_PROMPT,
        duration=CAUSAL_POSTERIOR_DURATION,
        steps=8,
        init_noise_level=0.08,
        seed=0,
    )
    polished_path = CAUSAL_POSTERIOR_OUTPUT_DIR / "posterior_guided_polished.wav"
    decode_sa3_latents_to_file_cropped(polished, polished_path, duration=CAUSAL_POSTERIOR_DURATION)
    (CAUSAL_POSTERIOR_OUTPUT_DIR / "trace.json").write_text(json.dumps(trace, indent=2), encoding="utf-8")
    play_audio_files([direct_path, polished_path], labels=["posterior direct", "posterior polished"], title="SA3 Internal Trajectory Science: Audio Posterior Guidance")
    print("saved:", CAUSAL_POSTERIOR_OUTPUT_DIR)


## 4. SA3-over-SAME Coupled Editing. Neighborhood Renoise

This is the Tero-style loop neighborhood primitive.

Encode an existing loop as a SAME latent, partially renoise it, then let SA3 flow it back to the data manifold:

$$
z_0 = E(x)
$$

$$
z_\sigma \approx (1 - \sigma) z_0 + \sigma \epsilon,\qquad
\epsilon \sim \mathcal{N}(0, I)
$$

$$
\tilde{z} = \operatorname{sample}_\theta(z_\sigma, c=\emptyset)
$$

$$
\tilde{x} = D(\tilde{z})
$$

`init_noise_level` is the local-neighborhood radius:

```text
0.10-0.30: conservative timbral/performance variants
0.40-0.60: neighborhood variants, often keeps harmony/feel
0.70-1.00: increasingly free regeneration
```

This is not text steering. It is local manifold resampling around an audio memory.


In [ ]:
# @title 4. SA3-over-SAME Coupled Editing. Neighborhood renoise from an existing loop/audio

RUN_SAME_NEIGHBORHOOD_RENOISE = False

VARIATION_AUDIO = INPUT_DIR / "loop.wav"
VARIATION_OUTPUT_DIR = OUTPUT_DIR / "same_neighborhood_renoise"
VARIATION_MEMORY_DIR = MEMORY_DIR / "same_neighborhood_renoise"

VARIATION_PROMPT = ""  # Empty string + cfg_scale=1.0 is a practical "no semantic prompt" baseline.
VARIATION_DURATION = 8.0
VARIATION_STEPS = 8
VARIATION_CFG_SCALE = 1.0
VARIATION_NOISE_LEVELS = [0.25, 0.5, 0.75]
VARIATION_SEEDS = [0, 1, 2]

if RUN_SAME_NEIGHBORHOOD_RENOISE:
    require_models()
    VARIATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    audio, sample_rate = load_audio(VARIATION_AUDIO, duration=VARIATION_DURATION, stereo=True)
    items = []
    manifest = []

    for sigma in VARIATION_NOISE_LEVELS:
        for seed in VARIATION_SEEDS:
            latents = sa3.generate_latents(
                prompt=VARIATION_PROMPT,
                duration=VARIATION_DURATION,
                steps=VARIATION_STEPS,
                cfg_scale=VARIATION_CFG_SCALE,
                seed=seed,
                init_audio=(sample_rate, audio),
                init_noise_level=float(sigma),
            )
            item_id = f"renoise_sigma_{sigma:.2f}_seed_{seed}"
            out_path = VARIATION_OUTPUT_DIR / f"{item_id}.wav"
            decode_sa3_latents_to_file(latents, out_path)
            item = sa3_tensor_to_item(
                latents,
                item_id=item_id,
                prompt=VARIATION_PROMPT,
                metadata={
                    "source_audio": str(VARIATION_AUDIO),
                    "init_noise_level": float(sigma),
                    "seed": int(seed),
                    "duration": VARIATION_DURATION,
                    "steps": VARIATION_STEPS,
                    "cfg_scale": VARIATION_CFG_SCALE,
                },
            )
            items.append(item)
            manifest.append({"path": str(out_path), "init_noise_level": float(sigma), "seed": int(seed)})
            print("saved:", out_path)

    save_items(items, VARIATION_MEMORY_DIR)
    (VARIATION_OUTPUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print("saved latent memory:", VARIATION_MEMORY_DIR)
    try:
        display_audio_player(
            [entry["path"] for entry in manifest],
            labels=[f"sigma {entry['init_noise_level']:.2f} seed {entry['seed']}" for entry in manifest],
            title="SA3-over-SAME Coupled Editing: Neighborhood Renoise Variations",
        )
    except Exception as exc:
        print("Custom audio player skipped:", exc)


## 4. SA3-over-SAME Coupled Editing. Channel-Selective Renoise Lab

This is for one short source you know well, e.g. a 9-second loop.

Instead of renoising every latent channel equally:

$$
z_\sigma = (1 - \sigma)z_0 + \sigma \epsilon
$$

select a subset of SAME channels:

$$
S \subset \{0, \dots, 255\}
$$

and perturb only those channels at the sampler start:

$$
\epsilon_{S,c,t} =
\begin{cases}
\epsilon_{c,t}, & c \in S \\
z_{0,c,t}, & c \notin S
\end{cases}
$$

SA3 then starts from:

$$
z_{\mathrm{start}} = (1 - \sigma)z_0 + \sigma \epsilon_S
$$

Unselected channels are not mathematically frozen during denoising; they simply start unchanged. That is enough for a first probe of emergent channel structure.

Useful listening questions:

```text
Which masks preserve rhythm?
Which masks change timbre?
Which masks change stereo/space?
Which masks create artifacts?
Are effects reproducible for the same channel set?
Do high-variance and low-variance channels form different families?
```


In [ ]:
# @title 4. SA3-over-SAME Coupled Editing. Channel-selective renoise lab

RUN_SAME_SELECTIVE_RENOISE = False

SAME_SELECTIVE_AUDIO = INPUT_DIR / "known_9s.wav"
SAME_SELECTIVE_OUTPUT_DIR = OUTPUT_DIR / "same_selective_renoise"
SAME_SELECTIVE_DURATION = 9.0
SAME_SELECTIVE_PROMPT = ""
SAME_SELECTIVE_STEPS = 8
SAME_SELECTIVE_CFG = 1.0
SAME_SELECTIVE_NOISE = 0.40
SAME_SELECTIVE_BASE_SEED = 700

# Direct decode is cheap and off-manifold: useful as a microscope for SAME channels,
# but it usually sounds like damaged latent audio. Keep it off for large listening sweeps.
SAME_SELECTIVE_RUN_DIRECT_DECODE = False
SAME_SELECTIVE_RUN_SA3_SAMPLER = True
SAME_SELECTIVE_SAVE_PT = True
SAME_SELECTIVE_ANNOTATION_PATH = SAME_SELECTIVE_OUTPUT_DIR / "annotations.json"

# Deeper mapping sweeps. Keep these false for a first smoke run.
SAME_SELECTIVE_ADD_BLOCK_SWEEP = False
SAME_SELECTIVE_BLOCK_SIZE = 8
SAME_SELECTIVE_BLOCK_STRIDE = 8
SAME_SELECTIVE_MAX_BLOCKS = 32
SAME_SELECTIVE_ADD_RANDOM_SWEEP = False
SAME_SELECTIVE_RANDOM_FRACTIONS = [0.05, 0.10, 0.20]
SAME_SELECTIVE_RANDOM_MASK_SEEDS = list(range(8))
SAME_SELECTIVE_MAX_VARIANTS = 96

SAME_SELECTIVE_SPECS = [
    {"name": "random_10_s0", "mode": "random_channels", "fraction": 0.10, "seed": 0},
    {"name": "random_10_s1", "mode": "random_channels", "fraction": 0.10, "seed": 1},
    {"name": "random_25_s0", "mode": "random_channels", "fraction": 0.25, "seed": 0},
    {"name": "high_var_10", "mode": "high_variance", "fraction": 0.10, "seed": 0},
    {"name": "low_var_10", "mode": "low_variance", "fraction": 0.10, "seed": 0},
    {"name": "high_activity_10", "mode": "high_activity", "fraction": 0.10, "seed": 0},
    {"name": "block_000_16", "mode": "channel_block", "block_size": 16, "start_channel": 0},
    {"name": "block_128_16", "mode": "channel_block", "block_size": 16, "start_channel": 128},
]


def expand_selective_renoise_specs(base_specs, channel_count):
    specs = list(base_specs)
    if SAME_SELECTIVE_ADD_BLOCK_SWEEP:
        starts = range(0, channel_count, SAME_SELECTIVE_BLOCK_STRIDE)
        for block_index, start in enumerate(starts):
            if block_index >= SAME_SELECTIVE_MAX_BLOCKS:
                break
            specs.append(
                {
                    "name": f"block_{start:03d}_{SAME_SELECTIVE_BLOCK_SIZE}",
                    "mode": "channel_block",
                    "block_size": SAME_SELECTIVE_BLOCK_SIZE,
                    "start_channel": start,
                }
            )
    if SAME_SELECTIVE_ADD_RANDOM_SWEEP:
        for fraction in SAME_SELECTIVE_RANDOM_FRACTIONS:
            for seed in SAME_SELECTIVE_RANDOM_MASK_SEEDS:
                percent = int(round(100 * fraction))
                specs.append(
                    {
                        "name": f"random_{percent:02d}_s{seed}",
                        "mode": "random_channels",
                        "fraction": float(fraction),
                        "seed": int(seed),
                    }
                )
    deduped = []
    seen = set()
    for spec in specs:
        name = spec["name"]
        if name not in seen:
            deduped.append(spec)
            seen.add(name)
    return deduped[:SAME_SELECTIVE_MAX_VARIANTS]


def latent_mask_spec_from_dict(payload):
    return LatentMaskSpec(
        name=payload["name"],
        mode=payload.get("mode", "random_channels"),
        fraction=float(payload.get("fraction", 0.25)),
        seed=int(payload.get("seed", 0)),
        channels=tuple(payload["channels"]) if payload.get("channels") is not None else None,
        start_channel=payload.get("start_channel"),
        block_size=payload.get("block_size"),
    )


def safe_audio_name(name):
    return "".join(ch if ch.isalnum() or ch in "-_." else "_" for ch in name)[:96]


if RUN_SAME_SELECTIVE_RENOISE:
    require_models()
    SAME_SELECTIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    source_audio, source_sr = load_audio(
        SAME_SELECTIVE_AUDIO,
        duration=SAME_SELECTIVE_DURATION,
        stereo=True,
    )
    source_item = encode_audio_file_to_item(
        SAME_SELECTIVE_AUDIO,
        item_id="same_selective_source",
        duration=SAME_SELECTIVE_DURATION,
    )
    source_latents = item_to_sa3_tensor(
        source_item,
        device=DEVICE,
        dtype=next(sa3_model.model.model.parameters()).dtype,
    )

    player_paths = []
    player_labels = []
    player_metadata = []
    manifest = []
    source_path = SAME_SELECTIVE_OUTPUT_DIR / "source_reference.wav"
    torchaudio.save(str(source_path), source_audio.cpu(), source_sr)
    player_paths.append(source_path)
    player_labels.append("source reference")
    player_metadata.append({"kind": "source_reference", "path": str(source_path)})

    expanded_specs = expand_selective_renoise_specs(
        SAME_SELECTIVE_SPECS,
        channel_count=int(source_latents.shape[1]),
    )
    print("variants:", len(expanded_specs))

    for variant_index, spec_dict in enumerate(expanded_specs):
        spec = latent_mask_spec_from_dict(spec_dict)
        name = safe_audio_name(spec.name)
        seed = SAME_SELECTIVE_BASE_SEED + variant_index
        channels = select_latent_channels(source_latents, spec)
        sampler_spec = LatentMaskSpec(
            name=spec.name,
            mode=spec.mode,
            fraction=spec.fraction,
            seed=spec.seed,
            channels=tuple(channels),
            start_channel=spec.start_channel,
            block_size=spec.block_size,
        )
        entry = {
            "name": spec.name,
            "mode": spec.mode,
            "fraction": spec.fraction,
            "seed": spec.seed,
            "sampler_seed": seed,
            "noise_level": SAME_SELECTIVE_NOISE,
            "selected_channel_count": len(channels),
            "selected_channels": channels,
            "outputs": {},
        }

        if SAME_SELECTIVE_RUN_DIRECT_DECODE:
            mixed_latents = masked_latent_noise(
                source_latents,
                channels,
                sigma=SAME_SELECTIVE_NOISE,
                seed=seed,
            )
            direct_path = SAME_SELECTIVE_OUTPUT_DIR / f"direct_{name}.wav"
            decode_sa3_latents_to_file_cropped(
                mixed_latents,
                direct_path,
                duration=SAME_SELECTIVE_DURATION,
            )
            entry["outputs"]["direct_same_decode"] = str(direct_path)
            player_paths.append(direct_path)
            player_labels.append(f"direct {spec.name} ({len(channels)} ch)")
            player_metadata.append(
                {
                    "kind": "direct_same_decode",
                    "variant": spec.name,
                    "mask_mode": spec.mode,
                    "noise_level": SAME_SELECTIVE_NOISE,
                    "selected_channel_count": len(channels),
                    "selected_channels": channels,
                    "sampler_seed": seed,
                    "path": str(direct_path),
                }
            )

        if SAME_SELECTIVE_RUN_SA3_SAMPLER:
            result = selective_renoise_sa3(
                sa3_model,
                source_audio,
                source_sr,
                spec=sampler_spec,
                prompt=SAME_SELECTIVE_PROMPT,
                duration=SAME_SELECTIVE_DURATION,
                steps=SAME_SELECTIVE_STEPS,
                cfg_scale=SAME_SELECTIVE_CFG,
                init_noise_level=SAME_SELECTIVE_NOISE,
                seed=seed,
            )
            sampled_path = SAME_SELECTIVE_OUTPUT_DIR / f"sa3_{name}.wav"
            decode_sa3_latents_to_file_cropped(
                result.sampled_latents,
                sampled_path,
                duration=SAME_SELECTIVE_DURATION,
            )
            entry["outputs"]["sa3_sampled"] = str(sampled_path)
            player_paths.append(sampled_path)
            player_labels.append(f"SA3 {spec.name} ({len(channels)} ch)")
            player_metadata.append(
                {
                    "kind": "sa3_sampled",
                    "variant": spec.name,
                    "mask_mode": spec.mode,
                    "noise_level": SAME_SELECTIVE_NOISE,
                    "selected_channel_count": len(channels),
                    "selected_channels": channels,
                    "sampler_seed": seed,
                    "path": str(sampled_path),
                }
            )
            entry.update(result.metadata)
            if SAME_SELECTIVE_SAVE_PT:
                pt_path = SAME_SELECTIVE_OUTPUT_DIR / f"{name}.pt"
                torch.save(
                    {
                        "init_latents": result.init_latents.detach().cpu(),
                        "mixed_latents": result.mixed_latents.detach().cpu(),
                        "sampled_latents": result.sampled_latents.detach().cpu(),
                        "metadata": result.metadata,
                    },
                    pt_path,
                )
                entry["outputs"]["pt"] = str(pt_path)

        manifest.append(entry)
        print(spec.name, "channels:", len(channels), channels[:24])

    manifest_path = SAME_SELECTIVE_OUTPUT_DIR / "manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print("saved manifest:", manifest_path)
    play_audio_files(
        player_paths,
        labels=player_labels,
        metadata=player_metadata,
        title="SA3-over-SAME Coupled Editing: Channel-Selective Renoise Lab",
        max_embed_mb=160.0,
        annotation_path=SAME_SELECTIVE_ANNOTATION_PATH,
    )


## 4. SA3-over-SAME Coupled Editing. Cross-Audio Channel Graft

This is the donor-audio version of channel-selective latent editing.

Instead of putting Gaussian noise into selected source channels, use another encoded audio file as the replacement direction:

$$
z^{\mathrm{noise}}_{c,t} =
\begin{cases}
z^{\mathrm{donor}}_{c,t}, & c \in S \\
z^{\mathrm{source}}_{c,t}, & c \notin S
\end{cases}
$$

SA3 then starts from:

$$
z_{\mathrm{start}} = (1 - \alpha)z^{\mathrm{source}} + \alpha z^{\mathrm{noise}}
$$

so selected channels become:

$$
z_{\mathrm{start},c,t} = z^{\mathrm{source}}_{c,t} + \alpha(z^{\mathrm{donor}}_{c,t} - z^{\mathrm{source}}_{c,t})
$$

and unselected channels remain exactly the source at sampler start.

Interpretation:

```text
direct graft decode = literal source/donor channel splice through SAME
SA3 graft          = source/donor splice reprojected by SA3's learned music prior
alpha              = graft amount, analogous to init_noise_level
```

This is closer to latent transplantation than variation. It is useful for asking whether a SAME channel block carries portable rhythmic, timbral, spatial, or textural information across two audios.


In [ ]:
# @title 4. SA3-over-SAME Coupled Editing. Cross-audio latent channel graft

RUN_SAME_AUDIO_GRAFT = False

SAME_GRAFT_SOURCE_AUDIO = INPUT_DIR / "known_9s.wav"
SAME_GRAFT_DONOR_AUDIO = INPUT_DIR / "donor_9s.wav"
SAME_GRAFT_OUTPUT_DIR = OUTPUT_DIR / "same_audio_graft"
SAME_GRAFT_DURATION = 9.0
SAME_GRAFT_PROMPT = ""
SAME_GRAFT_STEPS = 8
SAME_GRAFT_CFG = 1.0
SAME_GRAFT_AMOUNT = 0.40
SAME_GRAFT_BASE_SEED = 880

SAME_GRAFT_RUN_DIRECT_DECODE = False
SAME_GRAFT_RUN_SA3_SAMPLER = True
SAME_GRAFT_SAVE_PT = True
SAME_GRAFT_ANNOTATION_PATH = SAME_GRAFT_OUTPUT_DIR / "annotations.json"

SAME_GRAFT_SPECS = [
    {"name": "random_10_s0", "mode": "random_channels", "fraction": 0.10, "seed": 0},
    {"name": "random_10_s1", "mode": "random_channels", "fraction": 0.10, "seed": 1},
    {"name": "random_25_s0", "mode": "random_channels", "fraction": 0.25, "seed": 0},
    {"name": "high_var_10", "mode": "high_variance", "fraction": 0.10, "seed": 0},
    {"name": "high_activity_10", "mode": "high_activity", "fraction": 0.10, "seed": 0},
    {"name": "block_000_16", "mode": "channel_block", "block_size": 16, "start_channel": 0},
    {"name": "block_064_16", "mode": "channel_block", "block_size": 16, "start_channel": 64},
    {"name": "block_128_16", "mode": "channel_block", "block_size": 16, "start_channel": 128},
    {"name": "block_192_16", "mode": "channel_block", "block_size": 16, "start_channel": 192},
]


def graft_mask_spec_from_dict(payload):
    return LatentMaskSpec(
        name=payload["name"],
        mode=payload.get("mode", "random_channels"),
        fraction=float(payload.get("fraction", 0.25)),
        seed=int(payload.get("seed", 0)),
        channels=tuple(payload["channels"]) if payload.get("channels") is not None else None,
        start_channel=payload.get("start_channel"),
        block_size=payload.get("block_size"),
    )


def safe_graft_audio_name(name):
    return "".join(ch if ch.isalnum() or ch in "-_." else "_" for ch in name)[:96]


if RUN_SAME_AUDIO_GRAFT:
    require_models()
    SAME_GRAFT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    source_audio, source_sr = load_audio(
        SAME_GRAFT_SOURCE_AUDIO,
        duration=SAME_GRAFT_DURATION,
        stereo=True,
    )
    donor_audio, donor_sr = load_audio(
        SAME_GRAFT_DONOR_AUDIO,
        duration=SAME_GRAFT_DURATION,
        stereo=True,
    )
    source_item = encode_audio_file_to_item(
        SAME_GRAFT_SOURCE_AUDIO,
        item_id="latent_graft_source",
        duration=SAME_GRAFT_DURATION,
    )
    donor_item = encode_audio_file_to_item(
        SAME_GRAFT_DONOR_AUDIO,
        item_id="latent_graft_donor",
        duration=SAME_GRAFT_DURATION,
    )
    source_latents = item_to_sa3_tensor(
        source_item,
        device=DEVICE,
        dtype=next(sa3_model.model.model.parameters()).dtype,
    )
    donor_latents = item_to_sa3_tensor(
        donor_item,
        device=DEVICE,
        dtype=next(sa3_model.model.model.parameters()).dtype,
    )

    player_paths = []
    player_labels = []
    player_metadata = []
    manifest = []

    source_path = SAME_GRAFT_OUTPUT_DIR / "source_reference.wav"
    donor_path = SAME_GRAFT_OUTPUT_DIR / "donor_reference.wav"
    torchaudio.save(str(source_path), source_audio.cpu(), source_sr)
    torchaudio.save(str(donor_path), donor_audio.cpu(), donor_sr)
    player_paths.extend([source_path, donor_path])
    player_labels.extend(["source reference", "donor reference"])
    player_metadata.extend(
        [
            {"kind": "source_reference", "path": str(source_path)},
            {"kind": "donor_reference", "path": str(donor_path)},
        ]
    )

    for variant_index, spec_dict in enumerate(SAME_GRAFT_SPECS):
        spec = graft_mask_spec_from_dict(spec_dict)
        name = safe_graft_audio_name(spec.name)
        seed = SAME_GRAFT_BASE_SEED + variant_index
        channels = select_latent_channels(source_latents, spec)
        sampler_spec = LatentMaskSpec(
            name=spec.name,
            mode=spec.mode,
            fraction=spec.fraction,
            seed=spec.seed,
            channels=tuple(channels),
            start_channel=spec.start_channel,
            block_size=spec.block_size,
        )
        entry = {
            "name": spec.name,
            "mode": spec.mode,
            "fraction": spec.fraction,
            "seed": spec.seed,
            "sampler_seed": seed,
            "graft_amount": SAME_GRAFT_AMOUNT,
            "selected_channel_count": len(channels),
            "selected_channels": channels,
            "source_audio": str(SAME_GRAFT_SOURCE_AUDIO),
            "donor_audio": str(SAME_GRAFT_DONOR_AUDIO),
            "outputs": {},
        }

        if SAME_GRAFT_RUN_DIRECT_DECODE:
            grafted_latents = graft_latent_channels(
                source_latents,
                donor_latents,
                channels,
                amount=SAME_GRAFT_AMOUNT,
            )
            direct_path = SAME_GRAFT_OUTPUT_DIR / f"direct_graft_{name}.wav"
            decode_sa3_latents_to_file_cropped(
                grafted_latents,
                direct_path,
                duration=SAME_GRAFT_DURATION,
            )
            entry["outputs"]["direct_same_decode"] = str(direct_path)
            player_paths.append(direct_path)
            player_labels.append(f"direct graft {spec.name} ({len(channels)} ch)")
            player_metadata.append(
                {
                    "kind": "direct_same_graft",
                    "variant": spec.name,
                    "mask_mode": spec.mode,
                    "graft_amount": SAME_GRAFT_AMOUNT,
                    "selected_channel_count": len(channels),
                    "selected_channels": channels,
                    "sampler_seed": seed,
                    "source_audio": str(SAME_GRAFT_SOURCE_AUDIO),
                    "donor_audio": str(SAME_GRAFT_DONOR_AUDIO),
                    "path": str(direct_path),
                }
            )

        if SAME_GRAFT_RUN_SA3_SAMPLER:
            result = selective_graft_sa3(
                sa3_model,
                source_audio,
                source_sr,
                donor_audio,
                donor_sr,
                spec=sampler_spec,
                prompt=SAME_GRAFT_PROMPT,
                duration=SAME_GRAFT_DURATION,
                steps=SAME_GRAFT_STEPS,
                cfg_scale=SAME_GRAFT_CFG,
                init_noise_level=SAME_GRAFT_AMOUNT,
                seed=seed,
            )
            sampled_path = SAME_GRAFT_OUTPUT_DIR / f"sa3_graft_{name}.wav"
            decode_sa3_latents_to_file_cropped(
                result.sampled_latents,
                sampled_path,
                duration=SAME_GRAFT_DURATION,
            )
            entry["outputs"]["sa3_grafted"] = str(sampled_path)
            player_paths.append(sampled_path)
            player_labels.append(f"SA3 graft {spec.name} ({len(channels)} ch)")
            player_metadata.append(
                {
                    "kind": "sa3_grafted",
                    "variant": spec.name,
                    "mask_mode": spec.mode,
                    "graft_amount": SAME_GRAFT_AMOUNT,
                    "selected_channel_count": len(channels),
                    "selected_channels": channels,
                    "sampler_seed": seed,
                    "source_audio": str(SAME_GRAFT_SOURCE_AUDIO),
                    "donor_audio": str(SAME_GRAFT_DONOR_AUDIO),
                    "path": str(sampled_path),
                }
            )
            entry.update(result.metadata)

            if SAME_GRAFT_SAVE_PT:
                pt_path = SAME_GRAFT_OUTPUT_DIR / f"{name}.pt"
                torch.save(
                    {
                        "source_latents": result.init_latents.detach().cpu(),
                        "donor_latents": result.donor_latents.detach().cpu(),
                        "mixed_latents": result.mixed_latents.detach().cpu(),
                        "sampled_latents": result.sampled_latents.detach().cpu(),
                        "metadata": result.metadata,
                    },
                    pt_path,
                )
                entry["outputs"]["pt"] = str(pt_path)

        manifest.append(entry)
        print(spec.name, "channels:", len(channels), channels[:24])

    manifest_path = SAME_GRAFT_OUTPUT_DIR / "manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print("saved manifest:", manifest_path)
    play_audio_files(
        player_paths,
        labels=player_labels,
        metadata=player_metadata,
        title="SA3-over-SAME Coupled Editing: Cross-Audio Latent Graft",
        max_embed_mb=180.0,
        annotation_path=SAME_GRAFT_ANNOTATION_PATH,
    )


## 4. SA3-over-SAME Coupled Editing. Blur Bottleneck Lab

Latent blur asks: what happens if we remove or smooth detail inside SAME space before decoding?

Temporal blur:

$$
z'_{c,t} = \sum_{\tau} K_\sigma(\tau) z_{c,t-\tau}
$$

Contiguous-frame box blur, closer to video blur:

$$
z'_{c,t} = \frac{1}{2r + 1}\sum_{k=-r}^{r} z_{c,t+k}
$$

One-sided motion blur:

$$
z'_{c,t} = \frac{1}{r + 1}\sum_{k=0}^{r} z_{c,t-k}
$$

or the same window looking forward in latent time. At SA3/SAME's approximate rate,
\(r=4\) means a centered window of \(9\) latent frames, roughly \(0.84\) seconds.

Channel blur:

$$
z'_{c,t} = \sum_{j} K_\sigma(j) z_{c-j,t}
$$

Low-rank blur:

$$
Z_{T \times C} \approx U_k \Sigma_k V_k^\top
$$

Detail attenuation:

$$
z' = \operatorname{blur}(z) + \gamma (z - \operatorname{blur}(z))
$$

where \(\gamma < 1\) suppresses latent detail residuals.

Latent sharpen / unsharp mask:

$$
z' = z + \beta (z - \operatorname{blur}(z))
$$

where \(\beta > 0\) amplifies detail residuals. This can emphasize transient-like or high-activity latent structure, but large values may push the latent off-manifold.

Latent FFT filters:

$$
\hat z'_{c,\omega} = g(\omega)\hat z_{c,\omega}
$$

where the FFT is over SAME latent frames, not decoded audio samples. This is a filter over latent-channel trajectories. It can damp fast latent-frame jitter before SAME decode, but it is not equivalent to an audio EQ.

Interpretation warnings:

```text
direct SAME decode = microscope, may be off-manifold or artifacty
SA3 polish        = blurred latent used as init_data with small noise, more manifold-like but less pure
channel blur      = probes whether channel index has structure; it may not
channel sharpen   = also a probe; channel adjacency is not guaranteed semantic
low-rank blur     = probes global vs fine latent components
FFT filter        = latent-time filter, not decoded-audio EQ
```


In [ ]:
# @title 4. SA3-over-SAME Coupled Editing. Blur bottleneck lab

RUN_SAME_BLUR_BOTTLENECK = False

SAME_BLUR_AUDIO = INPUT_DIR / "known_9s.wav"
SAME_BLUR_OUTPUT_DIR = OUTPUT_DIR / "same_blur_bottleneck"
SAME_BLUR_DURATION = 9.0
SAME_BLUR_PROMPT = ""
SAME_BLUR_STEPS = 20
SAME_BLUR_CFG = 1.0
SAME_BLUR_POLISH_NOISE = 0.06
SAME_BLUR_BASE_SEED = 900

# Main harshness knobs:
# - temporal_radius: r=1 is 3 latent frames (~0.28s), r=2 is 5 frames (~0.46s),
#   r=4 is 9 frames (~0.84s), r=8 is 17 frames (~1.58s).
# - strength: 0.0 is no edit, 0.10-0.35 is gentle, 1.0 is full replacement by the blurred latent.
# - SAME_BLUR_POLISH_NOISE: 0.03-0.08 is light SA3 projection, 0.12+ starts behaving more like regeneration.
# - SAME_BLUR_STEPS: 8 is fast/draft, 20 is usually much cleaner for off-manifold edits.
SAME_BLUR_RUN_DIRECT_DECODE = False
SAME_BLUR_RUN_SA3_POLISH = True
SAME_BLUR_SAVE_PT = True

# Optional filter on the final SA3-polished latent before SAME decode.
# This is useful when the sampler output is musically right but has latent-frame grit.
SAME_BLUR_POST_FILTER = None
# Example:
# SAME_BLUR_POST_FILTER = {
#     "name": "post_low_shelf_c065_g060_s025",
#     "mode": "fft_lowpass",
#     "filter_cutoff": 0.65,
#     "filter_high_gain": 0.60,
#     "strength": 0.25,
# }

# Gentle defaults. The earlier full-strength temporal blurs are intentionally not the default
# because SA3 polish often treats them as over-damaged latents.
SAME_BLUR_SPECS = [
    {"name": "box_r1_s015", "mode": "temporal", "temporal_kernel": "box", "temporal_radius": 1, "strength": 0.15},
    {"name": "box_r1_s030", "mode": "temporal", "temporal_kernel": "box", "temporal_radius": 1, "strength": 0.30},
    {"name": "box_r2_s020", "mode": "temporal", "temporal_kernel": "box", "temporal_radius": 2, "strength": 0.20},
    {"name": "box_r2_s040", "mode": "temporal", "temporal_kernel": "box", "temporal_radius": 2, "strength": 0.40},
    {"name": "gaussian_r2_s020", "mode": "temporal", "temporal_kernel": "gaussian", "temporal_radius": 2, "strength": 0.20},
    {"name": "gaussian_r4_s015", "mode": "temporal", "temporal_kernel": "gaussian", "temporal_radius": 4, "strength": 0.15},
    {"name": "motion_past_r2_s020", "mode": "temporal", "temporal_kernel": "box", "temporal_direction": "past", "temporal_radius": 2, "strength": 0.20},
    {"name": "motion_future_r2_s020", "mode": "temporal", "temporal_kernel": "box", "temporal_direction": "future", "temporal_radius": 2, "strength": 0.20},
    {"name": "channel_r1_s020", "mode": "channel", "channel_radius": 1, "strength": 0.20},
    {"name": "time_channel_t1_c1_s020", "mode": "temporal_channel", "temporal_kernel": "box", "temporal_radius": 1, "channel_radius": 1, "strength": 0.20},
    {"name": "filter_low_shelf_c065_g060_s025", "mode": "fft_lowpass", "filter_cutoff": 0.65, "filter_high_gain": 0.60, "strength": 0.25},
    {"name": "filter_low_shelf_c045_g070_s025", "mode": "fft_lowpass", "filter_cutoff": 0.45, "filter_high_gain": 0.70, "strength": 0.25},
    {"name": "filter_band_shelf_010_070_s025", "mode": "fft_bandpass", "filter_low_cutoff": 0.10, "filter_high_cutoff": 0.70, "filter_low_gain": 0.70, "filter_high_gain": 0.60, "strength": 0.25},
    {"name": "low_rank_8", "mode": "low_rank", "rank": 8, "strength": 1.0},
    {"name": "low_rank_24", "mode": "low_rank", "rank": 24, "strength": 1.0},
    {"name": "detail_gain_050", "mode": "detail_attenuate", "temporal_radius": 4, "detail_gain": 0.50, "strength": 1.0},
    {"name": "sharpen_r2_a025", "mode": "sharpen", "temporal_kernel": "box", "temporal_radius": 2, "sharpen_amount": 0.25, "strength": 1.0},
    {"name": "sharpen_r4_a050", "mode": "sharpen", "temporal_kernel": "box", "temporal_radius": 4, "sharpen_amount": 0.50, "strength": 1.0},
    {"name": "mean_blend_010", "mode": "mean_blend", "strength": 0.10},
]

# Optional harsh probes. Use these deliberately, e.g.:
# SAME_BLUR_SPECS = SAME_BLUR_HEAVY_SPECS
# or:
# SAME_BLUR_SPECS = SAME_BLUR_SPECS + SAME_BLUR_HEAVY_SPECS[:3]
SAME_BLUR_HEAVY_SPECS = [
    {"name": "heavy_box_r2_s100", "mode": "temporal", "temporal_kernel": "box", "temporal_radius": 2, "strength": 1.0},
    {"name": "heavy_box_r4_s100", "mode": "temporal", "temporal_kernel": "box", "temporal_radius": 4, "strength": 1.0},
    {"name": "heavy_box_r8_s100", "mode": "temporal", "temporal_kernel": "box", "temporal_radius": 8, "strength": 1.0},
    {"name": "heavy_motion_past_r4_s100", "mode": "temporal", "temporal_kernel": "box", "temporal_direction": "past", "temporal_radius": 4, "strength": 1.0},
    {"name": "heavy_motion_future_r4_s100", "mode": "temporal", "temporal_kernel": "box", "temporal_direction": "future", "temporal_radius": 4, "strength": 1.0},
    {"name": "heavy_gaussian_r4_s100", "mode": "temporal", "temporal_kernel": "gaussian", "temporal_radius": 4, "strength": 1.0},
    {"name": "heavy_channel_r4_s100", "mode": "channel", "channel_radius": 4, "strength": 1.0},
    {"name": "heavy_time_channel_s075", "mode": "temporal_channel", "temporal_kernel": "box", "temporal_radius": 3, "channel_radius": 2, "strength": 0.75},
]


def latent_blur_spec_from_dict(payload):
    return LatentBlurSpec(
        name=payload["name"],
        mode=payload.get("mode", "temporal"),
        strength=float(payload.get("strength", 1.0)),
        temporal_radius=int(payload.get("temporal_radius", 4)),
        temporal_sigma=payload.get("temporal_sigma"),
        temporal_kernel=payload.get("temporal_kernel", "gaussian"),
        temporal_direction=payload.get("temporal_direction", "centered"),
        channel_radius=int(payload.get("channel_radius", 2)),
        channel_sigma=payload.get("channel_sigma"),
        rank=int(payload.get("rank", 16)),
        detail_gain=float(payload.get("detail_gain", 0.25)),
        sharpen_amount=float(payload.get("sharpen_amount", 0.5)),
        filter_cutoff=float(payload.get("filter_cutoff", 0.5)),
        filter_low_cutoff=float(payload.get("filter_low_cutoff", 0.1)),
        filter_high_cutoff=float(payload.get("filter_high_cutoff", 0.6)),
        filter_low_gain=float(payload.get("filter_low_gain", 0.0)),
        filter_mid_gain=float(payload.get("filter_mid_gain", 1.0)),
        filter_high_gain=float(payload.get("filter_high_gain", 0.0)),
    )


if RUN_SAME_BLUR_BOTTLENECK:
    require_models()
    SAME_BLUR_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    source_audio, source_sr = load_audio(
        SAME_BLUR_AUDIO,
        duration=SAME_BLUR_DURATION,
        stereo=True,
    )
    source_item = encode_audio_file_to_item(
        SAME_BLUR_AUDIO,
        item_id="latent_blur_source",
        duration=SAME_BLUR_DURATION,
    )
    source_latents = item_to_sa3_tensor(
        source_item,
        device=DEVICE,
        dtype=next(sa3_model.model.model.parameters()).dtype,
    )

    player_paths = []
    player_labels = []
    manifest = []
    source_path = SAME_BLUR_OUTPUT_DIR / "source_reference.wav"
    torchaudio.save(str(source_path), source_audio.cpu(), source_sr)
    player_paths.append(source_path)
    player_labels.append("source reference")

    for variant_index, spec_dict in enumerate(SAME_BLUR_SPECS):
        spec = latent_blur_spec_from_dict(spec_dict)
        name = safe_audio_name(spec.name)
        seed = SAME_BLUR_BASE_SEED + variant_index
        blurred_latents = apply_latent_blur(source_latents, spec)
        entry = {
            "name": spec.name,
            "mode": spec.mode,
            "strength": spec.strength,
            "temporal_radius": spec.temporal_radius,
            "temporal_kernel": spec.temporal_kernel,
            "temporal_direction": spec.temporal_direction,
            "channel_radius": spec.channel_radius,
            "rank": spec.rank,
            "detail_gain": spec.detail_gain,
            "sharpen_amount": spec.sharpen_amount,
            "filter_cutoff": spec.filter_cutoff,
            "filter_low_cutoff": spec.filter_low_cutoff,
            "filter_high_cutoff": spec.filter_high_cutoff,
            "filter_low_gain": spec.filter_low_gain,
            "filter_mid_gain": spec.filter_mid_gain,
            "filter_high_gain": spec.filter_high_gain,
            "polish_noise": SAME_BLUR_POLISH_NOISE,
            "polish_steps": SAME_BLUR_STEPS,
            "seed": seed,
            "outputs": {},
        }

        if SAME_BLUR_RUN_DIRECT_DECODE:
            direct_path = SAME_BLUR_OUTPUT_DIR / f"direct_{name}.wav"
            decode_sa3_latents_to_file_cropped(
                blurred_latents,
                direct_path,
                duration=SAME_BLUR_DURATION,
            )
            entry["outputs"]["direct_same_decode"] = str(direct_path)
            player_paths.append(direct_path)
            player_labels.append(f"direct {spec.name}")

        if SAME_BLUR_RUN_SA3_POLISH:
            polished_latents = sa3_sample_from_init_latents(
                sa3_model,
                blurred_latents,
                prompt=SAME_BLUR_PROMPT,
                duration=SAME_BLUR_DURATION,
                steps=SAME_BLUR_STEPS,
                cfg_scale=SAME_BLUR_CFG,
                init_noise_level=SAME_BLUR_POLISH_NOISE,
                seed=seed,
            )
            if SAME_BLUR_POST_FILTER is not None:
                post_filter_spec = latent_blur_spec_from_dict(SAME_BLUR_POST_FILTER)
                polished_latents = apply_latent_blur(polished_latents, post_filter_spec)
                entry["post_filter"] = dict(SAME_BLUR_POST_FILTER)
            polished_path = SAME_BLUR_OUTPUT_DIR / f"sa3_polish_{name}.wav"
            decode_sa3_latents_to_file_cropped(
                polished_latents,
                polished_path,
                duration=SAME_BLUR_DURATION,
            )
            entry["outputs"]["sa3_polished"] = str(polished_path)
            player_paths.append(polished_path)
            player_labels.append(f"SA3 polish {spec.name}")

            if SAME_BLUR_SAVE_PT:
                pt_path = SAME_BLUR_OUTPUT_DIR / f"{name}.pt"
                torch.save(
                    {
                        "source_latents": source_latents.detach().cpu(),
                        "blurred_latents": blurred_latents.detach().cpu(),
                        "polished_latents": polished_latents.detach().cpu(),
                        "spec": spec_dict,
                    },
                    pt_path,
                )
                entry["outputs"]["pt"] = str(pt_path)
        elif SAME_BLUR_SAVE_PT:
            pt_path = SAME_BLUR_OUTPUT_DIR / f"{name}.pt"
            torch.save(
                {
                    "source_latents": source_latents.detach().cpu(),
                    "blurred_latents": blurred_latents.detach().cpu(),
                    "spec": spec_dict,
                },
                pt_path,
            )
            entry["outputs"]["pt"] = str(pt_path)

        manifest.append(entry)
        print(spec.name, "operation:", spec.mode)

    manifest_path = SAME_BLUR_OUTPUT_DIR / "manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print("saved manifest:", manifest_path)
    play_audio_files(
        player_paths,
        labels=player_labels,
        title="SA3-over-SAME Coupled Editing: Blur Bottleneck Lab",
        max_embed_mb=180.0,
    )


## 4. SA3-over-SAME Coupled Editing. Neural Latent DSP Lab

This experiment treats SAME latents as a learned 256-channel, low-rate signal:

$$
z \in \mathbb{R}^{B \times 256 \times T},
\qquad \mathrm{latent\_rate} \approx 10.77\ \mathrm{Hz}
$$

The operators are DSP-like, but not waveform DSP:

$$
D(O_z(E(x))) \neq O_x(x)
$$

So each operation is a hypothesis about the neural compressed representation.
The experiment can direct-decode the edited latent as a microscope and/or use SA3 as
a manifold reprojector:

```text
z_source -> O_z(z_source) -> SAME decode
z_source -> O_z(z_source) -> SA3 init_data polish -> SAME decode
```

Included operators:

```text
latent gain/compression/expansion/soft clipping
latent-time FFT EQ
latent-time FFT phase shift/randomization
donor magnitude / source phase grafts
per-clip PCA component gain, like learned macro-EQ
audio descriptor audit after decode
```

The MIR descriptors are audit signals, not final truth. Use them to ask whether
an edit moved brightness, flux, loudness, stereo width, etc. in a repeatable way.


In [ ]:
# @title 4. SA3-over-SAME Coupled Editing. Neural latent DSP lab

RUN_SAME_NEURAL_DSP = False

SAME_DSP_AUDIO = INPUT_DIR / "known_9s.wav"
SAME_DSP_DONOR_AUDIO = INPUT_DIR / "donor_9s.wav"
SAME_DSP_OUTPUT_DIR = OUTPUT_DIR / "same_neural_dsp"
SAME_DSP_ANNOTATION_PATH = OUTPUT_DIR / "same_neural_dsp_annotations.jsonl"
SAME_DSP_DURATION = 9.0
SAME_DSP_PROMPT = ""
SAME_DSP_STEPS = 20
SAME_DSP_CFG = 1.0
SAME_DSP_POLISH_NOISE = 0.06
SAME_DSP_BASE_SEED = 1200
SAME_DSP_RUN_DIRECT_DECODE = False
SAME_DSP_RUN_SA3_POLISH = True
SAME_DSP_SAVE_PT = True
SAME_DSP_INCLUDE_DONOR_SPECS = False

SAME_DSP_SPECS = [
    {
        "name": "gain_expand_115",
        "mode": "gain",
        "gain": 1.15,
        "center": "channel_mean",
        "strength": 1.0,
    },
    {
        "name": "compress_t080_r400",
        "mode": "compress",
        "threshold": 0.80,
        "ratio": 4.0,
        "center": "channel_mean",
        "strength": 1.0,
    },
    {
        "name": "expand_t130_r160",
        "mode": "expand",
        "threshold": 1.30,
        "ratio": 1.60,
        "center": "channel_mean",
        "strength": 0.70,
    },
    {
        "name": "softclip_drive160",
        "mode": "softclip",
        "drive": 1.60,
        "ceiling": 1.75,
        "center": "channel_mean",
        "strength": 1.0,
    },
    {
        "name": "fft_eq_slow_boost_fast_damp",
        "mode": "fft_eq",
        "fft_low_cutoff": 0.12,
        "fft_high_cutoff": 0.62,
        "fft_low_gain": 1.20,
        "fft_mid_gain": 1.00,
        "fft_high_gain": 0.70,
        "strength": 0.75,
    },
    {
        "name": "fft_eq_fast_push",
        "mode": "fft_eq",
        "fft_low_cutoff": 0.15,
        "fft_high_cutoff": 0.70,
        "fft_low_gain": 0.90,
        "fft_mid_gain": 1.00,
        "fft_high_gain": 1.30,
        "strength": 0.45,
    },
    {
        "name": "phase_shift_quarter",
        "mode": "fft_phase_shift",
        "phase_shift_fraction": 0.25,
        "strength": 0.40,
    },
    {
        "name": "phase_random_012",
        "mode": "fft_phase_randomize",
        "phase_random_amount": 0.12,
        "seed": 123,
        "strength": 1.0,
    },
    {
        "name": "pca_macro_1_boost",
        "mode": "pca_gain",
        "pca_component_gains": [1.20, 1.00, 1.00, 1.00, 1.00, 1.00],
        "strength": 0.70,
    },
    {
        "name": "pca_macro_top_smooth",
        "mode": "pca_gain",
        "pca_component_gains": [0.85, 0.90, 0.95, 1.00, 1.00, 1.00],
        "strength": 0.80,
    },
]

SAME_DSP_DONOR_SPECS = [
    {
        "name": "donor_magnitude_source_phase",
        "mode": "fft_mag_phase_graft",
        "magnitude_amount": 0.50,
        "strength": 1.0,
    },
    {
        "name": "source_magnitude_donor_phase",
        "mode": "fft_phase_blend",
        "phase_blend_amount": 0.35,
        "strength": 1.0,
    },
]


def latent_dsp_spec_from_dict(payload):
    gains = payload.get("pca_component_gains")
    return LatentDSPSpec(
        name=payload["name"],
        mode=payload.get("mode", "gain"),
        strength=float(payload.get("strength", 1.0)),
        gain=float(payload.get("gain", 1.0)),
        center=payload.get("center", "channel_mean"),
        threshold=float(payload.get("threshold", 1.0)),
        ratio=float(payload.get("ratio", 4.0)),
        makeup_gain=float(payload.get("makeup_gain", 1.0)),
        drive=float(payload.get("drive", 1.0)),
        ceiling=float(payload.get("ceiling", 2.0)),
        fft_low_cutoff=float(payload.get("fft_low_cutoff", 0.15)),
        fft_high_cutoff=float(payload.get("fft_high_cutoff", 0.65)),
        fft_low_gain=float(payload.get("fft_low_gain", 1.0)),
        fft_mid_gain=float(payload.get("fft_mid_gain", 1.0)),
        fft_high_gain=float(payload.get("fft_high_gain", 1.0)),
        phase_shift_fraction=float(payload.get("phase_shift_fraction", 0.0)),
        phase_random_amount=float(payload.get("phase_random_amount", 1.0)),
        phase_blend_amount=float(payload.get("phase_blend_amount", 1.0)),
        magnitude_amount=float(payload.get("magnitude_amount", 1.0)),
        pca_rank=payload.get("pca_rank"),
        pca_component_gains=tuple(float(v) for v in gains) if gains is not None else None,
        seed=int(payload.get("seed", 0)),
    )


def latent_dsp_operation_requires_donor(operation):
    return operation.lower() in {
        "fft_mag_phase_graft",
        "mag_phase_graft",
        "magnitude_from_donor",
        "donor_magnitude",
        "fft_phase_from_donor",
        "donor_phase",
        "fft_phase_blend",
        "phase_blend",
    }


def tensor_periodicity_dict(latents, item_id):
    item = sa3_tensor_to_item(latents, item_id=item_id)
    report = periodicity_report(item)
    return {
        "best_lag": report.best_lag,
        "best_score": report.best_score,
        "boundary_l2": report.boundary_l2,
        "velocity_l2": report.velocity_l2,
        "spectral_centroid": report.spectral_centroid,
    }


if RUN_SAME_NEURAL_DSP:
    require_models()
    SAME_DSP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    source_audio, source_sr = load_audio(
        SAME_DSP_AUDIO,
        duration=SAME_DSP_DURATION,
        stereo=True,
    )
    source_descriptors = audio_descriptor_report(source_audio, source_sr)
    source_item = encode_audio_file_to_item(
        SAME_DSP_AUDIO,
        item_id="latent_dsp_source",
        duration=SAME_DSP_DURATION,
    )
    source_latents = item_to_sa3_tensor(
        source_item,
        device=DEVICE,
        dtype=next(sa3_model.model.model.parameters()).dtype,
    )

    donor_latents = None
    if Path(SAME_DSP_DONOR_AUDIO).exists():
        donor_item = encode_audio_file_to_item(
            SAME_DSP_DONOR_AUDIO,
            item_id="latent_dsp_donor",
            duration=SAME_DSP_DURATION,
        )
        donor_latents = item_to_sa3_tensor(
            donor_item,
            device=DEVICE,
            dtype=next(sa3_model.model.model.parameters()).dtype,
        )

    specs = list(SAME_DSP_SPECS)
    if SAME_DSP_INCLUDE_DONOR_SPECS:
        specs.extend(SAME_DSP_DONOR_SPECS)

    player_paths = []
    player_labels = []
    manifest = []
    source_path = SAME_DSP_OUTPUT_DIR / "source_reference.wav"
    torchaudio.save(str(source_path), source_audio.cpu(), source_sr)
    player_paths.append(source_path)
    player_labels.append("source reference")
    source_periodicity = tensor_periodicity_dict(source_latents, "latent_dsp_source_periodicity")

    for variant_index, spec_dict in enumerate(specs):
        spec = latent_dsp_spec_from_dict(spec_dict)
        if latent_dsp_operation_requires_donor(spec.mode) and donor_latents is None:
            print("skipping donor operation without donor audio:", spec.name)
            continue
        name = safe_audio_name(spec.name)
        seed = SAME_DSP_BASE_SEED + variant_index
        edited_latents = apply_latent_dsp(source_latents, spec, donor_latents=donor_latents)
        latent_delta = latent_change_report(source_latents, edited_latents)
        entry = {
            "name": spec.name,
            "mode": spec.mode,
            "spec": dict(spec_dict),
            "polish_noise": SAME_DSP_POLISH_NOISE,
            "polish_steps": SAME_DSP_STEPS,
            "seed": seed,
            "latent_delta": latent_delta,
            "source_periodicity": source_periodicity,
            "edited_periodicity": tensor_periodicity_dict(edited_latents, f"{name}_edited_periodicity"),
            "source_audio_descriptors": source_descriptors,
            "outputs": {},
        }

        if SAME_DSP_RUN_DIRECT_DECODE:
            direct_path = SAME_DSP_OUTPUT_DIR / f"direct_{name}.wav"
            direct_path, direct_desc = decode_sa3_latents_to_file_with_descriptors(
                edited_latents,
                direct_path,
                duration=SAME_DSP_DURATION,
            )
            entry["outputs"]["direct_same_decode"] = str(direct_path)
            entry["direct_audio_descriptors"] = direct_desc
            entry["direct_descriptor_delta"] = descriptor_delta(source_descriptors, direct_desc)
            player_paths.append(direct_path)
            player_labels.append(f"direct {spec.name}")

        if SAME_DSP_RUN_SA3_POLISH:
            polished_latents = sa3_sample_from_init_latents(
                sa3_model,
                edited_latents,
                prompt=SAME_DSP_PROMPT,
                duration=SAME_DSP_DURATION,
                steps=SAME_DSP_STEPS,
                cfg_scale=SAME_DSP_CFG,
                init_noise_level=SAME_DSP_POLISH_NOISE,
                seed=seed,
            )
            polished_path = SAME_DSP_OUTPUT_DIR / f"sa3_polish_{name}.wav"
            polished_path, polished_desc = decode_sa3_latents_to_file_with_descriptors(
                polished_latents,
                polished_path,
                duration=SAME_DSP_DURATION,
            )
            entry["outputs"]["sa3_polished"] = str(polished_path)
            entry["polished_latent_delta"] = latent_change_report(source_latents, polished_latents)
            entry["polished_periodicity"] = tensor_periodicity_dict(polished_latents, f"{name}_polished_periodicity")
            entry["polished_audio_descriptors"] = polished_desc
            entry["polished_descriptor_delta"] = descriptor_delta(source_descriptors, polished_desc)
            player_paths.append(polished_path)
            player_labels.append(f"SA3 polish {spec.name}")

            if SAME_DSP_SAVE_PT:
                pt_path = SAME_DSP_OUTPUT_DIR / f"{name}.pt"
                torch.save(
                    {
                        "source_latents": source_latents.detach().cpu(),
                        "donor_latents": donor_latents.detach().cpu() if donor_latents is not None else None,
                        "edited_latents": edited_latents.detach().cpu(),
                        "polished_latents": polished_latents.detach().cpu(),
                        "spec": spec_dict,
                    },
                    pt_path,
                )
                entry["outputs"]["pt"] = str(pt_path)
        elif SAME_DSP_SAVE_PT:
            pt_path = SAME_DSP_OUTPUT_DIR / f"{name}.pt"
            torch.save(
                {
                    "source_latents": source_latents.detach().cpu(),
                    "donor_latents": donor_latents.detach().cpu() if donor_latents is not None else None,
                    "edited_latents": edited_latents.detach().cpu(),
                    "spec": spec_dict,
                },
                pt_path,
            )
            entry["outputs"]["pt"] = str(pt_path)

        manifest.append(entry)
        print(spec.name, "operation:", spec.mode, "delta_rms:", round(latent_delta["delta_rms"], 5))

    manifest_path = SAME_DSP_OUTPUT_DIR / "manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print("saved manifest:", manifest_path)
    play_audio_files(
        player_paths,
        labels=player_labels,
        title="SA3-over-SAME Coupled Editing: Neural Latent DSP Lab",
        max_embed_mb=180.0,
        annotation_path=SAME_DSP_ANNOTATION_PATH,
    )


## 4. SA3-over-SAME Coupled Editing. Cyclic Loop Repair Lab

This is a cyclic-origin shift: move the boundary into the interior, let SA3/SAME repair that region, then restore the original temporal origin.

Cyclic roll operator:

$$
R_s(z)_t = z_{(t-s) \bmod T}
$$

Latent roll polish:

$$
z_{\mathrm{rolled}} = R_s(z),\quad
\tilde{z}_{\mathrm{rolled}} = F_\theta(z_{\mathrm{rolled}}),\quad
\tilde{z} = R_{-s}(\tilde{z}_{\mathrm{rolled}})
$$

Audio seam inpaint:

```text
roll audio by half
the original loop boundary is now in the middle
inpaint a small window around that middle seam
roll audio back
repeat it several times and listen to the seam
```

This is not a perfect-loop guarantee. It is a cyclic augmentation experiment that gives SA3 a chance to repair the loop boundary as normal musical interior rather than as the edge of a file.

Important distinction:

- The single-pass probes below roll once, repair once, then unroll.
- The iterative probes roll by a schedule such as `[0.5, 0.25, 0.75]`, repair, unroll, then repeat from the repaired audio/latents. This repeatedly changes which boundary is exposed as ordinary interior material.
- This still is not "inside every diffusion step". Doing that would require sampler-level surgery: applying cyclic shifts, loop losses, or seam guidance inside the denoising/flow loop.


In [ ]:
# @title 4. SA3-over-SAME Coupled Editing. Cyclic loop repair lab

RUN_SAME_LOOP_REPAIR = False

LOOP_AUDIO = INPUT_DIR / "known_9s.wav"
LOOP_OUTPUT_DIR = OUTPUT_DIR / "same_loop_repair"
LOOP_DURATION = 9.0
LOOP_PROMPT = ""
LOOP_STEPS = 20
LOOP_CFG = 1.0
LOOP_BASE_SEED = 970

LOOP_SHIFT_FRACTIONS = [0.5]
LOOP_PREVIEW_REPEATS = 4

# Latent path: roll SAME latents, SA3-polish, unroll, decode.
LOOP_RUN_SAME_ROLL_POLISH = True
LOOP_SAME_POLISH_NOISE = 0.25

# Audio path: roll waveform, inpaint seam window, unroll waveform.
LOOP_RUN_AUDIO_ROLL_INPAINT = True
LOOP_INPAINT_WINDOW_SECONDS = 1.0

# Iterative cyclic augmentation. These work on arbitrary audio clips, not only
# pre-existing loops. They repeatedly move the temporal origin so several
# would-be seams become ordinary interior material.
LOOP_RUN_ITERATIVE_SAME_ROLL_POLISH = False
LOOP_RUN_ITERATIVE_AUDIO_ROLL_INPAINT = False
LOOP_ITERATIONS = 3
LOOP_ITERATIVE_SHIFT_FRACTIONS = [0.5, 0.25, 0.75]
LOOP_ITERATIVE_STEPS = 12
LOOP_ITERATIVE_SAME_POLISH_NOISE = 0.14


def safe_loop_name(value):
    return "".join(ch if ch.isalnum() or ch in "-_." else "_" for ch in str(value))[:96]


def save_loop_preview(audio_path, repeats, label):
    audio, sample_rate = torchaudio.load(str(audio_path))
    preview = repeated_loop_preview_audio(audio, repeats=repeats).cpu().float().clamp(-1, 1)
    preview_path = audio_path.with_name(f"{audio_path.stem}_x{repeats}_{safe_loop_name(label)}.wav")
    torchaudio.save(str(preview_path), preview, sample_rate)
    return preview_path


def metrics_payload(latents, *, k=8):
    metrics = loop_boundary_metrics(latents, window_frames=k)
    return {
        "state_l2": metrics.state_l2,
        "velocity_l2": metrics.velocity_l2,
        "total": metrics.total,
        "window_frames": metrics.window_frames,
    }


def cyclic_schedule_value(schedule, index):
    if not schedule:
        raise ValueError("shift schedule cannot be empty")
    return float(schedule[index % len(schedule)])


if RUN_SAME_LOOP_REPAIR:
    require_models()
    LOOP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    source_audio, source_sr = load_audio(
        LOOP_AUDIO,
        target_sample_rate=sa3.sample_rate,
        duration=LOOP_DURATION,
        stereo=True,
    )
    source_item = encode_audio_file_to_item(
        LOOP_AUDIO,
        item_id="loop_source",
        duration=LOOP_DURATION,
    )
    source_latents = item_to_sa3_tensor(
        source_item,
        device=DEVICE,
        dtype=next(sa3_model.model.model.parameters()).dtype,
    )

    manifest = []
    player_paths = []
    player_labels = []

    source_path = LOOP_OUTPUT_DIR / "source_reference.wav"
    torchaudio.save(str(source_path), source_audio.cpu(), source_sr)
    source_preview_path = save_loop_preview(source_path, LOOP_PREVIEW_REPEATS, "preview")
    player_paths.extend([source_path, source_preview_path])
    player_labels.extend(["source reference", f"source x{LOOP_PREVIEW_REPEATS} seam preview"])
    source_metrics = metrics_payload(source_latents)
    print("source latent loop metrics:", source_metrics)

    for index, shift_fraction in enumerate(LOOP_SHIFT_FRACTIONS):
        shift_tag = safe_loop_name(f"{shift_fraction:.3f}")
        seed = LOOP_BASE_SEED + index
        shift_frames = frames_from_fraction(source_latents, shift_fraction)
        shift_samples = samples_from_fraction(source_audio, shift_fraction)
        start_sec, end_sec = seam_inpaint_bounds(
            LOOP_DURATION,
            shift_fraction,
            LOOP_INPAINT_WINDOW_SECONDS,
        )
        entry = {
            "shift_fraction": float(shift_fraction),
            "shift_frames": int(shift_frames),
            "shift_samples": int(shift_samples),
            "seed": int(seed),
            "source_metrics": source_metrics,
            "outputs": {},
        }

        if LOOP_RUN_SAME_ROLL_POLISH:
            rolled_latents = cyclic_roll_latents(source_latents, shift_frames)
            polished_rolled = sa3_sample_from_init_latents(
                sa3_model,
                rolled_latents,
                prompt=LOOP_PROMPT,
                duration=LOOP_DURATION,
                steps=LOOP_STEPS,
                cfg_scale=LOOP_CFG,
                init_noise_level=LOOP_SAME_POLISH_NOISE,
                seed=seed,
            )
            unrolled_latents = cyclic_roll_latents(polished_rolled, -shift_frames)
            latent_path = LOOP_OUTPUT_DIR / f"latent_roll_polish_shift_{shift_tag}.wav"
            decode_sa3_latents_to_file_cropped(
                unrolled_latents,
                latent_path,
                duration=LOOP_DURATION,
            )
            latent_preview_path = save_loop_preview(latent_path, LOOP_PREVIEW_REPEATS, "preview")
            latent_metrics = metrics_payload(unrolled_latents)
            entry["latent_roll_polish"] = {
                "polish_noise": LOOP_SAME_POLISH_NOISE,
                "steps": LOOP_STEPS,
                "metrics": latent_metrics,
            }
            entry["outputs"]["latent_roll_polish"] = str(latent_path)
            entry["outputs"]["latent_roll_polish_preview"] = str(latent_preview_path)
            player_paths.extend([latent_path, latent_preview_path])
            player_labels.extend(
                [
                    f"SAME roll polish shift {shift_fraction:.2f}",
                    f"SAME roll polish shift {shift_fraction:.2f} x{LOOP_PREVIEW_REPEATS}",
                ]
            )
            print("latent roll metrics", shift_fraction, latent_metrics)

        if LOOP_RUN_AUDIO_ROLL_INPAINT:
            rolled_audio = cyclic_roll_audio(source_audio, shift_samples)
            with torch.inference_mode():
                repaired_rolled = sa3_model.generate(
                    prompt=LOOP_PROMPT,
                    duration=LOOP_DURATION,
                    steps=LOOP_STEPS,
                    cfg_scale=LOOP_CFG,
                    seed=seed + 100,
                    inpaint_audio=(source_sr, rolled_audio),
                    inpaint_mask_start_seconds=start_sec,
                    inpaint_mask_end_seconds=end_sec,
                    truncate_output_to_duration=True,
                )
            repaired_audio = cyclic_roll_audio(repaired_rolled[0], -shift_samples).cpu().float().clamp(-1, 1)
            inpaint_path = LOOP_OUTPUT_DIR / f"audio_roll_inpaint_shift_{shift_tag}.wav"
            torchaudio.save(str(inpaint_path), repaired_audio, sa3.sample_rate)
            inpaint_preview_path = save_loop_preview(inpaint_path, LOOP_PREVIEW_REPEATS, "preview")
            entry["audio_roll_inpaint"] = {
                "inpaint_start_seconds": start_sec,
                "inpaint_end_seconds": end_sec,
                "steps": LOOP_STEPS,
            }
            entry["outputs"]["audio_roll_inpaint"] = str(inpaint_path)
            entry["outputs"]["audio_roll_inpaint_preview"] = str(inpaint_preview_path)
            player_paths.extend([inpaint_path, inpaint_preview_path])
            player_labels.extend(
                [
                    f"audio roll inpaint shift {shift_fraction:.2f}",
                    f"audio roll inpaint shift {shift_fraction:.2f} x{LOOP_PREVIEW_REPEATS}",
                ]
            )

        manifest.append(entry)

    if LOOP_RUN_ITERATIVE_SAME_ROLL_POLISH:
        current_latents = source_latents
        trace = []
        for iteration in range(int(LOOP_ITERATIONS)):
            shift_fraction = cyclic_schedule_value(LOOP_ITERATIVE_SHIFT_FRACTIONS, iteration)
            shift_frames = frames_from_fraction(current_latents, shift_fraction)
            rolled_latents = cyclic_roll_latents(current_latents, shift_frames)
            polished_rolled = sa3_sample_from_init_latents(
                sa3_model,
                rolled_latents,
                prompt=LOOP_PROMPT,
                duration=LOOP_DURATION,
                steps=LOOP_ITERATIVE_STEPS,
                cfg_scale=LOOP_CFG,
                init_noise_level=LOOP_ITERATIVE_SAME_POLISH_NOISE,
                seed=LOOP_BASE_SEED + 1000 + iteration,
            )
            current_latents = cyclic_roll_latents(polished_rolled, -shift_frames).detach()
            iteration_metrics = metrics_payload(current_latents)
            trace.append(
                {
                    "iteration": iteration,
                    "shift_fraction": shift_fraction,
                    "shift_frames": int(shift_frames),
                    "seed": int(LOOP_BASE_SEED + 1000 + iteration),
                    "metrics": iteration_metrics,
                }
            )
            print("iterative latent roll", iteration, shift_fraction, iteration_metrics)

        iterative_latent_path = LOOP_OUTPUT_DIR / "iterative_latent_roll_polish.wav"
        decode_sa3_latents_to_file_cropped(
            current_latents,
            iterative_latent_path,
            duration=LOOP_DURATION,
        )
        iterative_latent_preview_path = save_loop_preview(
            iterative_latent_path,
            LOOP_PREVIEW_REPEATS,
            "preview",
        )
        manifest.append(
            {
                "method": "iterative_latent_roll_polish",
                "iterations": int(LOOP_ITERATIONS),
                "shift_schedule": [float(v) for v in LOOP_ITERATIVE_SHIFT_FRACTIONS],
                "steps": int(LOOP_ITERATIVE_STEPS),
                "polish_noise": float(LOOP_ITERATIVE_SAME_POLISH_NOISE),
                "trace": trace,
                "outputs": {
                    "audio": str(iterative_latent_path),
                    "preview": str(iterative_latent_preview_path),
                },
            }
        )
        player_paths.extend([iterative_latent_path, iterative_latent_preview_path])
        player_labels.extend(
            [
                "iterative SAME roll polish",
                f"iterative SAME roll polish x{LOOP_PREVIEW_REPEATS}",
            ]
        )

    if LOOP_RUN_ITERATIVE_AUDIO_ROLL_INPAINT:
        current_audio = source_audio.cpu().float()
        trace = []
        for iteration in range(int(LOOP_ITERATIONS)):
            shift_fraction = cyclic_schedule_value(LOOP_ITERATIVE_SHIFT_FRACTIONS, iteration)
            shift_samples = samples_from_fraction(current_audio, shift_fraction)
            start_sec, end_sec = seam_inpaint_bounds(
                LOOP_DURATION,
                shift_fraction,
                LOOP_INPAINT_WINDOW_SECONDS,
            )
            rolled_audio = cyclic_roll_audio(current_audio, shift_samples).cpu().float()
            with torch.inference_mode():
                repaired_rolled = sa3_model.generate(
                    prompt=LOOP_PROMPT,
                    duration=LOOP_DURATION,
                    steps=LOOP_ITERATIVE_STEPS,
                    cfg_scale=LOOP_CFG,
                    seed=LOOP_BASE_SEED + 2000 + iteration,
                    inpaint_audio=(source_sr, rolled_audio),
                    inpaint_mask_start_seconds=start_sec,
                    inpaint_mask_end_seconds=end_sec,
                    truncate_output_to_duration=True,
                )
            current_audio = cyclic_roll_audio(repaired_rolled[0], -shift_samples).cpu().float().clamp(-1, 1)
            trace.append(
                {
                    "iteration": iteration,
                    "shift_fraction": shift_fraction,
                    "shift_samples": int(shift_samples),
                    "seed": int(LOOP_BASE_SEED + 2000 + iteration),
                    "inpaint_start_seconds": float(start_sec),
                    "inpaint_end_seconds": float(end_sec),
                }
            )
            print("iterative audio roll", iteration, shift_fraction, (start_sec, end_sec))

        iterative_audio_path = LOOP_OUTPUT_DIR / "iterative_audio_roll_inpaint.wav"
        torchaudio.save(str(iterative_audio_path), current_audio.cpu(), sa3.sample_rate)
        iterative_audio_preview_path = save_loop_preview(
            iterative_audio_path,
            LOOP_PREVIEW_REPEATS,
            "preview",
        )
        manifest.append(
            {
                "method": "iterative_audio_roll_inpaint",
                "iterations": int(LOOP_ITERATIONS),
                "shift_schedule": [float(v) for v in LOOP_ITERATIVE_SHIFT_FRACTIONS],
                "steps": int(LOOP_ITERATIVE_STEPS),
                "inpaint_window_seconds": float(LOOP_INPAINT_WINDOW_SECONDS),
                "trace": trace,
                "outputs": {
                    "audio": str(iterative_audio_path),
                    "preview": str(iterative_audio_preview_path),
                },
            }
        )
        player_paths.extend([iterative_audio_path, iterative_audio_preview_path])
        player_labels.extend(
            [
                "iterative audio roll inpaint",
                f"iterative audio roll inpaint x{LOOP_PREVIEW_REPEATS}",
            ]
        )

    manifest_path = LOOP_OUTPUT_DIR / "manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print("saved manifest:", manifest_path)
    play_audio_files(
        player_paths,
        labels=player_labels,
        title="SA3-over-SAME Coupled Editing: Cyclic Loop Repair Lab",
        max_embed_mb=220.0,
    )


## 4. SA3-over-SAME Coupled Editing. SAME Style Profile

This edits the generated SAME latent before decoding.

Dataset statistics:

$$
\mu_D = \mathbb{E}_{i,t}[z_i(t)]
$$

$$
\sigma_D = \operatorname{std}_{i,t}[z_i(t)]
$$

AdaIN-like latent profile attraction:

$$
\tilde{z} =
\sigma_D \frac{z - \mu_z}{\sigma_z} + \mu_D
$$

$$
z' = (1 - \alpha)z + \alpha \tilde{z}
$$


In [ ]:
# @title 4. SA3-over-SAME Coupled Editing. SAME latent style profile

RUN_SAME_STYLE_PROFILE = False

STYLE_PROFILE_PATH = OUTPUT_DIR / "same_style_profile.npz"
STYLE_PROFILE_ALPHA = 0.75
STYLE_PROFILE_TEST_PROMPT = "a sparse evolving electronic texture, detailed, wide stereo"

if RUN_SAME_STYLE_PROFILE:
    require_models()
    run_duration = dataset_effective_duration()
    dataset_items = encode_audio_folder_to_items(
        DATASET_DIR,
        limit=DATASET_LIMIT,
        duration=DATASET_DURATION,
        prompt_from_path=True,
        use_chunks=DATASET_USE_CHUNKS,
        chunk_duration=DATASET_CHUNK_DURATION,
        hop_duration=DATASET_HOP_DURATION,
        max_chunks_per_file=DATASET_MAX_CHUNKS_PER_FILE,
        drop_last=DATASET_DROP_LAST_CHUNK,
    )
    profile = fit_style_profile(dataset_items, name="positive_dataset")
    save_style_profile(profile, STYLE_PROFILE_PATH)

    latents = sa3.generate_latents(
        prompt=STYLE_PROFILE_TEST_PROMPT,
        duration=run_duration,
        steps=8,
        cfg_scale=1.0,
        seed=42,
    )
    item = sa3_tensor_to_item(latents, item_id="profile_source", prompt=STYLE_PROFILE_TEST_PROMPT)
    styled_z = apply_profile_attraction(item, profile, alpha=STYLE_PROFILE_ALPHA, match_std=True)
    styled_latents = torch.from_numpy(styled_z.T[None, :, :]).to(
        device=DEVICE,
        dtype=latents.dtype,
    )
    out_path = OUTPUT_DIR / "same_style_profile_output.wav"
    decode_sa3_latents_to_file(styled_latents, out_path)
    print("saved:", out_path)
    play_audio_files([out_path], title="SA3-over-SAME Coupled Editing: Profile Styled Output")


## 4. SA3-over-SAME Coupled Editing. SAME Style Direction

A contrastive direction in SAME latent space:

$$
v_{A-B} = \mu_A - \mu_B
$$

Apply:

$$
z' = z + \alpha v_{A-B}
$$

This requires a reference/baseline set if you want a true direction. With only positive examples, use style-profile attraction instead.


In [ ]:
# @title 4. SA3-over-SAME Coupled Editing. SAME latent direction from positive/reference folders

RUN_SAME_STYLE_DIRECTION = False

STYLE_DIRECTION_PATH = OUTPUT_DIR / "same_style_direction.npz"
STYLE_DIRECTION_ALPHA = 1.0

if RUN_SAME_STYLE_DIRECTION:
    require_models()
    run_duration = dataset_effective_duration()
    positive_items = encode_audio_folder_to_items(
        DATASET_DIR,
        limit=DATASET_LIMIT,
        duration=DATASET_DURATION,
        use_chunks=DATASET_USE_CHUNKS,
        chunk_duration=DATASET_CHUNK_DURATION,
        hop_duration=DATASET_HOP_DURATION,
        max_chunks_per_file=DATASET_MAX_CHUNKS_PER_FILE,
        drop_last=DATASET_DROP_LAST_CHUNK,
    )
    reference_items = encode_audio_folder_to_items(
        REFERENCE_DIR,
        limit=DATASET_LIMIT,
        duration=DATASET_DURATION,
        use_chunks=DATASET_USE_CHUNKS,
        chunk_duration=DATASET_CHUNK_DURATION,
        hop_duration=DATASET_HOP_DURATION,
        max_chunks_per_file=DATASET_MAX_CHUNKS_PER_FILE,
        drop_last=DATASET_DROP_LAST_CHUNK,
    )
    positive_profile = fit_style_profile(positive_items, name="positive")
    reference_profile = fit_style_profile(reference_items, name="reference")
    direction = style_direction(positive_profile, reference_profile, name="positive_minus_reference")
    save_style_direction(direction, STYLE_DIRECTION_PATH)

    latents = sa3.generate_latents(
        prompt=STYLE_PROFILE_TEST_PROMPT,
        duration=run_duration,
        steps=8,
        cfg_scale=1.0,
        seed=43,
    )
    item = sa3_tensor_to_item(latents, item_id="direction_source", prompt=STYLE_PROFILE_TEST_PROMPT)
    steered_z = apply_style_direction(item, direction, alpha=STYLE_DIRECTION_ALPHA)
    steered_latents = torch.from_numpy(steered_z.T[None, :, :]).to(device=DEVICE, dtype=latents.dtype)
    out_path = OUTPUT_DIR / "same_style_direction_output.wav"
    decode_sa3_latents_to_file(steered_latents, out_path)
    print("saved:", out_path)
    play_audio_files([out_path], title="SA3-over-SAME Coupled Editing: Direction Output")


In [ ]:
# @title 4. SA3-over-SAME Coupled Editing. Latent OT style transfer bench

RUN_SAME_OT_STYLE_TRANSFER = False

SAME_OT_SOURCE_AUDIO = INPUT_DIR / "source.wav"
SAME_OT_REFERENCE_DIR = INPUT_DIR / "reference_style"
SAME_OT_DURATION = 12.0
SAME_OT_ALPHA = 0.75
SAME_OT_PROMPT = "audio texture"
SAME_OT_POLISH = True
SAME_OT_POLISH_NOISE = 0.08
SAME_OT_POLISH_STEPS = 8
SAME_OT_OUTPUT_DIR = OUTPUT_DIR / "same_ot_style_transfer"


def same_ot_item_to_output(item, name):
    tensor = item_to_sa3_tensor(item, device=DEVICE)
    path = SAME_OT_OUTPUT_DIR / f"{name}.wav"
    decode_sa3_latents_to_file_cropped(tensor, path, duration=SAME_OT_DURATION)
    return path


if RUN_SAME_OT_STYLE_TRANSFER:
    require_models()
    SAME_OT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    source_item = encode_audio_file_to_item(SAME_OT_SOURCE_AUDIO, item_id="same_ot_source", duration=SAME_OT_DURATION)
    reference_items = encode_audio_folder_to_items(SAME_OT_REFERENCE_DIR, duration=SAME_OT_DURATION, limit=DATASET_LIMIT)
    if not reference_items:
        raise ValueError("Latent OT style transfer needs at least one reference audio file.")

    profile = fit_style_profile(reference_items, name="same_ot_reference")
    profile_item = apply_profile_to_item(source_item, profile, alpha=SAME_OT_ALPHA, item_id="same_ot_profile")
    transported = covariance_transport(source_item, reference_items, alpha=SAME_OT_ALPHA)
    transport_item = LatentItem(
        item_id="same_ot_covariance_transport",
        latent=transported,
        latent_rate=source_item.latent_rate,
        sample_rate=source_item.sample_rate,
        prompt=SAME_OT_PROMPT,
        metadata={"operator": "covariance_transport", "alpha": SAME_OT_ALPHA},
    )
    output_items = [source_item, profile_item, transport_item]
    equal_length_refs = [item for item in reference_items if item.latent.shape == source_item.latent.shape]
    if equal_length_refs:
        bary = latent_barycenter([source_item, equal_length_refs[0]], weights=[1.0 - SAME_OT_ALPHA, SAME_OT_ALPHA])
        output_items.append(
            LatentItem(
                item_id="same_ot_barycenter",
                latent=bary,
                latent_rate=source_item.latent_rate,
                sample_rate=source_item.sample_rate,
                prompt=SAME_OT_PROMPT,
                metadata={"operator": "latent_barycenter", "alpha": SAME_OT_ALPHA},
            )
        )

    output_paths = []
    labels = []
    reports = []
    for item in output_items:
        direct_path = same_ot_item_to_output(item, item.item_id)
        output_paths.append(direct_path)
        labels.append(item.item_id)
        audio, sample_rate = load_audio(direct_path, stereo=True)
        reports.append({"item_id": item.item_id, "direct_path": str(direct_path), "descriptors": audio_descriptor_report(audio, sample_rate)})
        if SAME_OT_POLISH and item.item_id != source_item.item_id:
            init_tensor = item_to_sa3_tensor(item, device=DEVICE)
            polished = sa3_sample_from_init_latents(
                sa3_model,
                init_tensor,
                prompt=SAME_OT_PROMPT,
                duration=SAME_OT_DURATION,
                steps=SAME_OT_POLISH_STEPS,
                init_noise_level=SAME_OT_POLISH_NOISE,
                seed=0,
            )
            polished_path = SAME_OT_OUTPUT_DIR / f"{item.item_id}_polished.wav"
            decode_sa3_latents_to_file_cropped(polished, polished_path, duration=SAME_OT_DURATION)
            output_paths.append(polished_path)
            labels.append(f"{item.item_id} polished")
            reports[-1]["polished_path"] = str(polished_path)

    payload = {"source": str(SAME_OT_SOURCE_AUDIO), "reference_dir": str(SAME_OT_REFERENCE_DIR), "reports": reports}
    (SAME_OT_OUTPUT_DIR / "report.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")
    play_audio_files(output_paths, labels=labels, title="SA3-over-SAME Coupled Editing: Latent OT Style Transfer")
    print("saved:", SAME_OT_OUTPUT_DIR / "report.json")


## 5. Evidence Utilities. External Comparison. Underfit LoRA Handoff

Use
[dada-bots/underfit](https://github.com/dada-bots/underfit) for SA3 LoRA
training, run monitoring, checkpoint selection, and adapter-specific logic.

This notebook compares audio exported from an Underfit run through the ordinary
descriptor, memory, player, and annotation cells.

Recommended handoff flow:

```text
local dataset / selected clips
  -> prompt and metadata notes
  -> Underfit LoRA run
  -> exported audio examples and run notes
  -> SA3 Native Lab descriptor/listening comparison
```


In [ ]:
# @title 5. Evidence Utilities. External Comparison. Audio-output baseline harness

RUN_EXTERNAL_CROSS_MODEL = False

CROSS_MODEL_PROMPTS = [
    "sparse glassy ambient loop",
    "bright rhythmic synthetic music",
]
CROSS_MODEL_DURATION = 8.0
CROSS_MODEL_STEPS = 8
CROSS_MODEL_SEED = 42
CROSS_MODEL_EXTERNAL_COMMANDS = []  # Optional audio generators only; compare exported audio through local evidence packets.
CROSS_MODEL_OUTPUT_DIR = OUTPUT_DIR / "external_cross_model"


def format_cross_model_command(command, *, prompt, output):
    return [str(part).format(prompt=prompt, output=output, duration=CROSS_MODEL_DURATION, seed=CROSS_MODEL_SEED) for part in command]


if RUN_EXTERNAL_CROSS_MODEL:
    require_models()
    import subprocess

    CROSS_MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    rows = []
    paths = []
    labels = []
    for prompt_index, prompt in enumerate(CROSS_MODEL_PROMPTS):
        sa3_latents = sa3.generate_latents(
            prompt=prompt,
            duration=CROSS_MODEL_DURATION,
            steps=CROSS_MODEL_STEPS,
            cfg_scale=1.0,
            seed=CROSS_MODEL_SEED + prompt_index,
        )
        sa3_path = CROSS_MODEL_OUTPUT_DIR / f"sa3_{prompt_index}.wav"
        decode_sa3_latents_to_file_cropped(sa3_latents, sa3_path, duration=CROSS_MODEL_DURATION)
        audio, sample_rate = load_audio(sa3_path, stereo=True)
        rows.append({"model": "sa3", "prompt": prompt, "path": str(sa3_path), "descriptors": audio_descriptor_report(audio, sample_rate)})
        paths.append(sa3_path)
        labels.append(f"sa3 - {prompt}")
        for model_index, command_template in enumerate(CROSS_MODEL_EXTERNAL_COMMANDS):
            out_path = CROSS_MODEL_OUTPUT_DIR / f"external_{model_index}_{prompt_index}.wav"
            command = format_cross_model_command(command_template, prompt=prompt, output=out_path)
            print(" ".join(map(str, command)))
            subprocess.run(command, check=True)
            audio, sample_rate = load_audio(out_path, stereo=True)
            rows.append({"model": f"external_{model_index}", "prompt": prompt, "path": str(out_path), "descriptors": audio_descriptor_report(audio, sample_rate)})
            paths.append(out_path)
            labels.append(f"external {model_index} - {prompt}")
    (CROSS_MODEL_OUTPUT_DIR / "report.json").write_text(json.dumps(rows, indent=2), encoding="utf-8")
    play_audio_files(paths, labels=labels, title="External Comparison Bench: Audio-Output Baseline Harness")
    print("saved:", CROSS_MODEL_OUTPUT_DIR / "report.json")


## 5. Evidence Utilities. Ledger And Decisions. Combined Evidence Chain

A practical first research chain:

- target audio -> SAME latent
- SAME latent -> soft prompt \(c^\star\)
- \(c^\star\) -> candidate latent
- candidate latent -> residual steering alpha sweep
- steered latent -> SAME profile attraction
- final latent -> decode -> memory item -> evidence packet

This combines:

- SA3 flow and conditioning science: audio -> soft prompt
- SA3 internal trajectory science: residual controls
- SA3-over-SAME coupled editing: SAME latent editing
- SAME memory and composition: saved latent memory
- Ledger and decisions: baseline, measurement, audition, decision


In [ ]:
# @title 5. Evidence Utilities. Combined chain scaffold

RUN_EVIDENCE_COMBINED_CHAIN = False

if RUN_EVIDENCE_COMBINED_CHAIN:
    require_models()
    soft_state = SoftPromptState.load(SOFT_PROMPT_PATH)
    profile = load_style_profile(STYLE_PROFILE_PATH)
    vectors = SteeringVectors.load(PROMPT_VECTOR_DIR / "steering_vectors.pt")
    steerer = ResidualSteerer(sa3_model, vectors, top_k=1)

    with steerer.steer(alpha=1.0):
        latents = generate_with_soft_prompt(
            sa3_model,
            soft_state,
            steps=8,
            cfg_scale=1.0,
            seed=500,
            return_latents=True,
        )

    item = sa3_tensor_to_item(latents, item_id="combined_source", prompt="soft_prompt")
    styled_z = apply_profile_attraction(item, profile, alpha=0.5, match_std=True)
    styled_latents = torch.from_numpy(styled_z.T[None, :, :]).to(device=DEVICE, dtype=latents.dtype)
    out_path = OUTPUT_DIR / "combined_soft_residual_profile.wav"
    decode_sa3_latents_to_file(styled_latents, out_path)
    save_items([sa3_tensor_to_item(styled_latents, item_id="combined_final", prompt="soft+causal_residual+same_profile")], MEMORY_DIR / "combined")
    print("saved:", out_path)
    play_audio_files([out_path], title="Combined chain output")


## 5. Evidence Utilities. Ledger And Decisions. Experiment Log Template

For each run, save:

```text
input audio paths
native object transition
prompts
seeds
duration
steps
cfg scale
method flags
vector layer ids
alpha values
loss curves
output audio paths
listening notes
failure cases
maturity update
```

Minimum report:

```text
What object changed?
What baseline was used?
Was the change measurable?
Was the change audible?
Was it controllable with alpha/seed/prompt?
Did it generalize to another prompt or source?
Did SAME statistics move in the intended direction?
Did residual probe/loss values predict listening behavior?
Should the method stay microscope, become selector, become intervention candidate, promote, revise, or drop?
```


In [ ]:
# @title 5. Evidence Utilities. Save an experiment manifest

manifest = {
    "work_dir": str(WORK_DIR),
    "sa3_model": SA3_MODEL_NAME,
    "same_model": SAME_MODEL_NAME,
    "device": DEVICE,
    "dataset_use_chunks": DATASET_USE_CHUNKS,
    "dataset_chunk_duration": DATASET_CHUNK_DURATION,
    "dataset_hop_duration": DATASET_HOP_DURATION,
    "dataset_max_chunks_per_file": DATASET_MAX_CHUNKS_PER_FILE,
    "research_layers": [
        "SAME representation",
        "SA3 flow/conditioning",
        "SA3 internal trajectory",
        "SA3-over-SAME coupled editing",
    ],
    "evidence_utility": "player/annotations/disagreement/ledger",
    "experiments": {
        "same_neighborhood_renoise": RUN_SAME_NEIGHBORHOOD_RENOISE,
        "same_selective_renoise": RUN_SAME_SELECTIVE_RENOISE,
        "evidence_annotation_retrieval": RUN_EVIDENCE_ANNOTATION_RETRIEVAL,
        "same_audio_graft": RUN_SAME_AUDIO_GRAFT,
        "same_blur_bottleneck": RUN_SAME_BLUR_BOTTLENECK,
        "same_neural_dsp": RUN_SAME_NEURAL_DSP,
        "same_loop_repair": RUN_SAME_LOOP_REPAIR,
        "causal_cyclic_trajectory": RUN_CAUSAL_CYCLIC_TRAJECTORY,
        "flow_sign_diagnostic": RUN_FLOW_SIGN_DIAGNOSTIC,
        "flow_conditioning_soft_prompt_inversion": RUN_FLOW_SOFT_PROMPT_INVERSION,
        "flow_conditioning_soft_prompt_audition": RUN_FLOW_SOFT_PROMPT_AUDITION,
        "flow_conditioning_hard_prompt_search": RUN_FLOW_HARD_PROMPT_SEARCH,
        "flow_conditioning_readable_prompt_search": RUN_FLOW_READABLE_PROMPT_SEARCH,
        "flow_conditioning_dataset_soft_prompt": RUN_FLOW_DATASET_SOFT_PROMPT,
        "dataset_prompt_family": RUN_DATASET_PROMPT_FAMILY,
        "same_style_profile": RUN_SAME_STYLE_PROFILE,
        "same_style_direction": RUN_SAME_STYLE_DIRECTION,
        "causal_prompt_residual": RUN_CAUSAL_PROMPT_RESIDUAL,
        "causal_audio_residual": RUN_CAUSAL_AUDIO_RESIDUAL,
        "causal_flow_state_optimization": RUN_CAUSAL_FLOW_STATE_OPTIMIZATION,
        "dataset_continuation_composition": RUN_DATASET_CONTINUATION_COMPOSITION,
        "causal_control_head": RUN_CAUSAL_CONTROL_HEAD,
        "external_underfit_handoff": "underfit",
        "dataset_memory_index": RUN_DATASET_MEMORY_INDEX,
        "same_geometry_audit": RUN_SAME_GEOMETRY_AUDIT,
        "flow_conditioning_attribution": RUN_FLOW_ATTRIBUTION,
        "flow_conditioning_timestep_panel": RUN_FLOW_TIMESTEP_PANEL,
        "prompt_semantic_transparency": RUN_PROMPT_SEMANTIC_TRANSPARENCY,
        "semantic_bottleneck_disagreement": RUN_SEMANTIC_BOTTLENECK_DISAGREEMENT,
        "evidence_control_lanes": RUN_EVIDENCE_CONTROL_LANES,
        "dataset_memory_curriculum": RUN_DATASET_MEMORY_CURRICULUM,
        "same_ot_style_transfer": RUN_SAME_OT_STYLE_TRANSFER,
        "dataset_bridge_search": RUN_DATASET_BRIDGE_SEARCH,
        "causal_residual_feature_atlas": RUN_CAUSAL_RESIDUAL_FEATURE_ATLAS,
        "flow_conditioning_null_inversion": RUN_FLOW_NULL_INVERSION,
        "causal_gradient_edit": RUN_CAUSAL_GRADIENT_EDIT,
        "causal_audio_posterior": RUN_CAUSAL_AUDIO_POSTERIOR,
        "external_audio_output_baseline": RUN_EXTERNAL_CROSS_MODEL,
        "evidence_combined_chain": RUN_EVIDENCE_COMBINED_CHAIN,
    },

}

manifest_path = OUTPUT_DIR / "experiment_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("saved:", manifest_path)
